# BASELINE & MODEL

In [19]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

print("===== LOADING DATASET & TOKENIZER =====")

gsm8k = load_dataset("openai/gsm8k", "main")

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-Math-1.5B"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("GSM8K loaded successfully.")
print("Train samples:", len(gsm8k["train"]))
print("Test samples:", len(gsm8k["test"]))
print("Tokenizer loaded successfully.")

===== LOADING DATASET & TOKENIZER =====
GSM8K loaded successfully.
Train samples: 7473
Test samples: 1319
Tokenizer loaded successfully.


In [20]:
import re

def extract_gsm8k_answer(text):
    """
    Extract final numerical answer from GSM8K-style model output.
    """

    if text is None:
        return None

    # 1. GSM8K format: #### 18
    match = re.search(
        r"####\s*([-+]?\d+(?:\.\d+)?)",
        text
    )

    if match:
        return match.group(1)

    # 2. LaTeX boxed format: \boxed{18}
    match = re.search(
        r"\\boxed\{([-+]?\d+(?:\.\d+)?)\}",
        text
    )

    if match:
        return match.group(1)

    return None


def normalize_answer(answer):
    if answer is None:
        return None

    answer = str(answer).strip()
    answer = answer.replace(",", "")

    # 18.0 -> 18
    if answer.endswith(".0"):
        answer = answer[:-2]

    return answer


def is_correct(predicted, gold):
    predicted = normalize_answer(predicted)
    gold = normalize_answer(gold)

    return predicted == gold

In [21]:
gold = extract_gsm8k_answer(gsm8k["test"][0]["answer"])

print("Gold answer:", gold)

Gold answer: 18


In [22]:
print("Test 1:", is_correct("18", "18"))
print("Test 2:", is_correct("18.0", "18"))
print("Test 3:", is_correct("17", "18"))

Test 1: True
Test 2: True
Test 3: False


In [23]:
def generate_answer(question, max_new_tokens=512):
    prompt = (
        "Please reason step by step, and put your final answer within \\boxed{}.\n\n"
        f"{question}\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response

In [24]:
from transformers import AutoModelForCausalLM

print("===== LOADING BASE MODEL =====")

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Math-1.5B",
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Base model loaded successfully.")
print("Model type:", type(model))
print("Device:", model.device)

===== LOADING BASE MODEL =====


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model loaded successfully.
Model type: <class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
Device: cuda:0


In [25]:
question = gsm8k["test"][0]["question"]

response = generate_answer(question)

print(response)

To determine how much Janet makes every day at the farmers' market, we need to follow these steps:

1. Calculate the number of eggs that are left after accounting for the eggs she eats and the eggs she bakes.
2. Determine how many of these remaining eggs are sold at the farmers' market.
3. Calculate the total revenue from selling these eggs.

Let's break it down step by step:

1. Janet's ducks lay 16 eggs per day.
2. She eats 3 eggs for breakfast every morning, so she eats \(3 \times 7 = 21\) eggs in a week (assuming a 7-day week).
3. She bakes 4 eggs for her friends every day, so she bakes \(4 \times 7 = 28\) eggs in a week.
4. The total number of eggs she uses in a week is \(21 + 28 = 49\) eggs.
5. The number of eggs left after using 49 eggs is \(16 \times 7 - 49 = 112 - 49 = 63\) eggs.
6. Since she sells the remaining eggs at the farmers' market, and she sells them at $2 per egg, the total revenue is \(63 \times 2 = 126\) dollars.

Let's confirm this with Python code.
```python
# Co

In [26]:
predicted = extract_gsm8k_answer(response)
gold = extract_gsm8k_answer(gsm8k["test"][0]["answer"])

print("Predicted:", predicted)
print("Gold:", gold)
print("Correct:", is_correct(predicted, gold))

Predicted: 126
Gold: 18
Correct: False


In [27]:
import time

results = []

for i in range(10):
    question = gsm8k["test"][i]["question"]
    gold = extract_gsm8k_answer(gsm8k["test"][i]["answer"])

    start_time = time.time()

    response = generate_answer(question)

    latency = time.time() - start_time

    predicted = extract_gsm8k_answer(response)
    correct = is_correct(predicted, gold)

    # Số token model thực sự sinh ra
    inputs = tokenizer(
        "Please reason step by step, and put your final answer within \\boxed{}.\n\n"
        + question,
        return_tensors="pt"
    )

    input_length = inputs["input_ids"].shape[1]

    output_tokens = tokenizer(
        response,
        return_tensors="pt"
    )["input_ids"].shape[1]

    results.append({
        "id": i,
        "predicted": predicted,
        "gold": gold,
        "correct": correct,
        "latency": latency,
        "output_tokens": output_tokens
    })

    print(
        f"[{i+1}/10] "
        f"Predicted={predicted} | "
        f"Gold={gold} | "
        f"Correct={correct} | "
        f"Latency={latency:.2f}s | "
        f"Tokens={output_tokens}"
    )

[1/10] Predicted=126 | Gold=18 | Correct=False | Latency=34.36s | Tokens=512
[2/10] Predicted=3 | Gold=3 | Correct=True | Latency=8.21s | Tokens=192
[3/10] Predicted=70000 | Gold=70000 | Correct=True | Latency=17.53s | Tokens=384
[4/10] Predicted=180 | Gold=540 | Correct=False | Latency=7.34s | Tokens=171
[5/10] Predicted=140 | Gold=20 | Correct=False | Latency=19.01s | Tokens=413
[6/10] Predicted=64 | Gold=64 | Correct=True | Latency=9.61s | Tokens=215
[7/10] Predicted=260 | Gold=260 | Correct=True | Latency=11.71s | Tokens=263
[8/10] Predicted=None | Gold=160 | Correct=False | Latency=23.11s | Tokens=512
[9/10] Predicted=None | Gold=45 | Correct=False | Latency=22.81s | Tokens=512
[10/10] Predicted=460 | Gold=460 | Correct=True | Latency=14.22s | Tokens=319


In [28]:
correct_count = sum(r["correct"] for r in results)
total_count = len(results)

accuracy = correct_count / total_count * 100
avg_latency = sum(r["latency"] for r in results) / total_count
avg_tokens = sum(r["output_tokens"] for r in results) / total_count

print("===== BASELINE 10 SAMPLES =====")
print(f"Correct: {correct_count}/{total_count}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"Average latency: {avg_latency:.2f}s")
print(f"Average generated tokens: {avg_tokens:.2f}")

===== BASELINE 10 SAMPLES =====
Correct: 5/10
Accuracy: 50.00%
Average latency: 16.79s
Average generated tokens: 349.30


# SFT-Cot BASELINE

In [29]:
train_sample = gsm8k["train"][0]

print("===== QUESTION =====")
print(train_sample["question"])

print("\n===== SOLUTION =====")
print(train_sample["answer"])

===== QUESTION =====
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

===== SOLUTION =====
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [30]:

# ============================================================
# CELL 14 — SFT-CoT PROMPT / COMPLETION DATASET
# Standardized prompt for the entire USC-DPO pipeline.
# ============================================================

# ------------------------------------------------------------
# 1. DEFINE THE CANONICAL MATH REASONING PROMPT
# This prompt will also be reused in candidate generation
# and final evaluation in later cells.
# ------------------------------------------------------------

def make_math_prompt(question):

    return (
        "Solve this math word problem carefully. "
        "Show concise reasoning with the key calculations. "
        "Avoid repeated explanations and unnecessary self-talk. "
        "Check the arithmetic before answering.\n"
        "End with the final answer in this exact format: #### <number>\n"
        "Do not write anything after the final-answer line.\n\n"
        f"Problem:\n{question}\n\n"
        "Solution:\n"
    )


# ------------------------------------------------------------
# 2. FORMAT GSM8K EXAMPLES
# ------------------------------------------------------------

def format_sft_example(example):

    return {
        "prompt": make_math_prompt(example["question"]),
        "completion": example["answer"],
    }


# ------------------------------------------------------------
# 3. BUILD THE FULL SFT DATASET
# ------------------------------------------------------------

sft_train = gsm8k["train"].map(
    format_sft_example,
    desc="Preparing standardized SFT-CoT dataset"
)


# ------------------------------------------------------------
# 4. VALIDATE DATASET
# ------------------------------------------------------------

required_columns = {
    "question",
    "answer",
    "prompt",
    "completion",
}

missing_columns = (
    required_columns - set(sft_train.column_names)
)

if missing_columns:
    raise RuntimeError(
        f"Missing required columns: {missing_columns}"
    )

if len(sft_train) == 0:
    raise RuntimeError("SFT dataset is empty.")

# Check that all examples have valid prompt/completion fields.
invalid_examples = [
    i for i, example in enumerate(sft_train)
    if not example["prompt"].strip()
    or not example["completion"].strip()
]

if invalid_examples:
    raise RuntimeError(
        f"Found {len(invalid_examples)} invalid SFT examples."
    )

# Check that GSM8K reference solutions contain final-answer markers.
missing_answer_markers = sum(
    "####" not in example["answer"]
    for example in sft_train
)

if missing_answer_markers:
    raise RuntimeError(
        f"{missing_answer_markers} GSM8K solutions "
        "do not contain the expected #### answer marker."
    )


# ------------------------------------------------------------
# 5. PRINT DATASET SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("STANDARDIZED SFT-CoT DATASET")
print("=" * 65)

print("Base model:", "Qwen/Qwen2.5-Math-1.5B")
print("Dataset:", "GSM8K train")
print("Total samples:", len(sft_train))
print("Invalid examples:", len(invalid_examples))
print("Missing answer markers:", missing_answer_markers)

print("\n===== SAMPLE SFT PROMPT =====")
print(sft_train[0]["prompt"])

print("\n===== SAMPLE SFT COMPLETION =====")
print(sft_train[0]["completion"])

print("\nSFT-CoT dataset prepared successfully.")


STANDARDIZED SFT-CoT DATASET
Base model: Qwen/Qwen2.5-Math-1.5B
Dataset: GSM8K train
Total samples: 7473
Invalid examples: 0
Missing answer markers: 0

===== SAMPLE SFT PROMPT =====
Solve this math word problem carefully. Show concise reasoning with the key calculations. Avoid repeated explanations and unnecessary self-talk. Check the arithmetic before answering.
End with the final answer in this exact format: #### <number>
Do not write anything after the final-answer line.

Problem:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Solution:


===== SAMPLE SFT COMPLETION =====
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

SFT-CoT dataset prepared successfully.


In [31]:

# ============================================================
# CELL 15 — SFT TRAINING DATASET PREPARATION
# Full SFT training set + isolated preference generation pool
# ============================================================

import random

# ------------------------------------------------------------
# 1. REPRODUCIBILITY
# ------------------------------------------------------------

SPLIT_SEED = 42

random.seed(SPLIT_SEED)

# Reserve the same number of questions used by the current
# USC-DPO candidate-generation setup in Cell 24.
NUM_PREFERENCE_QUESTIONS = 80

# ------------------------------------------------------------
# 2. VALIDATE SOURCE DATASET
# ------------------------------------------------------------

required_columns = {
    "question",
    "answer",
    "prompt",
    "completion"
}

missing_columns = required_columns - set(sft_train.column_names)

if missing_columns:
    raise ValueError(
        f"SFT dataset is missing columns: {missing_columns}. "
        "Run the SFT dataset preparation cell again."
    )

if len(sft_train) <= NUM_PREFERENCE_QUESTIONS:
    raise ValueError(
        "Not enough GSM8K training samples to create "
        "separate SFT and preference datasets."
    )

# ------------------------------------------------------------
# 3. SPLIT SFT DATA AND PREFERENCE DATA
# ------------------------------------------------------------

dataset_split = sft_train.train_test_split(
    test_size=NUM_PREFERENCE_QUESTIONS,
    seed=SPLIT_SEED,
    shuffle=True
)

# Main supervised fine-tuning dataset.
sft_train_final = dataset_split["train"]

# Held-out questions reserved for USC-DPO candidate generation.
preference_pool = dataset_split["test"]

# Compatibility alias:
# The current SFTTrainer cell uses the variable sft_debug.
# Keep this alias so that the next cell can use the new dataset
# without changing its variable reference immediately.
sft_debug = sft_train_final

# ------------------------------------------------------------
# 4. CHECK DATASET SEPARATION
# ------------------------------------------------------------

sft_questions = set(sft_train_final["question"])
preference_questions = set(preference_pool["question"])

overlap = sft_questions.intersection(preference_questions)

if overlap:
    raise RuntimeError(
        f"Data leakage detected: {len(overlap)} overlapping questions."
    )

if len(sft_train_final) + len(preference_pool) != len(sft_train):
    raise RuntimeError(
        "Dataset split size mismatch."
    )

# Verify that the held-out preference pool comes from GSM8K train,
# not from the GSM8K test set.
gsm8k_test_questions = set(gsm8k["test"]["question"])

test_overlap = preference_questions.intersection(
    gsm8k_test_questions
)

if test_overlap:
    raise RuntimeError(
        "Preference pool contains questions from GSM8K test."
    )

# ------------------------------------------------------------
# 5. PRINT DATASET SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("SFT-CoT AND USC-DPO DATASET SPLIT")
print("=" * 65)

print(f"Original GSM8K train samples : {len(sft_train)}")
print(f"SFT-CoT training samples     : {len(sft_train_final)}")
print(f"USC-DPO preference questions : {len(preference_pool)}")

print("\nDataset separation:")
print(f"SFT / preference overlap     : {len(overlap)}")
print(f"Preference / test overlap    : {len(test_overlap)}")

print("\n===== SFT TRAINING SAMPLE =====")
print("PROMPT:")
print(sft_train_final[0]["prompt"])

print("\nCOMPLETION:")
print(sft_train_final[0]["completion"])

print("\n===== USC-DPO PREFERENCE SAMPLE =====")
print("QUESTION:")
print(preference_pool[0]["question"])

print("\nGOLD SOLUTION:")
print(preference_pool[0]["answer"])

print("\nSFT and preference datasets prepared successfully.")


SFT-CoT AND USC-DPO DATASET SPLIT
Original GSM8K train samples : 7473
SFT-CoT training samples     : 7393
USC-DPO preference questions : 80

Dataset separation:
SFT / preference overlap     : 0
Preference / test overlap    : 0

===== SFT TRAINING SAMPLE =====
PROMPT:
Solve this math word problem carefully. Show concise reasoning with the key calculations. Avoid repeated explanations and unnecessary self-talk. Check the arithmetic before answering.
End with the final answer in this exact format: #### <number>
Do not write anything after the final-answer line.

Problem:
Nathan plays amateur baseball. He played for 3 hours for two weeks, every day. His friend Tobias played for 5 hours every day, but only for one week. How many hours did Nathan and Tobias play in total?

Solution:


COMPLETION:
Two weeks are 14 days, so Nathan played for 3 * 14 = <<14*3=42>>42 hours.
Tobias played for 7 days, so he played a total of 5 * 7 = <<5*7=35>>35 hours.
Nathan and Tobias played together for 42 + 35 

In [32]:
!pip install -q -U "torchao>=0.16.0"

In [33]:
!pip install -q trl peft
import trl
import peft

print("TRL version:", trl.__version__)
print("PEFT version:", peft.__version__)

TRL version: 1.17.1
PEFT version: 0.21.2


In [34]:

# ============================================================
# CELL 18 — SFT-CoT FULL TRAINING CONFIGURATION
# Qwen2.5-Math-1.5B + LoRA
# ============================================================

import os
import torch

from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# ------------------------------------------------------------
# 1. REPRODUCIBILITY AND ENVIRONMENT
# ------------------------------------------------------------

SEED = 42

torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for SFT-CoT training. "
        "Enable GPU in Colab before continuing."
    )

torch.cuda.manual_seed_all(SEED)

print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 2. MODEL AND TRAINING CONFIGURATION
# ------------------------------------------------------------

MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"

SFT_OUTPUT_DIR = (
    "/content/USC_DPO_artifacts/sft_training_checkpoints"
)

os.makedirs(SFT_OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 3. LoRA CONFIGURATION
# Keep this compatible with USC-DPO Cell 36.
# ------------------------------------------------------------

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

# ------------------------------------------------------------
# 4. SFT TRAINING CONFIGURATION
# ------------------------------------------------------------

sft_config = SFTConfig(
    output_dir=SFT_OUTPUT_DIR,

    # Train on the full SFT dataset prepared in Cell 15.
    num_train_epochs=1,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    # Conservative learning rate for LoRA fine-tuning.
    learning_rate=1e-5,

    lr_scheduler_type="cosine",
    warmup_steps=56,

    weight_decay=0.0,
    max_grad_norm=1.0,

    # Sequence configuration.
    max_length=1024,
    completion_only_loss=True,
    packing=False,

    # GPU memory optimization.
    fp16=True,
    gradient_checkpointing=True,

    # Logging and checkpointing.
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True,

    save_strategy="epoch",
    save_total_limit=1,

    # Reproducibility.
    seed=SEED,
    data_seed=SEED,

    report_to="none",
)

# ------------------------------------------------------------
# 5. CONFIGURATION VALIDATION
# ------------------------------------------------------------

assert lora_config.r == 8
assert lora_config.lora_alpha == 16

assert sft_config.num_train_epochs == 1
assert sft_config.per_device_train_batch_size == 1
assert sft_config.gradient_accumulation_steps == 4

assert sft_config.completion_only_loss is True

print("=" * 65)
print("SFT-CoT — FULL TRAINING CONFIGURATION")
print("=" * 65)

print("\nModel:")
print("  Model:", MODEL_NAME)

print("\nLoRA:")
print("  Rank:", lora_config.r)
print("  Alpha:", lora_config.lora_alpha)
print("  Dropout:", lora_config.lora_dropout)
print("  Target modules:", lora_config.target_modules)

print("\nTraining:")
print("  Epochs:", sft_config.num_train_epochs)
print("  Batch size:", sft_config.per_device_train_batch_size)
print("  Gradient accumulation:", sft_config.gradient_accumulation_steps)
print("  Effective batch size:", (
    sft_config.per_device_train_batch_size
    * sft_config.gradient_accumulation_steps
))
print("  Learning rate:", sft_config.learning_rate)
print("  Scheduler:", sft_config.lr_scheduler_type)
print("  Max sequence length:", sft_config.max_length)
print("  Completion-only loss:", sft_config.completion_only_loss)

print("\nCheckpoint directory:")
print(" ", SFT_OUTPUT_DIR)

print("\nSFT-CoT configuration validated successfully.")


GPU: Tesla T4
SFT-CoT — FULL TRAINING CONFIGURATION

Model:
  Model: Qwen/Qwen2.5-Math-1.5B

LoRA:
  Rank: 8
  Alpha: 16
  Dropout: 0.05
  Target modules: {'q_proj', 'v_proj'}

Training:
  Epochs: 1
  Batch size: 1
  Gradient accumulation: 4
  Effective batch size: 4
  Learning rate: 1e-05
  Scheduler: SchedulerType.COSINE
  Max sequence length: 1024
  Completion-only loss: True

Checkpoint directory:
  /content/USC_DPO_artifacts/sft_training_checkpoints

SFT-CoT configuration validated successfully.


In [35]:

# ============================================================
# CELL 19 — INITIALIZE SFT-CoT TRAINER
# Qwen2.5-Math-1.5B + LoRA
# ============================================================

import gc
import torch

from transformers import AutoModelForCausalLM
from trl import SFTTrainer

# ------------------------------------------------------------
# 1. VALIDATE PREPARED DATASET
# ------------------------------------------------------------

if "sft_debug" not in globals():
    raise RuntimeError(
        "SFT dataset not found. Run Cell 15 first."
    )

if len(sft_debug) == 0:
    raise ValueError("SFT training dataset is empty.")

required_columns = {"prompt", "completion"}

missing_columns = required_columns - set(sft_debug.column_names)

if missing_columns:
    raise ValueError(
        f"Missing SFT columns: {missing_columns}"
    )

print("=" * 65)
print("INITIALIZING SFT-CoT TRAINER")
print("=" * 65)

print("Training samples:", len(sft_debug))
print("Model:", MODEL_NAME)

# ------------------------------------------------------------
# 2. RELEASE PREVIOUS MODEL REFERENCES
# The Base model was loaded earlier for baseline evaluation.
# Release it before loading the training model.
# ------------------------------------------------------------

for variable_name in ("trainer", "sft_model", "model"):

    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 3. LOAD A FRESH BASE MODEL
# ------------------------------------------------------------

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available.")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

# Gradient checkpointing requires cache to be disabled.
model.config.use_cache = False

model.train()

print("\nBase model loaded successfully.")
print("Model type:", type(model))
print("Model device:", model.device)
print("Model dtype:", next(model.parameters()).dtype)

# ------------------------------------------------------------
# 4. INITIALIZE SFT TRAINER
# ------------------------------------------------------------

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_debug,
    processing_class=tokenizer,
    peft_config=lora_config,
)

# ------------------------------------------------------------
# 5. VALIDATE TRAINER
# ------------------------------------------------------------

trainable_parameters = 0
total_parameters = 0

for parameter in trainer.model.parameters():

    total_parameters += parameter.numel()

    if parameter.requires_grad:
        trainable_parameters += parameter.numel()

trainable_percentage = (
    100.0 * trainable_parameters / total_parameters
)

actual_dataset_size = len(trainer.train_dataset)

if actual_dataset_size != len(sft_debug):
    raise RuntimeError(
        "Trainer dataset size does not match the prepared SFT dataset."
    )

if trainable_parameters == 0:
    raise RuntimeError(
        "No trainable parameters found. Check LoRA configuration."
    )

print("\n" + "=" * 65)
print("SFT-CoT TRAINER VALIDATION")
print("=" * 65)

print("Training dataset size:", actual_dataset_size)
print("Trainable parameters:", trainable_parameters)
print("Total parameters:", total_parameters)
print(f"Trainable percentage: {trainable_percentage:.4f}%")

print("LoRA adapters:", list(
    trainer.model.peft_config.keys()
) if hasattr(trainer.model, "peft_config") else "Not detected")

print("\nSFTTrainer initialized successfully.")
print("Ready for Cell 20 training.")


INITIALIZING SFT-CoT TRAINER
Training samples: 7393
Model: Qwen/Qwen2.5-Math-1.5B


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Base model loaded successfully.
Model type: <class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
Model device: cuda:0
Model dtype: torch.float16

SFT-CoT TRAINER VALIDATION
Training dataset size: 7393
Trainable parameters: 1089536
Total parameters: 1544803840
Trainable percentage: 0.0705%
LoRA adapters: ['default']

SFTTrainer initialized successfully.
Ready for Cell 20 training.


In [36]:
print("===== START SFT DEBUG TRAINING =====")

train_result = trainer.train()

print("\n===== SFT DEBUG TRAINING FINISHED =====")
print(train_result)

===== START SFT DEBUG TRAINING =====


Step,Training Loss
1,0.877074
50,0.792259
100,0.753700
150,0.736904
200,0.680648
250,0.683269
300,0.683823
350,0.663095
400,0.626108
450,0.599249



===== SFT DEBUG TRAINING FINISHED =====
TrainOutput(global_step=1849, training_loss=0.5515524961614429, metrics={'train_runtime': 2084.078, 'train_samples_per_second': 3.547, 'train_steps_per_second': 0.887, 'total_flos': 1.3988196522196992e+16, 'train_loss': 0.5515524961614429, 'entropy': 0.5107677054683162, 'mean_token_accuracy': 0.8641431591053701, 'num_tokens': 1777728.0, 'epoch': 1.0})


In [37]:
# ============================================================
# FINAL SFT-CoT TEST
# Uses the updated prompt and shared answer parser.
# ============================================================

import torch

print("===== FINAL SFT-CoT TEST =====")

if "trainer" not in globals():
    raise RuntimeError("SFT trainer not found. Run Cell 19 and Cell 20.")

if "make_math_prompt" not in globals():
    raise RuntimeError("make_math_prompt not found. Check Cell 14.")

sft_model = trainer.model
sft_model.eval()

print("Model type:", type(sft_model))
print("Has PEFT config:", hasattr(sft_model, "peft_config"))

if hasattr(sft_model, "peft_config"):
    print("PEFT adapters:", list(sft_model.peft_config.keys()))

trainable = sum(
    p.numel() for p in sft_model.parameters() if p.requires_grad
)
total = sum(p.numel() for p in sft_model.parameters())

print("Trainable:", trainable)
print("Total:", total)
print("Trainable %:", 100 * trainable / total)

# Use the first held-out GSM8K test example
example = gsm8k["test"][0]

prompt = make_math_prompt(example["question"])

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(sft_model.device)

with torch.no_grad():
    outputs = sft_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

prediction = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

predicted = extract_gsm8k_answer(prediction)
gold = extract_gsm8k_answer(example["answer"])

print("\n===== QUESTION =====")
print(example["question"])

print("\n===== GOLD SOLUTION =====")
print(example["answer"])

print("\n===== SFT-CoT OUTPUT =====")
print(prediction)

print("\n===== ANSWER CHECK =====")
print("Predicted:", predicted)
print("Gold:", gold)
print("Correct:", is_correct(predicted, gold))

print("\n===== OUTPUT TOKENS =====")
print(len(generated_tokens))

===== FINAL SFT-CoT TEST =====
Model type: <class 'peft.peft_model.PeftModelForCausalLM'>
Has PEFT config: True
PEFT adapters: ['default']
Trainable: 1089536
Total: 1544803840
Trainable %: 0.07052908413277896

===== QUESTION =====
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

===== GOLD SOLUTION =====
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18

===== SFT-CoT OUTPUT =====
Janet's ducks lay a total of 16 eggs per day.
She eats 3 eggs for breakfast, leaving 16 - 3 = <<16-3=13>>13 eggs.
She bakes 4 eggs into muffins, leaving 13 - 4 = <<13-4=9>>9 eggs.
She sells these 9 eggs for $2 each, making 9 * $2 = <<9*2=18>>18 dollars.
#### 18

===== ANSWER CHECK =====
Predicted: 18
G

In [38]:
# ============================================================
# SFT-CoT BENCHMARK — 10 SAMPLES
# Uses the same prompt and answer parser as the updated pipeline.
# ============================================================

import time
import torch

print("===== SFT-CoT BENCHMARK — 10 SAMPLES =====")

sft_model = trainer.model
sft_model.eval()

results_sft = []

for i in range(10):

    question = gsm8k["test"][i]["question"]
    gold_answer = extract_gsm8k_answer(
        gsm8k["test"][i]["answer"]
    )

    # Use the shared prompt from Cell 14
    prompt = make_math_prompt(question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(sft_model.device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.time()

    with torch.no_grad():
        outputs = sft_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    latency = time.time() - start_time

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # Shared strict parser: no arbitrary last-number fallback
    predicted_answer = extract_gsm8k_answer(prediction)

    correct = is_correct(
        predicted_answer,
        gold_answer
    )

    results_sft.append({
        "index": i + 1,
        "predicted": predicted_answer,
        "gold": gold_answer,
        "correct": correct,
        "latency": latency,
        "tokens": len(generated_tokens),
        "output": prediction
    })

    print(
        f"[{i+1}] "
        f"Predicted={predicted_answer} | "
        f"Gold={gold_answer} | "
        f"Correct={correct} | "
        f"{latency:.2f}s | "
        f"{len(generated_tokens)} tokens"
    )

correct_count = sum(
    r["correct"] for r in results_sft
)

total_count = len(results_sft)

accuracy = correct_count / total_count * 100

unparsed_count = sum(
    r["predicted"] is None for r in results_sft
)

avg_latency = sum(
    r["latency"] for r in results_sft
) / total_count

avg_tokens = sum(
    r["tokens"] for r in results_sft
) / total_count

print("\n===== SFT-CoT BENCHMARK SUMMARY =====")
print(f"Correct: {correct_count}/{total_count}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"Unparsed predictions: {unparsed_count}")
print(f"Average latency: {avg_latency:.2f}s")
print(f"Average generated tokens: {avg_tokens:.2f}")

===== SFT-CoT BENCHMARK — 10 SAMPLES =====
[1] Predicted=18 | Gold=18 | Correct=True | 5.25s | 108 tokens
[2] Predicted=3 | Gold=3 | Correct=True | 1.94s | 39 tokens
[3] Predicted=25000 | Gold=70000 | Correct=False | 10.10s | 219 tokens
[4] Predicted=180 | Gold=540 | Correct=False | 2.15s | 41 tokens
[5] Predicted=1 | Gold=20 | Correct=False | 4.60s | 103 tokens
[6] Predicted=64 | Gold=64 | Correct=True | 4.86s | 108 tokens
[7] Predicted=260 | Gold=260 | Correct=True | 4.85s | 94 tokens
[8] Predicted=120 | Gold=160 | Correct=False | 8.61s | 191 tokens
[9] Predicted=375 | Gold=45 | Correct=False | 4.49s | 93 tokens
[10] Predicted=460 | Gold=460 | Correct=True | 8.03s | 179 tokens

===== SFT-CoT BENCHMARK SUMMARY =====
Correct: 5/10
Accuracy: 50.00%
Unparsed predictions: 0
Average latency: 5.49s
Average generated tokens: 117.50


# USC-DPO — REASONING CANDIDATE GENERATION

In [39]:
# ============================================================
# USC-DPO: Prepare a leakage-free preference training pool
# 80 questions x 4 candidates
# Uses the preference_pool separated in Cell 15.
# ============================================================

NUM_QUESTIONS = 80
NUM_CANDIDATES_PER_QUESTION = 4

print("===== USC-DPO PREFERENCE POOL =====")

# ------------------------------------------------------------
# 1. Validate prerequisites
# ------------------------------------------------------------

if "preference_pool" not in globals():
    raise RuntimeError(
        "preference_pool not found. Please run the updated Cell 15 first."
    )

if "gsm8k" not in globals():
    raise RuntimeError("GSM8K dataset not found.")

if len(preference_pool) != NUM_QUESTIONS:
    raise RuntimeError(
        f"Expected {NUM_QUESTIONS} preference questions, "
        f"but received {len(preference_pool)}."
    )

required_columns = {"question", "answer"}

if not required_columns.issubset(set(preference_pool.column_names)):
    raise RuntimeError(
        "preference_pool must contain question and answer columns."
    )

# ------------------------------------------------------------
# 2. Verify separation from SFT training data
# ------------------------------------------------------------

if "sft_train_final" in globals():
    sft_dataset_for_check = sft_train_final

elif "dataset_split" in globals():
    sft_dataset_for_check = dataset_split["train"]

else:
    raise RuntimeError(
        "Cannot verify SFT/preference separation. "
        "Expected sft_train_final or dataset_split from Cell 15."
    )

sft_questions = set(sft_dataset_for_check["question"])
preference_questions = set(preference_pool["question"])
test_questions = set(gsm8k["test"]["question"])

overlap_sft = sft_questions.intersection(preference_questions)
overlap_test = test_questions.intersection(preference_questions)

if overlap_sft:
    raise RuntimeError(
        f"Data leakage detected: {len(overlap_sft)} "
        "questions overlap with SFT training data."
    )

if overlap_test:
    raise RuntimeError(
        f"Test leakage detected: {len(overlap_test)} "
        "questions overlap with preference data."
    )

if len(preference_questions) != NUM_QUESTIONS:
    raise RuntimeError(
        "Duplicate questions detected in preference_pool."
    )

print("SFT/preference overlap:", len(overlap_sft))
print("Test/preference overlap:", len(overlap_test))
print("Preference questions verified:", len(preference_questions))

# ------------------------------------------------------------
# 3. Build candidate generation dataset
# ------------------------------------------------------------

candidate_data = []

for i, example in enumerate(preference_pool):

    question = str(example["question"]).strip()
    gold_answer = str(example["answer"]).strip()

    # Reuse the exact prompt created in Cell 14.
    if "prompt" in example and example["prompt"]:
        prompt = example["prompt"]

    elif "make_math_prompt" in globals():
        prompt = make_math_prompt(question)

    else:
        raise RuntimeError(
            "SFT prompt not found. "
            "Please check make_math_prompt in Cell 14."
        )

    candidate_data.append({
        "question_index": i,
        "question": question,
        "gold_answer": gold_answer,
        "prompt": prompt,
        "source_split": "gsm8k_train_preference_pool",
    })

# ------------------------------------------------------------
# 4. Final validation
# ------------------------------------------------------------

expected_candidates = (
    NUM_QUESTIONS * NUM_CANDIDATES_PER_QUESTION
)

assert len(candidate_data) == NUM_QUESTIONS

assert len({
    item["question"] for item in candidate_data
}) == NUM_QUESTIONS

assert all(
    item["prompt"] and item["gold_answer"]
    for item in candidate_data
)

print("\n===== PREFERENCE DATA SUMMARY =====")
print("Source: GSM8K train — held-out preference pool")
print("Questions:", len(candidate_data))
print("Candidates per question:", NUM_CANDIDATES_PER_QUESTION)
print("Expected candidates:", expected_candidates)

print("\n===== SAMPLE QUESTION =====")
print(candidate_data[0]["question"])

print("\n===== SAMPLE PROMPT =====")
print(candidate_data[0]["prompt"])

print("\n===== GOLD SOLUTION =====")
print(candidate_data[0]["gold_answer"])

print("\nPreference pool prepared successfully.")

===== USC-DPO PREFERENCE POOL =====
SFT/preference overlap: 0
Test/preference overlap: 0
Preference questions verified: 80

===== PREFERENCE DATA SUMMARY =====
Source: GSM8K train — held-out preference pool
Questions: 80
Candidates per question: 4
Expected candidates: 320

===== SAMPLE QUESTION =====
Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?

===== SAMPLE PROMPT =====
Solve this math word problem carefully. Show concise reasoning with the key calculations. Avoid repeated explanations and unnecessary self-talk. Check the arithmetic before answering.
End with the final answer in this exact format: #### <number>
Do not write anything after the final-answer line.

Problem:
Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells tha

In [40]:
# ============================================================
# USC-DPO — Optimized Candidate Generation
# Generate 80 x 4 candidates using the freshly trained SFT-CoT.
#
# IMPORTANT:
# - Uses trainer.model from the completed SFT training.
# - Does NOT reload the Base model for candidate generation.
# - Saves the SFT adapter before releasing GPU memory.
# - Uses a separate progress file to avoid reusing old Base outputs.
# ============================================================

import os
import gc
import json
import time
import hashlib
import torch

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"

NUM_CANDIDATES = NUM_CANDIDATES_PER_QUESTION
MAX_NEW_TOKENS = 384

TEMPERATURE = 0.85
TOP_P = 0.95
SEED = 42

ARTIFACT_DIR = "/content/USC_DPO_artifacts"

SFT_ADAPTER_DIR = os.path.join(
    ARTIFACT_DIR,
    "sft_lora_adapter"
)

# New filenames: do not reuse the old Base-generation cache.
PROGRESS_PATH = os.path.join(
    ARTIFACT_DIR,
    "candidate_generation_sft_80x4_progress.json"
)

RAW_PATH = os.path.join(
    ARTIFACT_DIR,
    "candidate_generation_sft_80x4_raw.json"
)

MANIFEST_PATH = os.path.join(
    ARTIFACT_DIR,
    "candidate_generation_sft_80x4_manifest.json"
)

os.makedirs(ARTIFACT_DIR, exist_ok=True)


# ------------------------------------------------------------
# 2. Validate prerequisites
# ------------------------------------------------------------

if "trainer" not in globals():
    raise RuntimeError(
        "SFT trainer not found. Run Cell 19 and Cell 20 first."
    )

if "candidate_data" not in globals() or not candidate_data:
    raise RuntimeError(
        "candidate_data is missing. Run the updated Cell 24 first."
    )

if "tokenizer" not in globals():
    raise RuntimeError("Tokenizer not found.")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required. Enable GPU in Colab."
    )

if len(candidate_data) != NUM_QUESTIONS:
    raise RuntimeError(
        f"Expected {NUM_QUESTIONS} questions, "
        f"received {len(candidate_data)}."
    )

if any(
    not item.get("prompt") or not item.get("question")
    for item in candidate_data
):
    raise RuntimeError(
        "Some candidate data is missing its question or prompt."
    )

if "make_math_prompt" not in globals():
    raise RuntimeError(
        "make_math_prompt is missing. Check Cell 14."
    )

# Verify that candidate prompts use exactly the prompt function
# established in Cell 14.
for item in candidate_data:
    expected_prompt = make_math_prompt(item["question"])

    if item["prompt"] != expected_prompt:
        raise RuntimeError(
            "Prompt mismatch detected. "
            "Please rerun Cell 24 using the updated Cell 14 prompt."
        )

print("===== PREREQUISITE CHECK =====")
print("Questions:", len(candidate_data))
print("Candidates per question:", NUM_CANDIDATES)
print("Expected total:", len(candidate_data) * NUM_CANDIDATES)
print("Prompt consistency: PASS")


# ------------------------------------------------------------
# 3. Get the SFT-CoT model from the completed trainer
# ------------------------------------------------------------

candidate_model = trainer.model
candidate_model.eval()

if not hasattr(candidate_model, "peft_config"):
    raise RuntimeError(
        "The current trainer model does not appear to be a PEFT model. "
        "Stop before candidate generation."
    )

sft_global_step = getattr(
    getattr(trainer, "state", None),
    "global_step",
    None
)

if not sft_global_step:
    raise RuntimeError(
        "Could not verify the completed SFT training step."
    )

if sft_global_step < 1:
    raise RuntimeError("SFT training has no completed optimizer steps.")

print("\n===== SFT MODEL VERIFIED =====")
print("Model:", MODEL_NAME)
print("Global step:", sft_global_step)
print("PEFT adapter:", list(candidate_model.peft_config.keys()))
print("Model type:", type(candidate_model))


# ------------------------------------------------------------
# 4. Save the exact SFT adapter for later USC-DPO training
# ------------------------------------------------------------

print("\n===== SAVING SFT ADAPTER =====")

candidate_model.save_pretrained(SFT_ADAPTER_DIR)
tokenizer.save_pretrained(SFT_ADAPTER_DIR)

adapter_config_path = os.path.join(
    SFT_ADAPTER_DIR,
    "adapter_config.json"
)

adapter_weight_paths = [
    os.path.join(SFT_ADAPTER_DIR, "adapter_model.safetensors"),
    os.path.join(SFT_ADAPTER_DIR, "adapter_model.bin"),
]

existing_weights = [
    path for path in adapter_weight_paths
    if os.path.isfile(path)
]

if not os.path.isfile(adapter_config_path) or not existing_weights:
    raise RuntimeError(
        "SFT adapter was not saved correctly. "
        "Do not continue to candidate generation."
    )

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


adapter_hash = sha256_file(existing_weights[0])

print("SFT adapter saved:", SFT_ADAPTER_DIR)
print("Adapter file:", os.path.basename(existing_weights[0]))
print("Adapter SHA256:", adapter_hash)


# ------------------------------------------------------------
# 5. Build generation provenance
# ------------------------------------------------------------

question_pool_hash = hashlib.sha256(
    json.dumps(
        [x["question"] for x in candidate_data],
        ensure_ascii=False
    ).encode("utf-8")
).hexdigest()

prompt_pool_hash = hashlib.sha256(
    json.dumps(
        [x["prompt"] for x in candidate_data],
        ensure_ascii=False
    ).encode("utf-8")
).hexdigest()

provenance = {
    "model": MODEL_NAME,
    "initialization": "SFT-CoT trained model from current trainer",
    "sft_global_step": int(sft_global_step),
    "sft_adapter_sha256": adapter_hash,
    "question_pool_sha256": question_pool_hash,
    "prompt_pool_sha256": prompt_pool_hash,
    "num_questions": len(candidate_data),
    "num_candidates_per_question": NUM_CANDIDATES,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "seed": SEED,
}

# ------------------------------------------------------------
# 6. Resume only matching SFT-generated progress
# ------------------------------------------------------------

multi_candidates = []

if os.path.isfile(PROGRESS_PATH):

    with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
        saved = json.load(f)

    if saved.get("provenance") == provenance:

        multi_candidates = saved.get("candidates", [])

        print(
            "\nMatching SFT generation progress found:",
            len(multi_candidates),
            "candidates"
        )

    else:
        print(
            "\nExisting progress does not match this SFT checkpoint."
        )
        print("Old progress will NOT be reused.")
        multi_candidates = []

# Check for duplicate candidate IDs before resuming.
existing_keys = [
    (
        int(c["question_index"]),
        int(c["candidate_id"])
    )
    for c in multi_candidates
]

if len(existing_keys) != len(set(existing_keys)):
    raise RuntimeError(
        "Duplicate candidate IDs found in saved progress."
    )

valid_question_ids = {
    int(item["question_index"])
    for item in candidate_data
}

if any(
    qid not in valid_question_ids
    or cid < 0
    or cid >= NUM_CANDIDATES
    for qid, cid in existing_keys
):
    raise RuntimeError(
        "Saved candidate progress contains invalid question/candidate IDs."
    )


def save_progress():

    payload = {
        "provenance": provenance,
        "candidates": multi_candidates,
    }

    with open(PROGRESS_PATH, "w", encoding="utf-8") as f:
        json.dump(
            payload,
            f,
            ensure_ascii=False,
            indent=2
        )


# ------------------------------------------------------------
# 7. Candidate generation function
# ------------------------------------------------------------

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

original_use_cache = getattr(
    candidate_model.config,
    "use_cache",
    None
)

candidate_model.config.use_cache = True


def generate_reasoning_candidates(question_item, count, seed):

    prompt = question_item["prompt"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(candidate_model.device)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    with torch.inference_mode():

        outputs = candidate_model.generate(
            **inputs,

            max_new_tokens=MAX_NEW_TOKENS,

            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,

            num_return_sequences=count,

            repetition_penalty=1.05,

            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,

            use_cache=True,
        )

    start_position = inputs["input_ids"].shape[1]

    generated_texts = [
        tokenizer.decode(
            output[start_position:],
            skip_special_tokens=True
        ).strip()
        for output in outputs
    ]

    del inputs, outputs

    return generated_texts


# ------------------------------------------------------------
# 8. Generate four candidates for each question
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("USC-DPO — SFT-CoT CANDIDATE GENERATION")
print("=" * 65)

start_time = time.time()

try:

    for position, item in enumerate(candidate_data, start=1):

        qid = int(item["question_index"])

        existing_ids = {
            int(c["candidate_id"])
            for c in multi_candidates
            if int(c["question_index"]) == qid
        }

        missing_ids = [
            cid
            for cid in range(NUM_CANDIDATES)
            if cid not in existing_ids
        ]

        if missing_ids:

            texts = generate_reasoning_candidates(
                item,
                len(missing_ids),
                SEED + qid,
            )

            if len(texts) != len(missing_ids):
                raise RuntimeError(
                    f"Expected {len(missing_ids)} outputs "
                    f"for question {qid}, got {len(texts)}."
                )

            for candidate_id, solution in zip(missing_ids, texts):

                multi_candidates.append({
                    "question_index": qid,
                    "question": item["question"],
                    "gold_answer": item["gold_answer"],
                    "candidate_id": candidate_id,
                    "generated_solution": solution,
                    "generation_model": "SFT-CoT",
                })

            # Save after every completed question.
            save_progress()

        if position % 10 == 0 or position == len(candidate_data):

            print(
                f"Questions: {position}/{len(candidate_data)}"
                f" | Candidates saved: {len(multi_candidates)}"
                f" | Elapsed: {(time.time() - start_time) / 60:.1f} min",
                flush=True
            )

finally:

    if original_use_cache is not None:
        candidate_model.config.use_cache = original_use_cache


# ------------------------------------------------------------
# 9. Validate and save final candidates
# ------------------------------------------------------------

multi_candidates.sort(
    key=lambda c: (
        int(c["question_index"]),
        int(c["candidate_id"])
    )
)

expected_total = len(candidate_data) * NUM_CANDIDATES

if len(multi_candidates) != expected_total:
    raise RuntimeError(
        f"Candidate count mismatch: "
        f"expected {expected_total}, "
        f"received {len(multi_candidates)}."
    )

final_keys = {
    (
        int(c["question_index"]),
        int(c["candidate_id"])
    )
    for c in multi_candidates
}

if len(final_keys) != expected_total:
    raise RuntimeError(
        "Duplicate or missing candidate IDs detected."
    )

if any(
    c.get("generation_model") != "SFT-CoT"
    for c in multi_candidates
):
    raise RuntimeError(
        "Candidate provenance validation failed."
    )

with open(RAW_PATH, "w", encoding="utf-8") as f:
    json.dump(
        multi_candidates,
        f,
        ensure_ascii=False,
        indent=2
    )

manifest = {
    "provenance": provenance,
    "total_candidates": len(multi_candidates),
    "generation_elapsed_minutes": round(
        (time.time() - start_time) / 60,
        2
    ),
    "raw_candidates_file": RAW_PATH,
}

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 10. Release SFT model memory
# ------------------------------------------------------------

del candidate_model

for variable_name in ("sft_model", "trainer", "model"):

    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# 11. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CANDIDATE GENERATION COMPLETED")
print("=" * 65)

print("Source model: SFT-CoT")
print("SFT global step:", sft_global_step)
print("Questions:", len(candidate_data))
print("Candidates per question:", NUM_CANDIDATES)
print("Total candidates:", len(multi_candidates))
print("Expected candidates:", expected_total)

print("\nSaved files:")
print("SFT adapter:", SFT_ADAPTER_DIR)
print("Raw candidates:", RAW_PATH)
print("Progress:", PROGRESS_PATH)
print("Manifest:", MANIFEST_PATH)

print("\n===== SAMPLE SFT-CoT CANDIDATE =====")
print(multi_candidates[0]["generated_solution"])

print("\nSFT model released from GPU memory.")
print("Ready for Cell 26.")

===== PREREQUISITE CHECK =====
Questions: 80
Candidates per question: 4
Expected total: 320
Prompt consistency: PASS

===== SFT MODEL VERIFIED =====
Model: Qwen/Qwen2.5-Math-1.5B
Global step: 1849
PEFT adapter: ['default']
Model type: <class 'peft.peft_model.PeftModelForCausalLM'>

===== SAVING SFT ADAPTER =====
SFT adapter saved: /content/USC_DPO_artifacts/sft_lora_adapter
Adapter file: adapter_model.safetensors
Adapter SHA256: d86b569741e1f8af24b104f49286b8828b1dfa4d7be665a41351f9b5b7f404b9

USC-DPO — SFT-CoT CANDIDATE GENERATION
Questions: 10/80 | Candidates saved: 40 | Elapsed: 1.4 min
Questions: 20/80 | Candidates saved: 80 | Elapsed: 2.4 min
Questions: 30/80 | Candidates saved: 120 | Elapsed: 3.6 min
Questions: 40/80 | Candidates saved: 160 | Elapsed: 4.7 min
Questions: 50/80 | Candidates saved: 200 | Elapsed: 5.5 min
Questions: 60/80 | Candidates saved: 240 | Elapsed: 6.6 min
Questions: 70/80 | Candidates saved: 280 | Elapsed: 8.0 min
Questions: 80/80 | Candidates saved: 320 | E

In [41]:
# ============================================================
# USC-DPO: Candidate Answer Verification — consolidated Cell 25
# Purpose: extract candidate final answer, compare with GSM8K gold,
# and keep ambiguous/contaminated outputs separate.
# No model loading, candidate generation, or training.
# ============================================================

import os
import re
import json
from decimal import Decimal
from fractions import Fraction
from collections import Counter

# ------------------------------
# 1. Numeric parsing utilities
# ------------------------------

NUM = r"(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?|\.\d+"
SIGNED = rf"[+-]?(?:{NUM})"
NUM_TOKEN = re.compile(rf"(?<![A-Za-z]){SIGNED}(?:\s*/\s*{SIGNED})?")
LATEX_FRACTION = re.compile(
    rf"\\(?:dfrac|tfrac|frac)\s*\{{\s*({SIGNED})\s*\}}\s*\{{\s*({SIGNED})\s*\}}"
)
ANSWER_CUE = re.compile(
    r"\b(?:final\s+answer|correct\s+answer|the\s+answer|answer)\b"
    r"\s*(?:(?:is|should\s+be|would\s+be)\s*)?(?:[:=]\s*)?",
    re.IGNORECASE,
)
CONCLUSION = re.compile(
    r"^\s*(?:therefore|thus|hence|so|overall|altogether|in\s+total|as\s+a\s+result)\b",
    re.IGNORECASE,
)
TIME_AMBIGUITY = re.compile(r"\b(?:from\s+now|ago|later)\b", re.IGNORECASE)
CHECK_CONTEXT = re.compile(
    r"\b(?:check|verification|verify|validation|calculations? are correct|sanity check)\b",
    re.IGNORECASE,
)


def parse_number(value):
    """Parse an integer, decimal or simple fraction without eval()."""
    value = str(value).strip().replace(",", "")
    try:
        if "/" in value:
            numerator, denominator = value.split("/", 1)
            denominator = Fraction(Decimal(denominator.strip()))
            if denominator == 0:
                return None
            return Fraction(Decimal(numerator.strip())) / denominator
        return Fraction(Decimal(value))
    except (ValueError, ArithmeticError, ZeroDivisionError):
        return None


def normalize_number(value):
    if value is None:
        return None
    if value.denominator == 1:
        return str(value.numerator)
    decimal_value = Decimal(value.numerator) / Decimal(value.denominator)
    if Fraction(decimal_value) == value:
        return format(decimal_value.normalize(), "f")
    return f"{value.numerator}/{value.denominator}"


def extract_boxes(text):
    """Extract nested LaTeX \boxed{...} spans, including styled boxes."""
    found = []
    for match in re.finditer(r"\\boxed\s*\{", text):
        depth = 1
        pos = match.end()
        while pos < len(text) and depth:
            if text[pos] == "{":
                depth += 1
            elif text[pos] == "}":
                depth -= 1
            pos += 1
        if depth == 0:
            found.append((text[match.end():pos - 1], match.start(), pos))
    return found


def parse_span(text):
    """Parse one numeric answer or a simple equation's final RHS."""
    text = str(text).strip()

    boxes = extract_boxes(text)
    if boxes:
        inner, _, _ = boxes[-1]
        fraction = LATEX_FRACTION.search(inner)
        if fraction:
            a = parse_number(fraction.group(1))
            b = parse_number(fraction.group(2))
            if a is not None and b not in (None, Fraction(0)):
                return a / b
        values = NUM_TOKEN.findall(inner)
        if len(values) == 1:
            return parse_number(values[0])

    fraction = LATEX_FRACTION.search(text)
    if fraction:
        a = parse_number(fraction.group(1))
        b = parse_number(fraction.group(2))
        if a is not None and b not in (None, Fraction(0)):
            return a / b

    if "=" in text:
        rhs = text.rsplit("=", 1)[1].strip()
        match = re.match(
            rf"^\s*(?:[$€£]\s*)?({SIGNED}(?:\s*/\s*{SIGNED})?)",
            rhs,
        )
        if match:
            return parse_number(match.group(1))

    values = NUM_TOKEN.findall(text)
    if len(values) == 1:
        return parse_number(values[0])
    return None


def numeric_prefix(text):
    """Parse a value only if a number begins the span."""
    match = re.match(
        rf"^\s*(?:[$€£]\s*)?({SIGNED}(?:\s*/\s*{SIGNED})?)",
        str(text).strip(),
    )
    return parse_number(match.group(1)) if match else None


def sentence_spans(text):
    """Split approximate sentence/line spans without splitting decimal points."""
    spans = []
    start = 0
    i = 0
    abbreviations = ("mrs.", "mr.", "ms.", "dr.", "e.g.", "i.e.")

    while i < len(text):
        char = text[i]
        if char == "\n":
            segment = text[start:i].strip()
            if segment:
                left = text.find(segment, start, i + 1)
                spans.append((left, left + len(segment), segment))
            start = i + 1
            i += 1
            continue

        decimal_point = (
            char == "." and i > 0 and i + 1 < len(text)
            and text[i - 1].isdigit() and text[i + 1].isdigit()
        )
        if char in ".!?" and not decimal_point and (
            i + 1 == len(text) or text[i + 1].isspace()
        ):
            segment = text[start:i + 1].strip()
            if segment:
                if segment.lower().endswith(abbreviations):
                    i += 1
                    continue
                left = text.find(segment, start, i + 2)
                spans.append((left, left + len(segment), segment))
            start = i + 1
        i += 1

    segment = text[start:].strip()
    if segment:
        left = text.find(segment, start)
        spans.append((left, left + len(segment), segment))
    return spans


def contaminated(text):
    """Detect an explicit separator that starts an unrelated second problem."""
    return bool(re.search(r"-{2,}\s*Problem\s*:", str(text), re.IGNORECASE))


def is_incomplete_tail(text):
    """Keep visibly truncated mathematical expressions unparsed."""
    stripped = str(text).rstrip()
    if not stripped:
        return True
    if re.search(
        r"(?:####\s*(?:answer)?\s*:?[ \t]*|(?:final\s+)?answer\s*:\s*)$",
        stripped,
        re.IGNORECASE,
    ):
        return True
    if stripped.endswith(("=", "+", "-", "*", "/", "×", "÷")):
        return True
    if stripped.count("$$") % 2 == 1:
        return True
    return False


# ------------------------------
# 2. Targeted answer extraction
# ------------------------------

# These patterns only recognize answer-bearing actions. A generic "is 3"
# pattern is deliberately absent because it mistakes facts like "Each letter
# is 3 pages long" for the answer.
ACTION_PATTERNS = [
    rf"\b(?:correct\s+answer|the\s+answer|answer|result)\s*(?:is|equals|would\s+be|should\s+be|:|=)\s*(?:[$€£]\s*)?({SIGNED}(?:\s*/\s*{SIGNED})?)",
    rf"\b(?:will\s+need|needs?|needed|requires?|required|should\s+read|must\s+read|will\s+read|would\s+read|will\s+take|takes?|took|paid|pay|will\s+pay|costs?|cost|sold|earns?|earned|makes?|made|gets?|got|eats?|ate|had\s+to\s+travel|should\s+travel|travels?|travelled|traveled)\s+(?:a\s+total\s+of\s+|the\s+total\s+of\s+|about\s+|approximately\s+)?(?:[$€£]\s*)?({SIGNED}(?:\s*/\s*{SIGNED})?)",
    # Narrowly capture total pages/letters/units, not generic "is N" facts.
    rf"\b(?:writes?|will\s+write|wrote)\s+(?:a\s+total\s+of\s+)?({SIGNED})\s+(?:pages?|letters?|clips?|flowers?|items?|people|hours?|minutes?|dollars?)\b",
    rf"\b(?:previous\s+monthly\s+income|original\s+monthly\s+income|monthly\s+income|annual\s+income|income|remaining\s+amount|total\s+amount|amount|cost|price|distance|time|number|total|sum|result)\s+(?:was|is|equals|equal\s+to|came\s+to|comes\s+to)\s*(?:[$€£]\s*)?({SIGNED}(?:\s*/\s*{SIGNED})?)",
]


def action_values(text):
    """Return numeric values tied to answer-bearing verbs/nouns, never generic 'is N'.

    Normalize the escaped-dollar formatting seen in generated solutions, e.g.
    ``paid $\$41$ for the shoes`` -> ``paid $41 for the shoes``.
    """
    found = []
    text = str(text)
    text = re.sub(r"\$\s*\\\$\s*(" + SIGNED + r")\s*\$", r"$\1", text)
    for pattern in ACTION_PATTERNS:
        for match in re.finditer(pattern, text, re.IGNORECASE):
            value = parse_number(match.group(1))
            if value is not None:
                found.append((match.start(1), value, match.group(0)))
    return sorted(found, key=lambda item: item[0])


def extract_final_answer(text):
    """Return numeric final answer + evidence, or None if evidence is insufficient."""
    text = "" if text is None else str(text)

    # If a clear answer-action appears before a later checking appendix,
    # ignore the appendix for answer extraction. Otherwise its arithmetic
    # can override the actual answer (e.g. shoe cost 41 vs total spent 184).
    check_match = re.search(r"\bcheck\s*:", text, re.IGNORECASE)
    if check_match:
        answer_prefix = text[:check_match.start()].rstrip()
        if action_values(answer_prefix):
            text = answer_prefix

    if not text.strip() or contaminated(text) or is_incomplete_tail(text):
        return None

    spans = sentence_spans(text)
    candidates = []

    def add(position, strength, value, method, evidence):
        if value is not None:
            candidates.append((
                position, strength, normalize_number(value), method,
                str(evidence)[:180],
            ))

    def sentence_end(position):
        for start, end, _ in spans:
            if start <= position < end:
                return end
        return len(text)

    def near_tail(position):
        # Don't take an intermediate statement from early reasoning unless
        # it is explicitly marked as the answer.
        return position >= max(0.40 * len(text), len(text) - 1400)

    # A. Explicit answer cues: inspect the relevant sentence, allowing a later
    # "correct answer is 52.50" to supersede boilerplate in "Final Answer: ...".
    for match in ANSWER_CUE.finditer(text):
        end = sentence_end(match.start())
        clause = text[match.end():end]
        actions = action_values(clause)
        value = actions[-1][1] if actions else parse_span(clause)

        if value is None:
            nums = NUM_TOKEN.findall(clause)
            if len(nums) == 1:
                value = parse_number(nums[0])

        add(match.start(), 100, value, "explicit_answer_cue", text[match.start():end])

    # B. A numeric answer marker accepts #### 72, #### <72>, but not a sentence
    # such as #### James writes ... 208 letters.
    for match in re.finditer(r"####\s*", text):
        line = text[match.end():].split("\n", 1)[0].strip()
        bracketed = re.match(rf"^<\s*({SIGNED})\s*>", line)
        numeric = re.match(rf"^({SIGNED})(?:\s|$|[.!])", line)
        if bracketed:
            value = parse_number(bracketed.group(1))
            add(match.start(), 95, value, "gsm8k_bracketed_marker", "#### " + line)
        elif numeric:
            value = parse_number(numeric.group(1))
            add(match.start(), 95, value, "gsm8k_numeric_marker", "#### " + line)

    # C. Boxed answers, but ignore boxes that only appear early in a long worked
    # solution. A later final answer cue still wins by position.
    for inner, start, end in extract_boxes(text):
        if start < max(0.25 * len(text), len(text) - 1800):
            continue
        value = parse_span(inner)
        if value is None:
            nums = NUM_TOKEN.findall(inner)
            if len(nums) == 1:
                value = parse_number(nums[0])
        add(start, 90, value, "boxed", text[start:end])

    # D. Look only near the end for conclusions, answer actions, and equations.
    # Explicit leading numeric conclusions handle e.g. "Therefore, 36 students
    # in Mr. Sanchez's Grade 5 class ..." without mistaking grade 5 for answer.
    for start, end, sentence in spans:
        if CHECK_CONTEXT.search(sentence):
            continue
        eligible = near_tail(start) or bool(CONCLUSION.search(sentence))
        if not eligible:
            continue

        if CONCLUSION.search(sentence):
            after_intro = re.sub(
                r"^\s*(?:therefore|thus|hence|so|overall|altogether|in\s+total|as\s+a\s+result)\b\s*,?\s*",
                "",
                sentence,
                flags=re.IGNORECASE,
            )
            leading = re.match(rf"^(?:[$€£]\s*)?({SIGNED})\b", after_intro.strip())
            if leading:
                add(start, 75, parse_number(leading.group(1)), "conclusion_leading_number", sentence)
                continue

        # Equations only near the end and only with a result-related word.
        if "=" in sentence and re.search(
            r"\b(?:total|answer|result|pages|year|amount|cost|price|flowers|clips|hours|income|shoes|letters|distance|students|difference)\b",
            sentence,
            re.IGNORECASE,
        ):
            value = parse_span(sentence)
            add(start, 70, value, "result_equation", sentence)
            continue

        actions = action_values(sentence)
        if actions:
            add(start, 68, actions[-1][1], "answer_action", sentence)
            continue

        if CONCLUSION.search(sentence) and not TIME_AMBIGUITY.search(sentence):
            # Conservative fallback: only one numeric value in the conclusion.
            nums = NUM_TOKEN.findall(sentence)
            if len(nums) == 1:
                add(start, 65, parse_number(nums[0]), "conclusion", sentence)

    # E. Final standalone value/unit only. Never search the whole explanation
    # for the last arbitrary numeric token.
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        line = lines[-1]
        position = text.rfind(line)
        if not TIME_AMBIGUITY.search(line):
            standalone = re.fullmatch(
                rf"\s*(?:[$€£]\s*)?{SIGNED}"
                rf"(?:\s*(?:hours?|minutes?|days?|weeks?|months?|years?|clips?|pages?|dollars?|items?|people|miles?|meters?|kg|g|%)?)?\s*[.!]?\s*",
                line,
                re.IGNORECASE,
            )
            if standalone:
                add(position, 10, parse_span(line), "final_standalone_value", line)
            elif (
                "=" in line
                and re.fullmatch(r"[\s\d,./+*()=\-]+", line)
            ):
                # Accept a clean standalone final equation such as 48 + 24 = 72.
                # This does not inspect arbitrary equation lines in the middle.
                add(position, 10, parse_span(line), "final_standalone_equation", line)

    if not candidates:
        return None

    # Latest reliable answer-like statement wins. Strength resolves ties only;
    # no generic fact of the form "X is 3" is ever admitted as an answer.
    best = max(candidates, key=lambda item: (item[0], item[1]))
    return {
        "value": best[2],
        "method": best[3],
        "evidence": best[4],
    }


def extract_gold_final_answer(text):
    """Gold GSM8K answer is the numeric value immediately after ####."""
    text = "" if text is None else str(text)
    for match in reversed(list(re.finditer(r"####\s*", text))):
        line = text[match.end():].split("\n", 1)[0].strip()
        bracketed = re.match(rf"^<\s*({SIGNED})\s*>", line)
        numeric = re.match(rf"^(?:[$€£]\s*)?({SIGNED})(?:\s|$|[.!])", line)
        if bracketed:
            return {
                "value": normalize_number(parse_number(bracketed.group(1))),
                "method": "gold_gsm8k_marker",
                "evidence": "#### " + line,
            }
        if numeric:
            value = parse_number(numeric.group(1))
            if value is not None:
                return {
                    "value": normalize_number(value),
                    "method": "gold_gsm8k_marker",
                    "evidence": "#### " + line,
                }
    return None


def extract_final_number(text):
    """Compatibility wrapper for downstream cells."""
    result = extract_final_answer(text)
    return result["value"] if result else None


# ============================================================
# 3. Regression tests — includes failure modes seen in outputs v4-v7
# ============================================================

def run_answer_parser_tests():
    cases = [
        ("GSM8K marker", "Working...\n#### 1,200", "1200"),
        ("Boxed answer", r"Final answer: \boxed{72}.", "72"),
        ("Explicit answer", "The answer is 64 glasses.", "64"),
        ("Pizza conclusion", "Therefore, Albert eats 48 pieces of pizza that day.", "48"),
        ("Standalone unit", "5 hours", "5"),
        ("Standalone number", "72", "72"),
        ("Fraction", r"The answer is \(\frac{1}{2}\).", "0.5"),
        ("Reject time phrase", "10 years from now.", None),
        ("Reject incidental number", "The meeting is in 4 weeks.", None),
        ("Reject truncated answer", "The final result may be", None),
        ("Reject dangling equation", "The total is 16 pieces -", None),
        ("Equation result", "48 + 24 = 72", "72"),
        ("Natural-language answer", "Answer: Natalia sold 72 clips altogether in April and May.", "72"),
        ("Intermediate conclusion then final answer", "Therefore, Betty needs $45 more. Answer: Betty needs $5 more.", "5"),
        ("Boilerplate final answer", "Final Answer: The final answer should be in the format <number>, so the correct answer is 52.50.", "52.5"),
        ("Answer after marker", "#### Answer: Julie should read 42 pages tomorrow.", "42"),
        ("Incomplete marker", "Yesterday she read 42 pages.\n#### Answer:", None),
        ("Styled boxed answer", r"Answer: $\boxed{\color{blue}{45}}$", "45"),
        ("Joy's time answer, not page count", "Final Answer: Joy will need 5 hours to read 120 pages of the book.", "5"),
        ("Roque's time answer", "Answer: Roque takes a total of 8 hours to get to and from work a week.", "8"),
        ("Mrs. Snyder's income", "Therefore, Mrs. Snyder's previous monthly income was $1000$.", "1000"),
        ("Julie final answer not intermediate 36", r"In total, she has read \(12 + 24 = 36\) pages so far. The book has 120 pages. She needs \(84 \div 2 = 42\) pages. #### Answer: Julie should read 42 pages tomorrow.", "42"),
        ("Q4/C2 final pages", "Therefore, James writes 312 pages in a year.", "312"),
        ("Q4/C3 does not mistake page-per-letter fact for answer", "First, determine the pages written. Each letter is 3 pages long. He writes twice a week. To find the total number of pages written in a year, break down the weekly task and extend the calculation carefully.", None),
        ("Mean is an intermediate fact, not answer", "The mean of the original data set is given by the formula. Since the mean is 17, we have: mean = (x1 + x2 + x3 + x4 + x5) / 5. Now let's find the mean of the new data set.", None),
        ("Conclusion with incidental grade number", "Therefore, 36 students in Mr. Sanchez's Grade 5 class got a final grade of B and above.", "36"),
        ("Car travel answer with turn number", "Answer: The car had to travel 10 meters after the 3rd turn.", "10"),
        ("Candy marker in angle brackets", "Difference = Anna's total - Billy's total = 840 - 825 = 15\n#### <15>\nAnna gets 15 more pieces of candy than Billy.", "15"),
        ("Answer before checking paragraph", "So, Alexis paid $41 for the shoes.\nCheck: The sum of the costs of all items is $184.\n$184 = $184. The calculations are correct.", "41"),
        ("Actual Q8/C1 escaped-dollar answer before checking appendix", r"So, Alexis paid $\$41$ for the shoes. Check: The sum of the costs of all items should equal the total amount spent. $30 + $46 + $38 + $11 + $18 + $41 = $184", "41"),
        ("Monthly income equation", r"The equation is \(0.4X = 0.25(X + 600)\). Solving gives \(X = 1000\). Therefore, Mrs. Snyder's previous monthly income was $1000$.", "1000"),
    ]

    failures = []
    for name, sample, expected in cases:
        result = extract_final_answer(sample)
        actual = result["value"] if result else None
        passed = actual == expected
        print(
            f"{'PASS' if passed else 'FAIL'} | {name} | "
            f"expected={expected} | actual={actual} | "
            f"method={result['method'] if result else None}"
        )
        if not passed:
            failures.append(name)

    contaminated_example = (
        "Natalia sold 72 clips.\n--- Problem: Find an unknown number.\n"
        "Therefore, the unknown number is 40."
    )
    detected = contaminated(contaminated_example)
    print(f"{'PASS' if detected else 'FAIL'} | Detect embedded second problem")
    if not detected:
        failures.append("Detect embedded second problem")

    gold_cases = [
        ("Gold GSM8K integer", "working...\n#### 624", "624"),
        ("Gold currency", "calculation\n#### $5", "5"),
        ("Gold angle marker", "calculation\n#### <15>", "15"),
        ("Do not guess gold without marker", "the result is 624", None),
    ]
    for name, sample, expected in gold_cases:
        result = extract_gold_final_answer(sample)
        actual = result["value"] if result else None
        passed = actual == expected
        print(f"{'PASS' if passed else 'FAIL'} | {name} | expected={expected} | actual={actual}")
        if not passed:
            failures.append(name)

    if failures:
        raise AssertionError("Answer parser tests failed: " + ", ".join(failures))
    print("ALL ANSWER-PARSER REGRESSION TESTS PASS")


run_answer_parser_tests()


# ============================================================
# 4. Verify the existing candidates against gold answers
# ============================================================

if "multi_candidates" not in globals() or not multi_candidates:
    raise RuntimeError(
        "multi_candidates is missing or empty. Restore the existing candidate pool; "
        "do not regenerate candidates."
    )

verified_candidates = []
for original_candidate in multi_candidates:
    candidate = original_candidate.copy()
    solution = str(candidate.get("generated_solution", ""))
    gold_text = str(candidate.get("gold_answer", ""))
    has_contamination = contaminated(solution)

    predicted_result = None if has_contamination else extract_final_answer(solution)
    gold_result = extract_gold_final_answer(gold_text)

    predicted = predicted_result["value"] if predicted_result else None
    gold = gold_result["value"] if gold_result else None

    candidate["predicted_answer"] = predicted
    candidate["gold_final_answer"] = gold
    candidate["answer_extraction_method"] = predicted_result["method"] if predicted_result else None
    candidate["answer_extraction_evidence"] = predicted_result["evidence"] if predicted_result else None
    candidate["gold_extraction_method"] = gold_result["method"] if gold_result else None
    candidate["data_quality_status"] = "CONTAMINATED" if has_contamination else "OK"

    if has_contamination:
        candidate["is_correct"] = None
        candidate["verification_status"] = "UNPARSED"
        candidate["verification_reason"] = (
            "Generated output contains a second-problem marker; "
            "comparison against the original gold answer is unsafe."
        )
    elif predicted is None or gold is None:
        candidate["is_correct"] = None
        candidate["verification_status"] = "UNPARSED"
        candidate["verification_reason"] = (
            "Could not confidently extract one numeric final answer or the gold answer."
        )
    elif predicted == gold:
        candidate["is_correct"] = True
        candidate["verification_status"] = "CORRECT"
        candidate["verification_reason"] = (
            "Extracted numeric answer matches GSM8K gold answer."
        )
    else:
        candidate["is_correct"] = False
        candidate["verification_status"] = "INCORRECT"
        candidate["verification_reason"] = (
            "Extracted numeric answer differs from GSM8K gold answer."
        )

    verified_candidates.append(candidate)


# ============================================================
# 5. Summary and audit examples
# ============================================================

num_total = len(verified_candidates)
num_correct = sum(c["verification_status"] == "CORRECT" for c in verified_candidates)
num_incorrect = sum(c["verification_status"] == "INCORRECT" for c in verified_candidates)
num_unparsed = sum(c["verification_status"] == "UNPARSED" for c in verified_candidates)
num_contaminated = sum(c["data_quality_status"] == "CONTAMINATED" for c in verified_candidates)
num_parsed = num_correct + num_incorrect
accuracy = 100 * num_correct / num_parsed if num_parsed else 0.0
method_counts = Counter(c["answer_extraction_method"] or "UNPARSED" for c in verified_candidates)

print("\n===== USC-DPO ANSWER VERIFICATION SUMMARY =====")
print("Total candidates:", num_total)
print("Correct candidates:", num_correct)
print("Incorrect candidates:", num_incorrect)
print("Unparsed candidates:", num_unparsed)
print("Contaminated outputs (included in UNPARSED):", num_contaminated)
print("Parsed candidates:", num_parsed)
print(f"Accuracy among parsed candidates: {accuracy:.2f}%")

print("\n===== EXTRACTION METHODS =====")
for method, count in method_counts.most_common():
    print(f"{method}: {count}")


def print_audit_samples(candidates, status, limit=5):
    samples = [c for c in candidates if c["verification_status"] == status]
    print(f"\n===== {status} SAMPLES (up to {limit}) =====")
    for candidate in samples[:limit]:
        solution = str(candidate.get("generated_solution", ""))
        print(
            f"\nQ{candidate.get('question_index')} / C{candidate.get('candidate_id')} | "
            f"Predicted: {candidate['predicted_answer']} | Gold: {candidate['gold_final_answer']} | "
            f"Method: {candidate['answer_extraction_method']} | Quality: {candidate['data_quality_status']}"
        )
        print("Reason:", candidate["verification_reason"])
        print("Evidence:", candidate["answer_extraction_evidence"])
        print("Solution start:", solution[:250].replace("\n", " "))
        print("Solution tail:", solution[-450:].replace("\n", " "))


print_audit_samples(verified_candidates, "INCORRECT")
print_audit_samples(verified_candidates, "UNPARSED")

contaminated_examples = [c for c in verified_candidates if c["data_quality_status"] == "CONTAMINATED"]
print("\n===== CONTAMINATED OUTPUTS (up to 5) =====")
for candidate in contaminated_examples[:5]:
    print(
        f"Q{candidate.get('question_index')} / C{candidate.get('candidate_id')} | "
        f"Gold: {candidate['gold_final_answer']}"
    )
    print("Reason:", candidate["verification_reason"])
    print("Solution tail:", str(candidate.get("generated_solution", ""))[-450:].replace("\n", " "))


# ============================================================
# 6. Save reproducible audit artifacts
# ============================================================

ARTIFACT_DIR = "/content/USC_DPO_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)
audit_path = os.path.join(ARTIFACT_DIR, "candidate_verification_audit.json")
summary_path = os.path.join(ARTIFACT_DIR, "candidate_verification_summary.json")

with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(verified_candidates, f, ensure_ascii=False, indent=2)

summary = {
    "method": "targeted_final_answer_extraction_vs_gsm8k_gold",
    "total_candidates": num_total,
    "correct_candidates": num_correct,
    "incorrect_candidates": num_incorrect,
    "unparsed_candidates": num_unparsed,
    "contaminated_candidates": num_contaminated,
    "parsed_candidates": num_parsed,
    "accuracy_among_parsed_candidates_percent": round(accuracy, 2),
    "extraction_method_counts": dict(method_counts),
    "regression_tests": "PASS",
    "audit_path": audit_path,
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n===== SAVED ARTIFACTS =====")
print("Candidate audit:", audit_path)
print("Verification summary:", summary_path)
print("Candidates preserved:", len(verified_candidates))

PASS | GSM8K marker | expected=1200 | actual=1200 | method=gsm8k_numeric_marker
PASS | Boxed answer | expected=72 | actual=72 | method=boxed
PASS | Explicit answer | expected=64 | actual=64 | method=explicit_answer_cue
PASS | Pizza conclusion | expected=48 | actual=48 | method=answer_action
PASS | Standalone unit | expected=5 | actual=5 | method=final_standalone_value
PASS | Standalone number | expected=72 | actual=72 | method=final_standalone_value
PASS | Fraction | expected=0.5 | actual=0.5 | method=explicit_answer_cue
PASS | Reject time phrase | expected=None | actual=None | method=None
PASS | Reject incidental number | expected=None | actual=None | method=None
PASS | Reject truncated answer | expected=None | actual=None | method=None
PASS | Reject dangling equation | expected=None | actual=None | method=None
PASS | Equation result | expected=72 | actual=72 | method=final_standalone_equation
PASS | Natural-language answer | expected=72 | actual=72 | method=explicit_answer_cue
PASS |

<>:217: SyntaxWarning: invalid escape sequence '\$'
<>:217: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_514/1668949817.py:217: SyntaxWarning: invalid escape sequence '\$'
  ``paid $\$41$ for the shoes`` -> ``paid $41 for the shoes``.


# USC-DPO — STEP FILTERING

In [42]:
# USC-DPO: Step Segmentation — LaTeX-aware version

import re

def is_formatting_only_line(line):
    """True only when a line contains structural formatting markers."""
    line = line.strip()

    if not line:
        return True

    patterns = [
        r"\\\[", r"\\\]", r"\\\(", r"\\\)",
        r"\$\$?", r"```",
        r"\\begin\{(?:aligned|array|cases|gathered|equation)\}",
        r"\\end\{(?:aligned|array|cases|gathered|equation)\}",
    ]

    return any(re.fullmatch(p, line) for p in patterns)


def split_reasoning_steps(reasoning):
    """
    Split reasoning into steps while excluding standalone LaTeX delimiters.
    Mathematical expressions and explanatory text are retained.
    """
    raw_lines = str(reasoning).splitlines()

    lines = [
        line.strip()
        for line in raw_lines
        if line.strip() and not is_formatting_only_line(line)
    ]

    if not lines:
        return []

    numbered_pattern = r"^(?:\d+\.\s+|Step\s+\d+\s*:\s*)"

    has_numbered_steps = any(
        re.match(numbered_pattern, line, re.IGNORECASE)
        for line in lines
    )

    if has_numbered_steps:
        steps = []
        current_step = []

        for line in lines:
            is_new_step = re.match(
                numbered_pattern, line, re.IGNORECASE
            )

            if is_new_step:
                if current_step:
                    steps.append(" ".join(current_step))
                current_step = [line]
            elif current_step:
                current_step.append(line)
            else:
                current_step = [line]

        if current_step:
            steps.append(" ".join(current_step))

        return steps

    return lines


# Existing sample test — no model loading or generation
sample_candidate = verified_candidates[0]
sample_steps = split_reasoning_steps(
    sample_candidate["generated_solution"]
)

print("===== STEP SEGMENTATION =====")
for i, step in enumerate(sample_steps, start=1):
    print(f"\nStep {i}:")
    print(step)

print("\nTotal detected steps:", len(sample_steps))

===== STEP SEGMENTATION =====

Step 1:
Mimi picked up <<2*12=<<2*12=<<2*12=24>>2>>2>>24>>24>>24 shells on the beach.

Step 2:
Kyle found <<24*2=<<24*2=<<24*2=48>>48>>48>>48 shells in his pocket.

Step 3:
Leigh grabbed one-third of the shells that <<48/3=<<48/3=<<48/3=16>>16>>16>>16 shells that <<48/3=<<48/3=<<48/3=16>>16>>16>>16 = <<16-8=<<16-8=<<16-8=8>>8>>8>>8 shells that <<48/3=<<48/3=<<48/3=16>>16>>16>>16 = <<16-8=<<16-8=<<16-8=8>>8>>8>>8 shells that <<16-8=<<16-8=<<16-8=8>>8>>8>>8 shells that <<16-8=<<16-8=<<16-8=8>>8>>8>>8 shells that <<16-8=<<16-8=<<16-8=8>>8>>8>>8 shells that <<16-8=<<16-8=<<16-8=8>>8>>8>>8 shells that <<16-8=<<16-8=<<16-8=8>>8>>8>>8 shells that <<16-8=<<16-8=<<16-8

Total detected steps: 3


In [43]:
# USC-DPO: Redundant Step Filtering — conservative version

def normalize_step_text(text):
    """Normalize text without turning formatting-only strings into matches."""
    text = str(text).lower().strip()

    # Remove standalone LaTeX wrappers and common formatting commands.
    for marker in (
        r"\[", r"\]", r"\(", r"\)",
        r"\begin{aligned}", r"\end{aligned}",
        r"\begin{array}", r"\end{array}",
    ):
        text = text.replace(marker, " ")

    text = text.replace(r"\times", "*")
    text = text.replace(r"\cdot", "*")
    text = text.replace(r"\div", "/")
    text = text.replace(r"\%", "%")

    # Ignore Markdown/LaTeX decoration, but retain numbers and operators.
    text = re.sub(r"[`$]", "", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s+\-*/=.,<>()%]", "", text)

    return text.strip()


def is_redundant_step(step, previous_steps):
    """
    Remove only generic introductions and nonempty exact duplicates.
    Empty normalized strings must never count as duplicate reasoning.
    """
    text = str(step).lower().strip()

    introductory_patterns = [
        r"^to solve this problem",
        r"^to determine",
        r"^let'?s break",
        r"^let'?s calculate",
        r"^first,? let'?s",
        r"^we need to",
        r"^we are asked to",
        r"^in order to solve",
    ]

    has_number = bool(re.search(r"\d", text))
    has_operation = bool(re.search(r"[\+\-\*/=]", text))

    for pattern in introductory_patterns:
        if (
            re.search(pattern, text)
            and not has_number
            and not has_operation
        ):
            return True

    normalized_current = normalize_step_text(step)

    # Critical safeguard: never remove a step because both strings
    # normalize to an empty string.
    if not normalized_current:
        return False

    for previous in previous_steps:
        normalized_previous = normalize_step_text(previous)

        if (
            normalized_previous
            and normalized_current == normalized_previous
        ):
            return True

    return False


def filter_reasoning_steps(steps):
    filtered_steps = []
    removed_steps = []

    for step in steps:
        if is_redundant_step(step, filtered_steps):
            removed_steps.append(step)
        else:
            filtered_steps.append(step)

    return filtered_steps, removed_steps


# Test on the existing first candidate
sample_steps = split_reasoning_steps(
    verified_candidates[0]["generated_solution"]
)

filtered_steps, removed_steps = filter_reasoning_steps(sample_steps)

print("===== ORIGINAL STEPS =====")
for i, step in enumerate(sample_steps, start=1):
    print(f"\nStep {i}: {step}")

print("\n===== FILTERED STEPS =====")
for i, step in enumerate(filtered_steps, start=1):
    print(f"\nStep {i}: {step}")

print("\nOriginal step count:", len(sample_steps))
print("Filtered step count:", len(filtered_steps))
print("Removed step count:", len(removed_steps))
print("Removed steps:", removed_steps)

===== ORIGINAL STEPS =====

Step 1: Mimi picked up <<2*12=<<2*12=<<2*12=24>>2>>2>>24>>24>>24 shells on the beach.

Step 2: Kyle found <<24*2=<<24*2=<<24*2=48>>48>>48>>48 shells in his pocket.

Step 3: Leigh grabbed one-third of the shells that <<48/3=<<48/3=<<48/3=16>>16>>16>>16 shells that <<48/3=<<48/3=<<48/3=16>>16>>16>>16 = <<16-8=<<16-8=<<16-8=8>>8>>8>>8 shells that <<48/3=<<48/3=<<48/3=16>>16>>16>>16 = <<16-8=<<16-8=<<16-8=8>>8>>8>>8 shells that <<16-8=<<16-8=<<16-8=8>>8>>8>>8 shells that <<16-8=<<16-8=<<16-8=8>>8>>8>>8 shells that <<16-8=<<16-8=<<16-8=8>>8>>8>>8 shells that <<16-8=<<16-8=<<16-8=8>>8>>8>>8 shells that <<16-8=<<16-8=<<16-8=8>>8>>8>>8 shells that <<16-8=<<16-8=<<16-8

===== FILTERED STEPS =====

Step 1: Mimi picked up <<2*12=<<2*12=<<2*12=24>>2>>2>>24>>24>>24 shells on the beach.

Step 2: Kyle found <<24*2=<<24*2=<<24*2=48>>48>>48>>48 shells in his pocket.

Step 3: Leigh grabbed one-third of the shells that <<48/3=<<48/3=<<48/3=16>>16>>16>>16 shells that <<48/3=<<4

In [44]:
# USC-DPO: Apply Step Filtering + Audit Removed Steps
# Replace the existing Cell 29. Do not create another cell.

filtered_candidates = []

total_original_steps = 0
total_filtered_steps = 0
total_removed_steps = 0


# Explain why a step triggered the existing filtering rule.
def explain_step_removal(step, previous_steps):
    text = step.lower().strip()

    introductory_patterns = [
        r"^to solve this problem",
        r"^to determine",
        r"^let'?s break",
        r"^let'?s calculate",
        r"^first,? let'?s",
        r"^we need to",
        r"^we are asked to",
        r"^in order to solve",
    ]

    has_number = bool(re.search(r"\d", text))
    has_operation = bool(re.search(r"[\+\-\*/=]", text))

    for pattern in introductory_patterns:
        if (
            re.search(pattern, text)
            and not has_number
            and not has_operation
        ):
            return "Introductory text without numbers or operations"

    normalized_current = normalize_step_text(step)

    for previous in previous_steps:
        if normalized_current == normalize_step_text(previous):
            return "Exact normalized duplicate of a retained step"

    return "Triggered existing redundancy filter; reason unclear"


for candidate in verified_candidates:

    # Split the original reasoning.
    steps = split_reasoning_steps(
        candidate["generated_solution"]
    )

    # Apply the existing filter and record the removed steps.
    filtered_steps, removed_steps = filter_reasoning_steps(steps)

    # Replay the same filtering rule to preserve each removed step's
    # original position and a human-readable explanation.
    retained_so_far = []
    removed_details = []

    for original_index, step in enumerate(steps, start=1):
        if is_redundant_step(step, retained_so_far):
            removed_details.append({
                "original_step_number": original_index,
                "text": step,
                "reason": explain_step_removal(
                    step,
                    retained_so_far
                ),
            })
        else:
            retained_so_far.append(step)

    result = candidate.copy()

    result["steps"] = steps
    result["filtered_steps"] = filtered_steps
    result["removed_steps"] = removed_steps
    result["removed_step_details"] = removed_details

    result["original_step_count"] = len(steps)
    result["filtered_step_count"] = len(filtered_steps)
    result["removed_step_count"] = len(removed_steps)

    filtered_candidates.append(result)

    total_original_steps += len(steps)
    total_filtered_steps += len(filtered_steps)
    total_removed_steps += len(removed_steps)


# =========================================================
# Summary
# =========================================================

print("===== STEP FILTERING SUMMARY =====")
print("Total candidates:", len(filtered_candidates))
print("Original steps:", total_original_steps)
print("Filtered steps:", total_filtered_steps)
print("Removed steps:", total_removed_steps)

if total_original_steps:
    print(
        "Step reduction rate:",
        f"{total_removed_steps / total_original_steps:.2%}"
    )


print("\n===== CANDIDATE RESULTS =====")

for candidate in filtered_candidates:
    print(
        f"Q{candidate['question_index']} | "
        f"C{candidate['candidate_id']} | "
        f"Correct: {candidate.get('is_correct')} | "
        f"Steps: {candidate['original_step_count']} -> "
        f"{candidate['filtered_step_count']} | "
        f"Removed: {candidate['removed_step_count']}"
    )


# =========================================================
# Audit candidates with substantial step reduction
# =========================================================

print("\n===== REMOVED-STEP AUDIT =====")

audit_targets = {
    (8, 3),
    (29, 2),
    (39, 3),
}

# Audit every candidate that actually lost one or more steps.
audit_candidates = [
    candidate
    for candidate in filtered_candidates
    if int(candidate.get("removed_step_count", 0)) > 0
]

print(
    "Candidates with removed steps:",
    len(audit_candidates)
)

if not audit_candidates:
    print("No candidates met the audit criteria.")

for candidate in audit_candidates:
    print("\n" + "=" * 80)

    print(
        f"Q{candidate['question_index']} / "
        f"C{candidate['candidate_id']} | "
        f"Correct: {candidate.get('is_correct')} | "
        f"Removed: {candidate['removed_step_count']}"
    )

    for detail in candidate["removed_step_details"]:
        print(
            f"\n[REMOVED ORIGINAL STEP "
            f"{detail['original_step_number']}]"
        )
        print("Reason:", detail["reason"])
        print("Text:", detail["text"])

print("\n===== END REMOVED-STEP AUDIT =====")

===== STEP FILTERING SUMMARY =====
Total candidates: 320
Original steps: 1397
Filtered steps: 1392
Removed steps: 5
Step reduction rate: 0.36%

===== CANDIDATE RESULTS =====
Q0 | C0 | Correct: None | Steps: 3 -> 3 | Removed: 0
Q0 | C1 | Correct: True | Steps: 4 -> 4 | Removed: 0
Q0 | C2 | Correct: True | Steps: 4 -> 4 | Removed: 0
Q0 | C3 | Correct: True | Steps: 4 -> 4 | Removed: 0
Q1 | C0 | Correct: None | Steps: 7 -> 7 | Removed: 0
Q1 | C1 | Correct: False | Steps: 2 -> 2 | Removed: 0
Q1 | C2 | Correct: False | Steps: 5 -> 5 | Removed: 0
Q1 | C3 | Correct: False | Steps: 6 -> 6 | Removed: 0
Q2 | C0 | Correct: True | Steps: 7 -> 7 | Removed: 0
Q2 | C1 | Correct: False | Steps: 10 -> 10 | Removed: 0
Q2 | C2 | Correct: False | Steps: 6 -> 6 | Removed: 0
Q2 | C3 | Correct: False | Steps: 4 -> 4 | Removed: 0
Q3 | C0 | Correct: False | Steps: 4 -> 4 | Removed: 0
Q3 | C1 | Correct: True | Steps: 6 -> 6 | Removed: 0
Q3 | C2 | Correct: True | Steps: 5 -> 5 | Removed: 0
Q3 | C3 | Correct: Tru

In [50]:

# USC-DPO: Evidence-Based Verification V3 — Arithmetic-Gated
# Physical Cell 31
# Replaces the previous semantic-verifier pilot.
#
# IMPORTANT:
# VALID   = explicit arithmetic evidence passed; NOT semantic certification.
# INVALID = at least one explicit numeric equality is mathematically false.
# UNKNOWN = insufficient safe arithmetic evidence.
# REDUNDANT = exact normalized duplicate of an earlier retained step.
#
# No verifier LLM is loaded. This avoids unreliable semantic INVALID verdicts.

import ast
import re
import operator
import gc
from fractions import Fraction

VERIFIER_VERSION = "usc_evidence_based_verifier_v3_arithmetic_gated"
LABELS = ["VALID", "INVALID", "REDUNDANT", "UNKNOWN"]

# ============================================================
# 1. Safe arithmetic evaluator
# ============================================================

_ALLOWED_BINARY_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
}

_ALLOWED_UNARY_OPS = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}


def usc_eval_arithmetic_ast(node):
    if isinstance(node, ast.Expression):
        return usc_eval_arithmetic_ast(node.body)

    if isinstance(node, ast.Constant):
        value = node.value
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise ValueError("Unsupported constant")
        if isinstance(value, float):
            return Fraction(str(value))
        return Fraction(value)

    if isinstance(node, ast.BinOp):
        op_type = type(node.op)

        if op_type not in _ALLOWED_BINARY_OPS:
            raise ValueError("Unsupported operator")

        left = usc_eval_arithmetic_ast(node.left)
        right = usc_eval_arithmetic_ast(node.right)

        if op_type is ast.Pow:
            if right.denominator != 1 or abs(right.numerator) > 10:
                raise ValueError("Unsupported exponent")

        if op_type is ast.Mod:
            if left.denominator != 1 or right.denominator != 1:
                raise ValueError("Modulo requires integers")

        return _ALLOWED_BINARY_OPS[op_type](left, right)

    if isinstance(node, ast.UnaryOp):
        op_type = type(node.op)

        if op_type not in _ALLOWED_UNARY_OPS:
            raise ValueError("Unsupported unary operator")

        return _ALLOWED_UNARY_OPS[op_type](
            usc_eval_arithmetic_ast(node.operand)
        )

    raise ValueError("Unsupported expression")


def usc_safe_eval_arithmetic(expression):
    expression = str(expression).strip()
    expression = expression.replace(",", "")
    expression = expression.replace("^", "**")
    expression = expression.replace("×", "*").replace("÷", "/")

    if not re.fullmatch(r"[\d\s()+\-*/.%]+", expression):
        raise ValueError("Not a pure numeric expression")

    # Percentages are not interpreted implicitly.
    if "%" in expression:
        raise ValueError("Percentage syntax is not supported")

    tree = ast.parse(expression, mode="eval")
    return usc_eval_arithmetic_ast(tree)


# ============================================================
# 2. Extract explicit arithmetic evidence
# ============================================================

def usc_extract_gsm8k_equations(text):
    """
    Extract innermost GSM8K markers such as:
        <<2*12=24>>
        <<48/3=16>>

    Nested malformed markers are handled conservatively:
    only innermost complete markers are extracted.
    """
    text = str(text)
    equations = []

    pattern = re.compile(r"<<([^<>]+)>>")

    for match in pattern.finditer(text):
        chunk = match.group(1).strip()
        if "=" in chunk:
            equations.append(chunk)

    return equations


def usc_normalize_latex_math(text):
    text = str(text)

    # Simple fractions: \frac{a}{b}, \dfrac{a}{b}, \tfrac{a}{b}
    fraction_pattern = re.compile(
        r"\\(?:dfrac|tfrac|frac)\s*\{([^{}]+)\}\s*\{([^{}]+)\}"
    )

    for _ in range(8):
        updated = fraction_pattern.sub(r"((\1)/(\2))", text)
        if updated == text:
            break
        text = updated

    text = re.sub(r"\\(?:left|right)\b", "", text)
    text = re.sub(r"\\(?:,|;|!|quad|qquad)", "", text)

    text = text.replace(r"\times", "*")
    text = text.replace(r"\cdot", "*")
    text = text.replace(r"\div", "/")
    text = text.replace(r"\%", "%")

    text = text.replace("{", "(").replace("}", ")")
    text = text.replace(",", "")

    return text.strip()


def usc_extract_latex_chunks(text):
    patterns = [
        r"\\\((.*?)\\\)",
        r"\\\[(.*?)\\\]",
        r"\$\$(.*?)\$\$",
        r"(?<!\$)\$(?!\$)(.*?)(?<!\$)\$(?!\$)",
    ]

    chunks = []
    for pattern in patterns:
        chunks.extend(
            re.findall(pattern, str(text), flags=re.DOTALL)
        )

    return chunks


def usc_extract_standalone_equations(text):
    """
    Only accept equation-like text containing arithmetic symbols.
    Avoid parsing arbitrary natural-language statements.
    """
    text = str(text)
    results = []

    for line in text.splitlines():
        line = line.strip()

        if "=" not in line:
            continue

        # Exclude natural-language sentences.
        if re.search(r"[A-Za-z]", line):
            continue

        if re.fullmatch(r"[\d\s()+\-*/×÷^.,=]+", line):
            results.append(line)

    return results


def usc_extract_equation_chunks(text):
    text = str(text)

    chunks = []
    chunks.extend(usc_extract_gsm8k_equations(text))
    chunks.extend(usc_extract_latex_chunks(text))
    chunks.extend(usc_extract_standalone_equations(text))

    # Preserve order while removing exact duplicates.
    unique = []
    seen = set()

    for chunk in chunks:
        key = str(chunk).strip()
        if key and key not in seen:
            seen.add(key)
            unique.append(key)

    return unique


# ============================================================
# 3. Check explicit equalities
# ============================================================

def usc_check_equation(equation):
    normalized = usc_normalize_latex_math(equation)

    # Reject quotient/remainder notation and ellipses.
    if "..." in normalized or "…" in normalized:
        return None

    if re.search(r"\b(?:remainder|quotient|modulo)\b", normalized, re.I):
        return None

    if normalized.count("=") < 1:
        return None

    sides = [part.strip() for part in normalized.split("=")]

    if len(sides) < 2 or any(not side for side in sides):
        return None

    checked = []

    for lhs, rhs in zip(sides, sides[1:]):
        try:
            left_value = usc_safe_eval_arithmetic(lhs)
            right_value = usc_safe_eval_arithmetic(rhs)

            is_correct = left_value == right_value

            checked.append({
                "equation": f"{lhs} = {rhs}",
                "left_value": str(left_value),
                "right_value": str(right_value),
                "is_correct": bool(is_correct),
            })

        except Exception:
            return None

    return checked


def check_step_arithmetic(step):
    """
    PASS: every extracted equality was evaluated and passed.
    FAIL: at least one evaluated equality is false.
    NOT_CHECKED: no safe explicit equality was available.
    """
    chunks = usc_extract_equation_chunks(step)

    checked_equations = []
    failed_equations = []

    for chunk in chunks:
        result = usc_check_equation(chunk)

        if result is None:
            continue

        checked_equations.extend(result)

        for equation in result:
            if not equation["is_correct"]:
                failed_equations.append(equation)

    if failed_equations:
        return {
            "status": "FAIL",
            "equations": checked_equations,
            "failed_equations": failed_equations,
        }

    if checked_equations:
        return {
            "status": "PASS",
            "equations": checked_equations,
            "failed_equations": [],
        }

    return {
        "status": "NOT_CHECKED",
        "equations": [],
        "failed_equations": [],
    }


# ============================================================
# 4. Conservative redundancy detection
# ============================================================

def usc_normalize_step(text):
    text = str(text).lower()
    text = re.sub(r"<<([^<>]+)>>", r"\1", text)
    text = re.sub(r"\\[()\[\]]", " ", text)
    text = re.sub(r"[^a-z0-9.]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def usc_detect_redundancy(step, previous_steps):
    current = usc_normalize_step(step)

    if not current:
        return None

    for previous in previous_steps:
        previous_normalized = usc_normalize_step(previous)

        if previous_normalized and current == previous_normalized:
            return "Exact normalized duplicate of an earlier retained step."

    return None


# ============================================================
# 5. Main verifier — compatible with Cell 32
# ============================================================

def verify_reasoning_steps(candidate):
    candidate = candidate.copy()

    steps = candidate.get("filtered_steps", [])

    if not isinstance(steps, list):
        raise TypeError("candidate['filtered_steps'] must be a list.")

    results = []

    for i, step in enumerate(steps):
        step_text = str(step)
        previous_steps = steps[:i]

        redundancy_reason = usc_detect_redundancy(
            step_text,
            previous_steps
        )

        arithmetic_result = check_step_arithmetic(step_text)

        if redundancy_reason:
            final_label = "REDUNDANT"
            method = "rule_based_redundancy"
            reason = redundancy_reason

        elif arithmetic_result["status"] == "FAIL":
            final_label = "INVALID"
            method = "deterministic_arithmetic"
            reason = (
                "At least one explicit numeric equality is mathematically false."
            )

        elif arithmetic_result["status"] == "PASS":
            final_label = "VALID"
            method = "deterministic_arithmetic"
            reason = (
                "All extracted explicit numeric equalities passed. "
                "This does not certify the full semantic reasoning."
            )

        else:
            final_label = "UNKNOWN"
            method = "insufficient_arithmetic_evidence"
            reason = (
                "No explicit numeric equality could be safely evaluated. "
                "Semantic correctness is not certified."
            )

        results.append({
            "step": i + 1,
            "text": step_text,
            "label": final_label,
            "model_label": None,
            "arithmetic_check": arithmetic_result["status"],
            "checked_equations": arithmetic_result["equations"],
            "raw_output": final_label,
            "method": method,
            "reason": reason,
            "verifier_version": VERIFIER_VERSION,
        })

    candidate["step_verification"] = results
    candidate["valid_step_count"] = sum(
        item["label"] == "VALID" for item in results
    )
    candidate["invalid_step_count"] = sum(
        item["label"] == "INVALID" for item in results
    )
    candidate["redundant_step_count"] = sum(
        item["label"] == "REDUNDANT" for item in results
    )
    candidate["unknown_step_count"] = sum(
        item["label"] == "UNKNOWN" for item in results
    )
    candidate["verifier_version"] = VERIFIER_VERSION

    return candidate


# ============================================================
# 6. Regression tests
# ============================================================

REGRESSION_TESTS = [
    ("2 * 12 = 24", "PASS"),
    ("24 * 2 = 48", "PASS"),
    ("48 / 3 = 16", "PASS"),
    (r"\(\frac{48}{2} = 24\)", "PASS"),
    ("920 / 10 = 102", "FAIL"),
    ("48 + 24 = 72", "PASS"),
    ("104 = 65% of the total books.", "NOT_CHECKED"),
    ("The quotient is 4 remainder 2.", "NOT_CHECKED"),
    ("x + 2 = 5", "NOT_CHECKED"),
]


print("=" * 72)
print("USC-DPO — EVIDENCE-BASED VERIFIER V3")
print("=" * 72)
print("Verifier version:", VERIFIER_VERSION)
print("Semantic model inference: DISABLED")
print()

regression_failures = []

for test_text, expected in REGRESSION_TESTS:
    actual = check_step_arithmetic(test_text)["status"]
    passed = actual == expected

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"Expected={expected:12s} Actual={actual:12s} | {test_text}"
    )

    if not passed:
        regression_failures.append({
            "text": test_text,
            "expected": expected,
            "actual": actual,
        })

if regression_failures:
    raise RuntimeError(
        f"Arithmetic regression tests failed: {regression_failures}"
    )

print("\nArithmetic regression tests: ALL PASSED")


# ============================================================
# 7. Check known false-positive examples
# ============================================================

KNOWN_FALSE_POSITIVES = [
    "Mimi picked up 2 * 12 = <<2*12=24>>24 shells",
    "Kyle found 24 * 2 = <<24*2=48>>48 shells",
    "Leigh grabbed 48 / 3 = <<48/3=16>>16 shells",
]

print("\n===== KNOWN FALSE-POSITIVE RECHECK =====")

for example in KNOWN_FALSE_POSITIVES:
    result = check_step_arithmetic(example)

    print(
        result["status"],
        "|",
        example,
        "|",
        result["equations"]
    )

# ============================================================
# 8. Verify first candidate as a schema smoke test
# ============================================================

if "filtered_candidates" not in globals() or not filtered_candidates:
    raise RuntimeError(
        "filtered_candidates is missing. Run candidate generation "
        "and filtering cells before Cell 31."
    )

test_candidate = verify_reasoning_steps(filtered_candidates[0])

print("\n===== FIRST CANDIDATE VERIFICATION =====")

for item in test_candidate["step_verification"]:
    print(
        f"Step {item['step']} | {item['label']} | "
        f"Arithmetic={item['arithmetic_check']} | "
        f"Method={item['method']}"
    )
    print("  Text:", item["text"])
    if item["reason"]:
        print("  Reason:", item["reason"])

print("\n===== SUMMARY =====")
print("VALID:", test_candidate["valid_step_count"])
print("INVALID:", test_candidate["invalid_step_count"])
print("REDUNDANT:", test_candidate["redundant_step_count"])
print("UNKNOWN:", test_candidate["unknown_step_count"])

# Release any verifier model left from an earlier Cell 31 execution.
# The V3 verifier does not use candidate_model.
if "candidate_model" in globals():
    try:
        del candidate_model
        gc.collect()
        if "torch" in globals() and torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("\nReleased previous candidate_model from memory.")
    except Exception as exc:
        print("Model cleanup warning:", exc)

print("\nCell 31 V3 completed successfully.")
print("Next: wait for audit of this output before running Cell 32.")


USC-DPO — EVIDENCE-BASED VERIFIER V3
Verifier version: usc_evidence_based_verifier_v3_arithmetic_gated
Semantic model inference: DISABLED

PASS | Expected=PASS         Actual=PASS         | 2 * 12 = 24
PASS | Expected=PASS         Actual=PASS         | 24 * 2 = 48
PASS | Expected=PASS         Actual=PASS         | 48 / 3 = 16
PASS | Expected=PASS         Actual=PASS         | \(\frac{48}{2} = 24\)
PASS | Expected=FAIL         Actual=FAIL         | 920 / 10 = 102
PASS | Expected=PASS         Actual=PASS         | 48 + 24 = 72
PASS | Expected=NOT_CHECKED  Actual=NOT_CHECKED  | 104 = 65% of the total books.
PASS | Expected=NOT_CHECKED  Actual=NOT_CHECKED  | The quotient is 4 remainder 2.
PASS | Expected=NOT_CHECKED  Actual=NOT_CHECKED  | x + 2 = 5

Arithmetic regression tests: ALL PASSED

===== KNOWN FALSE-POSITIVE RECHECK =====
PASS | Mimi picked up 2 * 12 = <<2*12=24>>24 shells | [{'equation': '2*12 = 24', 'left_value': '24', 'right_value': '24', 'is_correct': True}]
PASS | Kyle found 2

In [49]:
'''
# USC-DPO — Full Semantic Verification
# Replace the existing Cell 31. Reuse verifier functions from Cell 30.

import os
import json
import hashlib
from tqdm.auto import tqdm

# 1. Validate prerequisites
if "filtered_candidates" not in globals() or not filtered_candidates:
    raise RuntimeError(
        "filtered_candidates is missing. Run the existing candidate "
        "generation/filtering cells first."
    )

if not callable(globals().get("verify_reasoning_steps")):
    raise RuntimeError(
        "verify_reasoning_steps() is missing. Run Cell 30 first."
    )

if "candidate_model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "Verifier model/tokenizer missing. Run Cell 30 first."
    )

# 2. Create a signature so stale checkpoints are not reused
signature_data = [
    {
        "question_index": c.get("question_index"),
        "candidate_id": c.get("candidate_id"),
        "steps": c.get("filtered_steps", []),
    }
    for c in filtered_candidates
]

input_signature = hashlib.sha256(
    json.dumps(
        signature_data,
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    ).encode("utf-8")
).hexdigest()

# 3. Use Google Drive checkpoint when available
checkpoint_dir = "/content/USC_DPO_verification"

try:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    if os.path.isdir("/content/drive/MyDrive"):
        checkpoint_dir = (
            "/content/drive/MyDrive/USC_DPO_verification"
        )
except Exception as e:
    print("Google Drive unavailable; using local checkpoint:", e)

os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_path = os.path.join(
    checkpoint_dir, "semantic_verification.jsonl"
)
results_path = os.path.join(
    checkpoint_dir, "semantic_verified_candidates.json"
)

# 4. Load matching completed candidates
completed = {}

if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)
                if record.get("input_signature") == input_signature:
                    completed[int(record["index"])] = record["result"]
            except (json.JSONDecodeError, KeyError, ValueError):
                continue

print(f"Candidates in dataset: {len(filtered_candidates)}")
print(f"Already verified in checkpoint: {len(completed)}")
print(f"Remaining: {len(filtered_candidates) - len(completed)}")

# 5. Verify every candidate, saving each completed result
with open(checkpoint_path, "a", encoding="utf-8") as checkpoint:
    for idx, candidate in enumerate(
        tqdm(filtered_candidates, desc="Verifying candidates")
    ):
        if idx in completed:
            continue

        # The function verifies each reasoning step using the Cell 30 verifier.
        result = verify_reasoning_steps(candidate.copy())
        result["semantic_verification_skipped"] = False

        record = {
            "input_signature": input_signature,
            "index": idx,
            "result": result,
        }

        checkpoint.write(
            json.dumps(record, ensure_ascii=False, default=str) + "\n"
        )
        checkpoint.flush()

        completed[idx] = result

# 6. Restore original order
semantic_verified_candidates = [
    completed[i] for i in range(len(filtered_candidates))
]

# 7. Save consolidated results
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(
        semantic_verified_candidates,
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

# 8. Summarize step-level verification
all_steps = [
    step
    for candidate in semantic_verified_candidates
    for step in candidate.get("step_verification", [])
]

label_counts = {
    label: sum(step.get("label") == label for step in all_steps)
    for label in ("VALID", "INVALID", "REDUNDANT", "UNKNOWN")
}

expected_steps = sum(
    len(c.get("filtered_steps", []))
    for c in filtered_candidates
)

print("\n===== USC-DPO SEMANTIC VERIFICATION SUMMARY =====")
print(
    f"Candidates verified: "
    f"{len(semantic_verified_candidates)}/{len(filtered_candidates)}"
)
print(f"Expected reasoning steps: {expected_steps}")
print(f"Steps with verification labels: {len(all_steps)}")

for label, count in label_counts.items():
    print(f"{label}: {count}")

print(f"\nCheckpoint: {checkpoint_path}")
print(f"Results: {results_path}")

if len(semantic_verified_candidates) != len(filtered_candidates):
    raise RuntimeError("Some candidates are missing from verification.")

if len(all_steps) != expected_steps:
    raise RuntimeError(
        "Step count mismatch. Inspect verification results before proceeding."
    )

print("\nVerification pass completed. Review labels before training.")
print("Note: AI labels are judgments, not independent mathematical proof.")


USC-DPO — EVIDENCE-BASED VERIFIER V4
Verifier: usc_evidence_based_verifier_v4_integrity_gated
Semantic model inference: DISABLED
Candidates: 320
Input steps: 1392

===== REGRESSION TESTS =====
PASS | Expected=PASS               Actual=PASS               | 2 * 12 = 24
PASS | Expected=PASS               Actual=PASS               | 24 * 2 = 48
PASS | Expected=PASS               Actual=PASS               | \(\frac{48}{2} = 24\)
PASS | Expected=FAIL               Actual=FAIL               | 920 / 10 = 102
PASS | Expected=PASS               Actual=PASS               | 48 + 24 = 72
PASS | Expected=NOT_CHECKED        Actual=NOT_CHECKED        | 104 = 65% of the total books.
PASS | Expected=NOT_CHECKED        Actual=NOT_CHECKED        | The quotient is 4 remainder 2.
PASS | Expected=NOT_CHECKED        Actual=NOT_CHECKED        | x + 2 = 5
PASS | Expected=NOT_CHECKED        Actual=NOT_CHECKED        | 2 + 6 = 8 - x
PASS | Expected=PASS               Actual=PASS               | Mimi picked up <<2

In [51]:

# USC-DPO: Candidate Quality Classification V4 — Evidence-Aware
# REPLACE PHYSICAL CELL 33 ENTIRELY.
#
# Reads semantic_verified_candidates created by Cell 31 V4.
# Separates answer correctness, arithmetic evidence, and semantic certification.
# Does not equate arithmetic VALID with semantically correct reasoning.

import os
import json
from collections import Counter, defaultdict

# ============================================================
# 1. Validate V4 input
# ============================================================

if not globals().get("semantic_verified_candidates"):
    raise RuntimeError(
        "semantic_verified_candidates is missing. "
        "Run physical Cell 31 V4 first."
    )

VERIFIER_VERSION_EXPECTED = (
    "usc_evidence_based_verifier_v4_integrity_gated"
)

candidates = semantic_verified_candidates

if not all(
    c.get("verifier_version") == VERIFIER_VERSION_EXPECTED
    for c in candidates
):
    raise RuntimeError(
        "Mixed or outdated verifier versions detected. "
        "Cell 33 requires exclusively Cell 31 V4 outputs."
    )

if len(candidates) != 320:
    print(
        "WARNING: Expected 320 candidates, found",
        len(candidates)
    )

ARTIFACT_DIR = "/content/USC_DPO_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)


# ============================================================
# 2. Candidate classification
# ============================================================

def classify_candidate_quality(candidate):

    steps = candidate.get("step_verification", [])

    # Answer correctness must be an actual boolean.
    raw_answer_status = candidate.get("is_correct")

    if isinstance(raw_answer_status, bool):
        answer_correct = raw_answer_status
    else:
        answer_correct = None

    # Recompute all counts from the actual step records.
    labels = [
        step.get("label", "UNKNOWN")
        for step in steps
    ]

    valid_count = labels.count("VALID")
    invalid_count = labels.count("INVALID")
    redundant_count = labels.count("REDUNDANT")
    unknown_count = labels.count("UNKNOWN")

    # Identify malformed GSM8K annotations.
    malformed_steps = [
        step
        for step in steps
        if (
            not step.get("marker_integrity", {}).get(
                "is_well_formed", False
            )
        )
    ]

    malformed_count = len(malformed_steps)

    # Arithmetic evidence is distinct from semantic evidence.
    arithmetic_pass_count = sum(
        step.get("arithmetic_check") == "PASS"
        for step in steps
    )

    arithmetic_fail_count = sum(
        step.get("arithmetic_check") == "FAIL"
        for step in steps
    )

    arithmetic_not_checked_count = sum(
        step.get("arithmetic_check") in (
            "NOT_CHECKED",
            "MALFORMED_MARKERS",
        )
        for step in steps
    )

    has_arithmetic_evidence = arithmetic_pass_count > 0

    has_arithmetic_error = (
        invalid_count > 0
        or arithmetic_fail_count > 0
    )

    has_generation_corruption = malformed_count > 0

    has_known_reasoning_issue = (
        has_arithmetic_error
        or has_generation_corruption
    )

    # V4 does not perform semantic verification.
    semantic_certified = (
        candidate.get("semantic_certification") == "CERTIFIED"
        and not candidate.get("semantic_verification_skipped", True)
    )

    # High quality requires semantic certification, not merely
    # arithmetic consistency.
    is_high_quality = (
        answer_correct is True
        and semantic_certified
        and not has_known_reasoning_issue
        and unknown_count == 0
    )

    # A wrong answer or an explicit reasoning/data-integrity issue
    # is negative evidence.
    is_low_quality = (
        answer_correct is False
        or has_known_reasoning_issue
    )

    # Correct answer without semantic certification is provisional.
    if answer_correct is False:

        quality_status = "ANSWER_INCORRECT"

    elif answer_correct is None:

        quality_status = "ANSWER_UNPARSED"

    elif has_generation_corruption:

        quality_status = "ANSWER_CORRECT_WITH_CORRUPTED_REASONING"

    elif has_arithmetic_error:

        quality_status = "ANSWER_CORRECT_WITH_ARITHMETIC_ERROR"

    elif semantic_certified and is_high_quality:

        quality_status = "SEMANTICALLY_VERIFIED_HIGH_QUALITY"

    else:

        quality_status = (
            "ANSWER_CORRECT_REASONING_NOT_SEMANTICALLY_VERIFIED"
        )

    # A correct final answer plus passing arithmetic is useful
    # evidence, but it is not a reasoning-quality certificate.
    answer_correct_arithmetic_supported = (
        answer_correct is True
        and has_arithmetic_evidence
        and not has_arithmetic_error
        and not has_generation_corruption
    )

    # V4 cannot certify same-answer reasoning preferences.
    reasoning_pair_eligible = (
        is_high_quality
    )

    efficiency_pair_eligible = (
        is_high_quality
        and redundant_count > 0
    )

    # Update the candidate in place while preserving original fields.
    candidate["valid_step_count"] = valid_count
    candidate["invalid_step_count"] = invalid_count
    candidate["redundant_step_count"] = redundant_count
    candidate["unknown_step_count"] = unknown_count

    candidate["arithmetic_pass_count"] = arithmetic_pass_count
    candidate["arithmetic_fail_count"] = arithmetic_fail_count
    candidate["arithmetic_not_checked_count"] = (
        arithmetic_not_checked_count
    )

    candidate["malformed_marker_step_count"] = malformed_count
    candidate["has_arithmetic_evidence"] = has_arithmetic_evidence
    candidate["has_arithmetic_error"] = has_arithmetic_error
    candidate["has_generation_corruption"] = has_generation_corruption
    candidate["has_known_reasoning_issue"] = has_known_reasoning_issue

    candidate["semantic_certified"] = semantic_certified
    candidate["is_high_quality"] = is_high_quality
    candidate["is_low_quality"] = is_low_quality

    candidate["is_uncertain"] = (
        answer_correct is None
        or not semantic_certified
    )

    candidate["quality_status"] = quality_status

    candidate["answer_correct_arithmetic_supported"] = (
        answer_correct_arithmetic_supported
    )

    candidate["requires_manual_review"] = (
        not semantic_certified
        or has_known_reasoning_issue
        or answer_correct is None
    )

    candidate["reasoning_pair_eligible"] = reasoning_pair_eligible
    candidate["efficiency_pair_eligible"] = efficiency_pair_eligible

    candidate["answer_correction_rejected"] = (
        answer_correct is False
    )

    candidate["quality_classifier_version"] = (
        "usc_candidate_quality_v4_evidence_aware"
    )

    return candidate


# ============================================================
# 3. Run classification for all candidates
# ============================================================

for candidate in candidates:
    classify_candidate_quality(candidate)


# ============================================================
# 4. Aggregate candidate-level statistics
# ============================================================

quality_counts = Counter(
    c["quality_status"]
    for c in candidates
)

answer_counts = {
    "CORRECT": sum(
        c.get("is_correct") is True
        for c in candidates
    ),
    "INCORRECT": sum(
        c.get("is_correct") is False
        for c in candidates
    ),
    "UNPARSED": sum(
        not isinstance(c.get("is_correct"), bool)
        for c in candidates
    ),
}

reasoning_counts = {
    "WITH_KNOWN_ARITHMETIC_OR_INTEGRITY_ERROR": sum(
        c["has_known_reasoning_issue"]
        for c in candidates
    ),
    "WITHOUT_KNOWN_ARITHMETIC_OR_INTEGRITY_ERROR": sum(
        not c["has_known_reasoning_issue"]
        for c in candidates
    ),
    "WITH_ARITHMETIC_EVIDENCE": sum(
        c["has_arithmetic_evidence"]
        for c in candidates
    ),
    "WITH_MALFORMED_MARKERS": sum(
        c["has_generation_corruption"]
        for c in candidates
    ),
    "SEMANTICALLY_CERTIFIED": sum(
        c["semantic_certified"]
        for c in candidates
    ),
    "SEMANTICALLY_UNVERIFIED": sum(
        not c["semantic_certified"]
        for c in candidates
    ),
}

preference_eligibility = {
    "HIGH_QUALITY": sum(
        c["is_high_quality"]
        for c in candidates
    ),
    "LOW_QUALITY": sum(
        c["is_low_quality"]
        for c in candidates
    ),
    "UNCERTAIN": sum(
        c["is_uncertain"]
        for c in candidates
    ),
    "ANSWER_CORRECTION_REJECTED": sum(
        c["answer_correction_rejected"]
        for c in candidates
    ),
    "REASONING_PAIR_ELIGIBLE": sum(
        c["reasoning_pair_eligible"]
        for c in candidates
    ),
    "EFFICIENCY_PAIR_ELIGIBLE": sum(
        c["efficiency_pair_eligible"]
        for c in candidates
    ),
    "MANUAL_REVIEW_REQUIRED": sum(
        c["requires_manual_review"]
        for c in candidates
    ),
}


# ============================================================
# 5. Candidate-level answer x reasoning matrix
# ============================================================

answer_reasoning_matrix = defaultdict(int)

for candidate in candidates:

    answer_status = candidate["quality_status"]

    if candidate["has_generation_corruption"]:
        reasoning_status = "MARKER_CORRUPTED"

    elif candidate["has_arithmetic_error"]:
        reasoning_status = "ARITHMETIC_ERROR"

    elif candidate["has_arithmetic_evidence"]:
        reasoning_status = "ARITHMETIC_SUPPORTED_SEMANTICS_UNVERIFIED"

    else:
        reasoning_status = "NO_CHECKABLE_ARITHMETIC_SEMANTICALLY_UNVERIFIED"

    answer_reasoning_matrix[
        (answer_status, reasoning_status)
    ] += 1


# ============================================================
# 6. Consistency checks
# ============================================================

if sum(answer_counts.values()) != len(candidates):
    raise RuntimeError(
        "Answer classification count does not match candidate count."
    )

if (
    preference_eligibility["HIGH_QUALITY"]
    != preference_eligibility["REASONING_PAIR_ELIGIBLE"]
):
    raise RuntimeError(
        "Reasoning-pair eligibility is inconsistent."
    )

if (
    preference_eligibility["HIGH_QUALITY"]
    > reasoning_counts["SEMANTICALLY_CERTIFIED"]
):
    raise RuntimeError(
        "High-quality candidates exceed semantically certified candidates."
    )

if (
    reasoning_counts["SEMANTICALLY_CERTIFIED"] == 0
    and preference_eligibility["HIGH_QUALITY"] != 0
):
    raise RuntimeError(
        "A candidate cannot be high-quality without semantic certification."
    )


# ============================================================
# 7. Persist versioned quality audit
# ============================================================

AUDIT_PATH = os.path.join(
    ARTIFACT_DIR,
    "candidate_quality_audit_v4.json"
)

SUMMARY_PATH = os.path.join(
    ARTIFACT_DIR,
    "candidate_quality_summary_v4.json"
)

ANNOTATED_CANDIDATES_PATH = os.path.join(
    ARTIFACT_DIR,
    "semantic_verified_candidates_v4_quality.json"
)

audit_payload = {
    "classifier_version": "usc_candidate_quality_v4_evidence_aware",
    "verifier_version": VERIFIER_VERSION_EXPECTED,
    "candidate_count": len(candidates),
    "answer_counts": answer_counts,
    "quality_counts": dict(quality_counts),
    "reasoning_counts": reasoning_counts,
    "preference_eligibility": preference_eligibility,
    "answer_reasoning_matrix": [
        {
            "answer_status": key[0],
            "reasoning_status": key[1],
            "candidate_count": value,
        }
        for key, value in sorted(answer_reasoning_matrix.items())
    ],
    "candidates": candidates,
}

summary_payload = {
    "classifier_version": "usc_candidate_quality_v4_evidence_aware",
    "verifier_version": VERIFIER_VERSION_EXPECTED,
    "candidate_count": len(candidates),
    "answer_counts": answer_counts,
    "quality_counts": dict(quality_counts),
    "reasoning_counts": reasoning_counts,
    "preference_eligibility": preference_eligibility,
    "interpretation": {
        "VALID": (
            "Explicit numeric equalities passed deterministic arithmetic "
            "checks; this is not semantic certification."
        ),
        "UNKNOWN": (
            "Insufficient deterministic arithmetic evidence; "
            "not equivalent to an incorrect step."
        ),
        "HIGH_QUALITY": (
            "Requires semantic certification and a correct final answer."
        ),
    },
}

with open(AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        audit_payload,
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(
        summary_payload,
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(ANNOTATED_CANDIDATES_PATH, "w", encoding="utf-8") as f:
    json.dump(
        candidates,
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )


# ============================================================
# 8. Print audit
# ============================================================

print("=" * 72)
print("USC-DPO — CANDIDATE QUALITY CLASSIFICATION V4")
print("=" * 72)

print("\n===== ANSWER CORRECTNESS =====")

for label, count in answer_counts.items():
    print(f"{label}: {count}")

print("\n===== CANDIDATE QUALITY =====")

for label, count in sorted(quality_counts.items()):
    print(f"{label}: {count}")

print("\n===== REASONING EVIDENCE =====")

for label, count in reasoning_counts.items():
    print(f"{label}: {count}")

print("\n===== PREFERENCE ELIGIBILITY =====")

for label, count in preference_eligibility.items():
    print(f"{label}: {count}")

print("\n===== ANSWER x REASONING MATRIX =====")

for row in audit_payload["answer_reasoning_matrix"]:

    print(
        f"{row['answer_status']} "
        f"| {row['reasoning_status']} "
        f"| Candidates={row['candidate_count']}"
    )


# ============================================================
# 9. Show candidates with known errors
# ============================================================

print("\n===== CANDIDATES WITH KNOWN REASONING / INTEGRITY ISSUES =====")

problem_candidates = [
    c for c in candidates
    if c["has_known_reasoning_issue"]
]

for candidate in problem_candidates[:20]:

    print(
        f"Q{candidate.get('question_index')} "
        f"| C{candidate.get('candidate_id')} "
        f"| Answer={candidate.get('is_correct')} "
        f"| Invalid={candidate['invalid_step_count']} "
        f"| Malformed={candidate['malformed_marker_step_count']} "
        f"| Status={candidate['quality_status']}"
    )

if not problem_candidates:
    print("No candidates with explicit arithmetic or marker-integrity errors.")


# ============================================================
# 10. Final status
# ============================================================

print("\n===== SAVED ARTIFACTS =====")
print("Detailed audit:", AUDIT_PATH)
print("Summary:", SUMMARY_PATH)
print("Annotated candidates:", ANNOTATED_CANDIDATES_PATH)

print("\nCell 33 V4 completed successfully.")
print(
    "Important: arithmetic consistency is not semantic certification."
)
print(
    "Reasoning-correction and step-efficiency pairs remain disabled "
    "until semantic evidence is available."
)


USC-DPO — CANDIDATE QUALITY CLASSIFICATION V4

===== ANSWER CORRECTNESS =====
CORRECT: 186
INCORRECT: 126
UNPARSED: 8

===== CANDIDATE QUALITY =====
ANSWER_CORRECT_REASONING_NOT_SEMANTICALLY_VERIFIED: 149
ANSWER_CORRECT_WITH_ARITHMETIC_ERROR: 34
ANSWER_CORRECT_WITH_CORRUPTED_REASONING: 3
ANSWER_INCORRECT: 126
ANSWER_UNPARSED: 8

===== REASONING EVIDENCE =====
WITH_KNOWN_ARITHMETIC_OR_INTEGRITY_ERROR: 81
WITHOUT_KNOWN_ARITHMETIC_OR_INTEGRITY_ERROR: 239
WITH_ARITHMETIC_EVIDENCE: 274
WITH_MALFORMED_MARKERS: 9
SEMANTICALLY_CERTIFIED: 0
SEMANTICALLY_UNVERIFIED: 320

===== PREFERENCE ELIGIBILITY =====
HIGH_QUALITY: 0
LOW_QUALITY: 165
UNCERTAIN: 320
ANSWER_CORRECTION_REJECTED: 126
REASONING_PAIR_ELIGIBLE: 0
EFFICIENCY_PAIR_ELIGIBLE: 0
MANUAL_REVIEW_REQUIRED: 320

===== ANSWER x REASONING MATRIX =====
ANSWER_CORRECT_REASONING_NOT_SEMANTICALLY_VERIFIED | ARITHMETIC_SUPPORTED_SEMANTICS_UNVERIFIED | Candidates=137
ANSWER_CORRECT_REASONING_NOT_SEMANTICALLY_VERIFIED | NO_CHECKABLE_ARITHMETIC_SEMANT

# USC-DPO — SELF-CORRECTION

In [53]:

# ============================================================
# USC-DPO — Preference Dataset V5
# REPLACE PHYSICAL CELL 35 ENTIRELY.
#
# Cell 26 is the sole authority for final-answer labels.
# Cell 31 V4 provides deterministic arithmetic / marker evidence.
# Cell 33 V4 provides diagnostic classifications only.
#
# Objectives:
#   1. answer_correction
#   2. reasoning_correction (confirmed false arithmetic equality)
#   3. step_efficiency DISABLED
#
# No candidate regeneration, model loading, or GPU training.
# ============================================================

import os
import re
import json
import zipfile
from decimal import Decimal
from fractions import Fraction
from collections import Counter, defaultdict


# ============================================================
# 1. Preconditions
# ============================================================

ARTIFACT_DIR = "/content/USC_DPO_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

required_objects = (
    "candidate_data",
    "multi_candidates",
    "verified_candidates",
    "filtered_candidates",
    "semantic_verified_candidates",
    "make_math_prompt",
)

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required pipeline data: " + ", ".join(missing)
    )

if not all(globals().get(name) for name in (
    "candidate_data",
    "multi_candidates",
    "verified_candidates",
    "filtered_candidates",
    "semantic_verified_candidates",
)):
    raise RuntimeError("One or more candidate datasets are empty.")

EXPECTED_VERIFIER = (
    "usc_evidence_based_verifier_v4_integrity_gated"
)

wrong_versions = {
    str(c.get("verifier_version"))
    for c in semantic_verified_candidates
    if c.get("verifier_version") != EXPECTED_VERIFIER
}

if wrong_versions:
    raise RuntimeError(
        "Cell 31 V4 version mismatch: "
        + repr(sorted(wrong_versions))
    )

print("=" * 72)
print("USC-DPO — PREFERENCE DATASET V5")
print("=" * 72)
print("Answer labels: authoritative Cell 26")
print("Reasoning evidence: Cell 31 V4")
print("Semantic certification: NOT PERFORMED")
print("Candidate count:", len(multi_candidates))


# ============================================================
# 2. Exact numeric-label normalization
# ============================================================

def usc35_to_fraction(value):
    """
    Convert an already stored Cell 26 numeric label into an exact Fraction.

    This function does NOT extract an answer from generated reasoning.
    It only normalizes the value that Cell 26 already stored.
    """

    if value is None:
        return None

    text = (
        str(value)
        .strip()
        .replace(",", "")
        .replace("$", "")
        .replace("€", "")
        .replace("£", "")
    )

    if not text:
        return None

    try:
        if "/" in text:
            numerator, denominator = text.split("/", 1)

            denominator_value = Fraction(Decimal(denominator.strip()))

            if denominator_value == 0:
                return None

            return (
                Fraction(Decimal(numerator.strip()))
                / denominator_value
            )

        return Fraction(Decimal(text))

    except Exception:
        return None


def usc35_cell37_number(value):
    """
    Cell 37 uses Decimal(value) to validate stored answer labels.
    Convert only terminating decimals to a compatible string.
    Return None for repeating decimals that Cell 37 cannot parse safely.
    """

    fraction = usc35_to_fraction(value)

    if fraction is None:
        return None

    denominator = fraction.denominator

    while denominator % 2 == 0:
        denominator //= 2

    while denominator % 5 == 0:
        denominator //= 5

    if denominator != 1:
        return None

    if fraction.denominator == 1:
        return str(fraction.numerator)

    try:
        decimal_value = (
            Decimal(fraction.numerator)
            / Decimal(fraction.denominator)
        )

        if Fraction(decimal_value) != fraction:
            return None

        return format(decimal_value.normalize(), "f")

    except Exception:
        return None


def usc35_same_number(a, b):
    left = usc35_to_fraction(a)
    right = usc35_to_fraction(b)

    return (
        left is not None
        and right is not None
        and left == right
    )


# ============================================================
# 3. Index datasets by candidate identity
# ============================================================

def usc35_key(item):
    return (
        int(item["question_index"]),
        int(item["candidate_id"]),
    )


def usc35_unique_lookup(items, dataset_name):

    lookup = {}

    for item in items:
        key = usc35_key(item)

        if key in lookup:
            raise ValueError(
                f"Duplicate candidate identity in {dataset_name}: {key}"
            )

        lookup[key] = item

    return lookup


answer_lookup = usc35_unique_lookup(
    verified_candidates,
    "Cell 26",
)

raw_lookup = usc35_unique_lookup(
    multi_candidates,
    "Cell 25",
)

filtered_lookup = usc35_unique_lookup(
    filtered_candidates,
    "Cell 30",
)

reasoning_lookup = usc35_unique_lookup(
    semantic_verified_candidates,
    "Cell 31 V4",
)

all_keys = set(answer_lookup)

if (
    all_keys != set(raw_lookup)
    or all_keys != set(filtered_lookup)
    or all_keys != set(reasoning_lookup)
):
    raise RuntimeError(
        "Candidate identities do not match across Cells 25, 26, 30, and 31. "
        f"Cell26={len(all_keys)}, "
        f"Cell25={len(raw_lookup)}, "
        f"Cell30={len(filtered_lookup)}, "
        f"Cell31={len(reasoning_lookup)}."
    )

print("\n===== DATASET ALIGNMENT =====")
print("Cell 25:", len(raw_lookup))
print("Cell 26:", len(answer_lookup))
print("Cell 30:", len(filtered_lookup))
print("Cell 31 V4:", len(reasoning_lookup))
print("Identity mismatches: 0")


# ============================================================
# 4. Build authoritative GSM8K reference lookup
# ============================================================

gold_by_qid = {}

for key, answer_item in answer_lookup.items():

    qid, cid = key

    gold_solution = str(
        answer_item.get("gold_answer", "")
    ).strip()

    gold_label = answer_item.get("gold_final_answer")
    question = str(answer_item.get("question", "")).strip()

    if not gold_solution or not question:
        raise RuntimeError(
            f"Missing reference solution or question for Q{qid}/C{cid}."
        )

    gold_fraction = usc35_to_fraction(gold_label)

    if gold_fraction is None:
        raise RuntimeError(
            f"Cell 26 has no usable gold_final_answer for Q{qid}/C{cid}."
        )

    current = {
        "question": question,
        "gold_solution": gold_solution,
        "gold_answer": usc35_cell37_number(gold_label),
        "gold_fraction": gold_fraction,
    }

    previous = gold_by_qid.get(qid)

    if previous is None:
        gold_by_qid[qid] = current

    else:
        if previous["gold_fraction"] != current["gold_fraction"]:
            raise RuntimeError(
                f"Conflicting Cell 26 gold labels for question {qid}."
            )

        if previous["question"] != current["question"]:
            raise RuntimeError(
                f"Conflicting questions for question index {qid}."
            )

        if previous["gold_solution"] != current["gold_solution"]:
            raise RuntimeError(
                f"Conflicting GSM8K reference solutions for question {qid}."
            )

if not gold_by_qid:
    raise RuntimeError("No usable GSM8K reference questions found.")

# Cell 37 stores labels with Decimal(), so exclude non-terminating fractions.
unsupported_gold_qids = {
    qid
    for qid, item in gold_by_qid.items()
    if item["gold_answer"] is None
}

print("\n===== GOLD REFERENCE LOOKUP =====")
print("Questions:", len(gold_by_qid))
print("Gold labels incompatible with Cell 37:", len(unsupported_gold_qids))


# ============================================================
# 5. Reasoning evidence from Cell 31 V4
# ============================================================

def usc35_is_answer_only_step(text):
    """
    Detect an isolated terminal GSM8K answer line.
    It is not a reasoning step and must not count as UNKNOWN reasoning.
    """

    text = str(text or "").strip()

    patterns = [
        r"^####\s*[$]?\s*[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?$",
        r"^####\s*<\s*[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?\s*>$",
    ]

    return any(
        re.fullmatch(pattern, text, flags=re.IGNORECASE)
        for pattern in patterns
    )


def usc35_reasoning_evidence(semantic_item):

    step_results = semantic_item.get(
        "step_verification", []
    )

    generation_integrity = (
        semantic_item.get("generation_integrity", {}) or {}
    )

    malformed = (
        generation_integrity.get("status") == "CORRUPTED"
    )

    substantive_results = []

    for step in step_results:

        integrity = step.get("marker_integrity", {}) or {}

        if (
            not integrity.get("is_well_formed", True)
            or step.get("arithmetic_check") == "MALFORMED_MARKERS"
        ):
            malformed = True

        if usc35_is_answer_only_step(step.get("text", "")):
            continue

        substantive_results.append(step)

    labels = [
        str(item.get("label", "UNKNOWN")).upper()
        for item in substantive_results
    ]

    invalid_items = [
        item
        for item in substantive_results
        if str(item.get("label", "")).upper() == "INVALID"
    ]

    unknown_count = labels.count("UNKNOWN")

    failed_equations = []
    invalid_evidence_complete = True

    for item in invalid_items:

        failures = item.get("failed_equations") or []

        if not failures:
            invalid_evidence_complete = False

        for equation in failures:

            if equation.get("is_correct") is not False:
                invalid_evidence_complete = False

            failed_equations.append({
                "step": item.get("step"),
                **equation,
            })

    if malformed:
        status = "MARKER_CORRUPTED"

    elif (
        invalid_items
        and invalid_evidence_complete
        and unknown_count == 0
    ):
        status = "ARITHMETIC_INVALID_CONFIRMED"

    elif invalid_items:
        status = "ARITHMETIC_INVALID_WITH_UNCERTAINTY"

    elif unknown_count:
        status = "ARITHMETIC_INCOMPLETE"

    elif labels.count("VALID") > 0:
        status = "ARITHMETIC_SUPPORTED_SEMANTICS_UNVERIFIED"

    else:
        status = "NO_ARITHMETIC_EVIDENCE"

    return {
        "status": status,
        "invalid_steps": len(invalid_items),
        "unknown_steps": unknown_count,
        "valid_steps": labels.count("VALID"),
        "redundant_steps": labels.count("REDUNDANT"),
        "failed_equations": failed_equations,
        "malformed": malformed,
        "step_labels": [
            str(item.get("label", "UNKNOWN")).upper()
            for item in step_results
        ],
        "substantive_step_count": len(substantive_results),
        "answer_only_steps_excluded": (
            len(step_results) - len(substantive_results)
        ),
    }


# ============================================================
# 6. Create candidate audit using Cell 26 labels directly
# ============================================================

candidate_audit = []
records_by_key = {}

for key in sorted(all_keys):

    qid, cid = key

    answer_item = answer_lookup[key]
    raw_item = raw_lookup[key]
    filtered_item = filtered_lookup[key]
    semantic_item = reasoning_lookup[key]

    reference = gold_by_qid[qid]

    # AUTHORITATIVE Cell 26 labels. No second answer parser.
    verification_status = str(
        answer_item.get("verification_status", "UNPARSED")
    ).upper()

    is_correct = answer_item.get("is_correct")
    predicted_raw = answer_item.get("predicted_answer")
    gold_raw = answer_item.get("gold_final_answer")

    predicted_fraction = usc35_to_fraction(predicted_raw)
    gold_fraction = usc35_to_fraction(gold_raw)

    data_quality = str(
        answer_item.get("data_quality_status", "OK")
    ).upper()

    original_solution = str(
        raw_item.get("generated_solution", "")
    ).strip()

    if not original_solution:
        original_solution = str(
            answer_item.get("generated_solution", "")
        ).strip()

    filtered_steps = [
        str(step).strip()
        for step in semantic_item.get("filtered_steps", [])
        if str(step).strip()
    ]

    filtered_solution = "\n".join(filtered_steps).strip()

    reasoning = usc35_reasoning_evidence(semantic_item)

    # Validate the stored Cell 26 record internally without re-parsing text.
    label_consistency = True

    if verification_status == "CORRECT":

        label_consistency = (
            is_correct is True
            and predicted_fraction is not None
            and gold_fraction is not None
            and predicted_fraction == gold_fraction
        )

        answer_class = (
            "CORRECT" if label_consistency
            else "CELL26_LABEL_INCONSISTENCY"
        )

    elif verification_status == "INCORRECT":

        label_consistency = (
            is_correct is False
            and predicted_fraction is not None
            and gold_fraction is not None
            and predicted_fraction != gold_fraction
        )

        answer_class = (
            "INCORRECT" if label_consistency
            else "CELL26_LABEL_INCONSISTENCY"
        )

    else:
        answer_class = "UNPARSED"

    contaminated = (
        data_quality == "CONTAMINATED"
        or bool(re.search(
            r"-{2,}\s*Problem\s*:",
            original_solution,
            flags=re.IGNORECASE,
        ))
    )

    if contaminated:
        disposition = "EXCLUDED_CONTAMINATED"

    elif answer_class == "CELL26_LABEL_INCONSISTENCY":
        disposition = "EXCLUDED_CELL26_LABEL_INCONSISTENCY"

    elif answer_class == "UNPARSED":
        disposition = "EXCLUDED_UNPARSED"

    elif qid in unsupported_gold_qids:
        disposition = "EXCLUDED_GOLD_NOT_DECIMAL_COMPATIBLE"

    elif not original_solution:
        disposition = "EXCLUDED_EMPTY_COMPLETION"

    elif reasoning["malformed"]:
        disposition = "EXCLUDED_MALFORMED_MARKERS"

    elif answer_class == "INCORRECT":
        disposition = "ELIGIBLE_ANSWER_CORRECTION"

    elif (
        answer_class == "CORRECT"
        and reasoning["status"] == "ARITHMETIC_INVALID_CONFIRMED"
    ):
        disposition = "ELIGIBLE_REASONING_CORRECTION"

    elif answer_class == "CORRECT":
        disposition = "CORRECT_ANSWER_NO_CONFIRMED_ARITHMETIC_ERROR"

    else:
        disposition = "EXCLUDED_OTHER"

    record = {
        "question_index": qid,
        "candidate_id": cid,
        "question": reference["question"],
        "gold_solution": reference["gold_solution"],
        "gold_answer": reference["gold_answer"],

        "verification_status": verification_status,
        "is_correct": is_correct,
        "predicted_answer": predicted_raw,
        "gold_final_answer": gold_raw,
        "label_consistency": label_consistency,
        "answer_class": answer_class,

        "original_solution": original_solution,
        "filtered_solution": filtered_solution,
        "filtered_steps": filtered_steps,

        "reasoning_evidence": reasoning,
        "reasoning_status": reasoning["status"],

        "data_quality_status": data_quality,
        "pair_disposition": disposition,
    }

    candidate_audit.append(record)
    records_by_key[key] = record


# ============================================================
# 7. Answer-correction pairs
# ============================================================

correction_pairs = []
seen_answer_outputs = set()

for record in candidate_audit:

    if record["pair_disposition"] != "ELIGIBLE_ANSWER_CORRECTION":
        continue

    qid = record["question_index"]
    cid = record["candidate_id"]

    rejected_answer = usc35_cell37_number(
        record["predicted_answer"]
    )

    gold_answer = record["gold_answer"]

    if rejected_answer is None or gold_answer is None:
        record["pair_disposition"] = "EXCLUDED_NON_DECIMAL_LABEL"
        continue

    if usc35_same_number(rejected_answer, gold_answer):
        record["pair_disposition"] = "EXCLUDED_ANSWER_LABEL_CONFLICT"
        continue

    rejected = record["original_solution"]
    chosen = record["gold_solution"]

    dedup_key = (
        qid,
        re.sub(r"\s+", " ", rejected.lower()).strip()
    )

    if dedup_key in seen_answer_outputs:
        record["pair_disposition"] = "EXCLUDED_DUPLICATE_ANSWER_OUTPUT"
        continue

    seen_answer_outputs.add(dedup_key)

    pair = {
        "pair_type": "answer_correction",

        "prompt": make_math_prompt(record["question"]),
        "chosen": chosen,
        "rejected": rejected,

        "question_index": qid,
        "question": record["question"],

        "chosen_candidate_id": "gsm8k_reference_solution",
        "rejected_candidate_id": cid,

        "chosen_answer": gold_answer,
        "rejected_answer": rejected_answer,
        "gold_answer": gold_answer,

        "chosen_is_correct": True,
        "rejected_is_correct": False,

        "rejected_reasoning_status": record["reasoning_status"],
        "rejected_invalid_steps": (
            record["reasoning_evidence"]["invalid_steps"]
        ),
        "rejected_unknown_steps": (
            record["reasoning_evidence"]["unknown_steps"]
        ),
        "rejected_step_verification": (
            record["reasoning_evidence"]["step_labels"]
        ),

        "verifier_version": EXPECTED_VERIFIER,
        "semantic_certification": "NOT_PERFORMED",
        "preference_evidence": "cell26_gold_answer_mismatch",
    }

    assert pair["chosen_answer"] == pair["gold_answer"]
    assert pair["rejected_answer"] != pair["gold_answer"]
    assert pair["prompt"] == make_math_prompt(pair["question"])

    correction_pairs.append(pair)

    record["pair_disposition"] = "USED_ANSWER_CORRECTION"


# ============================================================
# 8. Reasoning-correction pairs
# ============================================================

reasoning_correction_pairs = []
seen_reasoning_outputs = set()

for record in candidate_audit:

    if record["pair_disposition"] != "ELIGIBLE_REASONING_CORRECTION":
        continue

    evidence = record["reasoning_evidence"]

    # Strict evidence gate.
    if evidence["status"] != "ARITHMETIC_INVALID_CONFIRMED":
        record["pair_disposition"] = "EXCLUDED_REASONING_EVIDENCE_NOT_CONFIRMED"
        continue

    if evidence["invalid_steps"] < 1:
        record["pair_disposition"] = "EXCLUDED_NO_INVALID_STEP"
        continue

    if evidence["unknown_steps"] != 0:
        record["pair_disposition"] = "EXCLUDED_UNKNOWN_REASONING"
        continue

    if evidence["malformed"]:
        record["pair_disposition"] = "EXCLUDED_MALFORMED_MARKERS"
        continue

    if not evidence["failed_equations"]:
        record["pair_disposition"] = "EXCLUDED_NO_FAILED_EQUATION_AUDIT"
        continue

    if any(
        item.get("is_correct") is not False
        for item in evidence["failed_equations"]
    ):
        record["pair_disposition"] = "EXCLUDED_FAILED_EQUATION_AUDIT_CONFLICT"
        continue

    qid = record["question_index"]
    cid = record["candidate_id"]

    gold_answer = record["gold_answer"]

    # Cell 26 confirms both final answers are the same correct answer.
    chosen_answer = usc35_cell37_number(record["gold_final_answer"])
    rejected_answer = usc35_cell37_number(record["predicted_answer"])

    if (
        chosen_answer is None
        or rejected_answer is None
        or chosen_answer != gold_answer
        or rejected_answer != gold_answer
    ):
        record["pair_disposition"] = "EXCLUDED_SAME_ANSWER_NOT_CONFIRMED"
        continue

    chosen = record["gold_solution"]
    rejected = record["original_solution"]

    dedup_key = (
        qid,
        re.sub(r"\s+", " ", rejected.lower()).strip()
    )

    if dedup_key in seen_reasoning_outputs:
        record["pair_disposition"] = "EXCLUDED_DUPLICATE_REASONING_OUTPUT"
        continue

    seen_reasoning_outputs.add(dedup_key)

    pair = {
        "pair_type": "reasoning_correction",

        "prompt": make_math_prompt(record["question"]),
        "chosen": chosen,
        "rejected": rejected,

        "question_index": qid,
        "question": record["question"],

        "chosen_candidate_id": "gsm8k_reference_solution",
        "rejected_candidate_id": cid,

        "chosen_answer": gold_answer,
        "rejected_answer": gold_answer,
        "gold_answer": gold_answer,

        "chosen_is_correct": True,
        "rejected_is_correct": True,

        # Required by Cell 37.
        "rejected_reasoning_status": "INVALID",

        "rejected_invalid_steps": evidence["invalid_steps"],
        "rejected_unknown_steps": evidence["unknown_steps"],
        "rejected_step_verification": evidence["step_labels"],
        "failed_equations": evidence["failed_equations"],

        "verifier_version": EXPECTED_VERIFIER,
        "semantic_certification": "NOT_PERFORMED",
        "preference_evidence": "deterministic_false_numeric_equality",
    }

    assert pair["chosen_answer"] == pair["gold_answer"]
    assert pair["rejected_answer"] == pair["gold_answer"]
    assert pair["rejected_reasoning_status"] == "INVALID"
    assert pair["rejected_invalid_steps"] > 0
    assert pair["rejected_unknown_steps"] == 0
    assert pair["prompt"] == make_math_prompt(pair["question"])

    reasoning_correction_pairs.append(pair)

    record["pair_disposition"] = "USED_REASONING_CORRECTION"


# ============================================================
# 9. Step-efficiency pairs
# ============================================================

# Disabled. V4 has not certified semantic equivalence or the validity
# of removed natural-language steps. Do not infer efficiency preferences
# from arithmetic validity alone.

efficiency_pairs = []


# ============================================================
# 10. Assemble and validate final pairs
# ============================================================

pairs = (
    correction_pairs
    + reasoning_correction_pairs
    + efficiency_pairs
)

for i, pair in enumerate(pairs):

    for field in ("prompt", "chosen", "rejected"):
        if not str(pair.get(field, "")).strip():
            raise ValueError(
                f"Pair {i} has an empty {field}."
            )

    if pair["prompt"] != make_math_prompt(pair["question"]):
        raise ValueError(
            f"Cell 14 prompt mismatch at pair {i}."
        )

    if pair["chosen_answer"] != pair["gold_answer"]:
        raise ValueError(
            f"Chosen answer differs from gold at pair {i}."
        )

    if pair["pair_type"] == "answer_correction":

        if pair["rejected_answer"] == pair["gold_answer"]:
            raise ValueError(
                f"Answer-correction pair {i} has no answer mismatch."
            )

    elif pair["pair_type"] == "reasoning_correction":

        if pair["rejected_answer"] != pair["gold_answer"]:
            raise ValueError(
                f"Reasoning-correction pair {i} has unequal final answers."
            )

        if pair["rejected_reasoning_status"] != "INVALID":
            raise ValueError(
                f"Reasoning-correction pair {i} lacks INVALID evidence."
            )

        if pair["rejected_invalid_steps"] < 1:
            raise ValueError(
                f"Reasoning-correction pair {i} has no confirmed invalid step."
            )

        if pair["rejected_unknown_steps"] != 0:
            raise ValueError(
                f"Reasoning-correction pair {i} contains UNKNOWN steps."
            )

    else:
        raise ValueError(
            f"Unsupported preference type at pair {i}."
        )

# Exact variable consumed by Cell 36 and Cell 37.
dpo_dataset_validated = pairs


# ============================================================
# 11. Candidate / pair audit
# ============================================================

candidate_dispositions = Counter(
    item["pair_disposition"]
    for item in candidate_audit
)

answer_classes = Counter(
    item["answer_class"]
    for item in candidate_audit
)

reasoning_statuses = Counter(
    item["reasoning_status"]
    for item in candidate_audit
)

pair_type_counts = Counter(
    pair["pair_type"]
    for pair in pairs
)

pair_audit = []

for i, pair in enumerate(pairs):

    pair_audit.append({
        "pair_index": i,
        "pair_type": pair["pair_type"],
        "question_index": pair["question_index"],
        "chosen_candidate_id": pair["chosen_candidate_id"],
        "rejected_candidate_id": pair["rejected_candidate_id"],
        "chosen_answer": pair["chosen_answer"],
        "rejected_answer": pair["rejected_answer"],
        "gold_answer": pair["gold_answer"],
        "rejected_reasoning_status": pair.get("rejected_reasoning_status"),
        "rejected_invalid_steps": pair.get("rejected_invalid_steps"),
        "rejected_unknown_steps": pair.get("rejected_unknown_steps"),
        "preference_evidence": pair["preference_evidence"],
    })


# ============================================================
# 12. Save V5 artifacts (do not overwrite V1/V2/V4)
# ============================================================

pairs_path = os.path.join(
    ARTIFACT_DIR,
    "dpo_preference_pairs_v5_authoritative_labels.json",
)

candidate_audit_path = os.path.join(
    ARTIFACT_DIR,
    "usc_dpo_candidate_audit_v5.json",
)

pair_audit_path = os.path.join(
    ARTIFACT_DIR,
    "usc_dpo_pair_audit_v5.json",
)

summary_path = os.path.join(
    ARTIFACT_DIR,
    "usc_dpo_preference_summary_v5.json",
)

with open(pairs_path, "w", encoding="utf-8") as f:
    json.dump(
        pairs,
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(candidate_audit_path, "w", encoding="utf-8") as f:
    json.dump(
        candidate_audit,
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(pair_audit_path, "w", encoding="utf-8") as f:
    json.dump(
        pair_audit,
        f,
        ensure_ascii=False,
        indent=2,
    )

summary = {
    "version": "usc_dpo_preference_dataset_v5_authoritative_labels",
    "answer_label_source": "Cell 26 stored verification_status/is_correct",
    "reasoning_evidence_source": EXPECTED_VERIFIER,
    "semantic_certification": "NOT_PERFORMED",

    "raw_candidates": len(multi_candidates),
    "candidate_records_audited": len(candidate_audit),
    "gold_questions": len(gold_by_qid),

    "answer_class_counts": dict(answer_classes),
    "reasoning_status_counts": dict(reasoning_statuses),
    "candidate_disposition_counts": dict(candidate_dispositions),

    "answer_correction_pairs": len(correction_pairs),
    "reasoning_correction_pairs": len(reasoning_correction_pairs),
    "step_efficiency_pairs": len(efficiency_pairs),
    "total_preference_pairs": len(pairs),

    "distinct_source_questions": len({
        pair["question_index"] for pair in pairs
    }),

    "independent_answer_reparse_used": False,
    "cell26_labels_reused_directly": True,

    "reasoning_correction_rule": (
        "Cell 26 correct final answer; Cell 31 V4 confirms at least one "
        "false explicit numeric equality; no UNKNOWN substantive steps; "
        "no malformed markers."
    ),

    "step_efficiency_rule": (
        "Disabled because V4 does not certify semantic equivalence "
        "or redundancy of natural-language steps."
    ),

    "artifact_paths": {
        "pairs": pairs_path,
        "candidate_audit": candidate_audit_path,
        "pair_audit": pair_audit_path,
    },
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2,
    )

zip_path = os.path.join(
    ARTIFACT_DIR,
    "USC_DPO_preference_artifacts_v5_authoritative_labels.zip",
)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:

    for path in (
        pairs_path,
        candidate_audit_path,
        pair_audit_path,
        summary_path,
    ):
        zf.write(
            path,
            arcname=os.path.basename(path),
        )


# ============================================================
# 13. Final report
# ============================================================

print("\n===== CELL 26 ANSWER LABELS =====")

for label, count in sorted(answer_classes.items()):
    print(f"{label}: {count}")

print("\n===== CELL 31 V4 REASONING EVIDENCE =====")

for label, count in sorted(reasoning_statuses.items()):
    print(f"{label}: {count}")

print("\n===== CANDIDATE DISPOSITIONS =====")

for label, count in sorted(candidate_dispositions.items()):
    print(f"{label}: {count}")

print("\n===== PREFERENCE DATASET =====")
print("Answer-correction:", len(correction_pairs))
print("Reasoning-correction:", len(reasoning_correction_pairs))
print("Step-efficiency:", len(efficiency_pairs), "(disabled)")
print("Total pairs:", len(pairs))
print(
    "Distinct source questions:",
    summary["distinct_source_questions"]
)

print("\n===== PARSER DESIGN =====")
print("Independent answer reparse: DISABLED")
print("Cell 26 stored labels: AUTHORITATIVE")
print("Cell 31 V4 arithmetic evidence: ENABLED")

print("\n===== SAVED V5 ARTIFACTS =====")
print("Pairs:", pairs_path)
print("Candidate audit:", candidate_audit_path)
print("Pair audit:", pair_audit_path)
print("Summary:", summary_path)
print("ZIP:", zip_path)

if not pairs:
    print(
        "\nWARNING: No preference pairs passed the V5 gates. "
        "Do NOT run Cell 37."
    )

print("\nCell 35 V5 completed successfully.")
print("Replace physical Cell 36 before running the pair audit.")


USC-DPO — PREFERENCE DATASET V5
Answer labels: authoritative Cell 26
Reasoning evidence: Cell 31 V4
Semantic certification: NOT PERFORMED
Candidate count: 320

===== DATASET ALIGNMENT =====
Cell 25: 320
Cell 26: 320
Cell 30: 320
Cell 31 V4: 320
Identity mismatches: 0

===== GOLD REFERENCE LOOKUP =====
Questions: 80
Gold labels incompatible with Cell 37: 0

===== CELL 26 ANSWER LABELS =====
CORRECT: 186
INCORRECT: 126
UNPARSED: 8

===== CELL 31 V4 REASONING EVIDENCE =====
ARITHMETIC_INCOMPLETE: 97
ARITHMETIC_INVALID_CONFIRMED: 49
ARITHMETIC_INVALID_WITH_UNCERTAINTY: 23
ARITHMETIC_SUPPORTED_SEMANTICS_UNVERIFIED: 142
MARKER_CORRUPTED: 9

===== CANDIDATE DISPOSITIONS =====
CORRECT_ANSWER_NO_CONFIRMED_ARITHMETIC_ERROR: 158
EXCLUDED_MALFORMED_MARKERS: 8
EXCLUDED_UNPARSED: 8
USED_ANSWER_CORRECTION: 121
USED_REASONING_CORRECTION: 25

===== PREFERENCE DATASET =====
Answer-correction: 121
Reasoning-correction: 25
Step-efficiency: 0 (disabled)
Total pairs: 146
Distinct source questions: 68

=====

In [55]:
# ============================================================
# CELL 36 — USC-DPO PREFERENCE PAIR AUDIT V6
# Prompt provenance: Cell 24 candidate_data
# Answer labels: authoritative Cell 26 / Cell 35 V5
# Reasoning evidence: Cell 31 V4
#
# Does not regenerate candidates or preference pairs.
# Does not modify chosen/rejected completions.
# Does not load a model or start GPU training.
# ============================================================

import os
import json
from decimal import Decimal, InvalidOperation
from collections import Counter


# ============================================================
# 1. CONFIGURATION
# ============================================================

ARTIFACT_DIR = "/content/USC_DPO_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

EXPECTED_VERIFIER = (
    "usc_evidence_based_verifier_v4_integrity_gated"
)

MAX_PROMPT_TOKENS = 256
MAX_COMPLETION_TOKENS = 384

EXPECTED_PAIR_COUNTS = {
    "answer_correction": 121,
    "reasoning_correction": 25,
    "step_efficiency": 0,
}

EXPECTED_TOTAL_PAIRS = sum(EXPECTED_PAIR_COUNTS.values())

DETAIL_PATH = os.path.join(
    ARTIFACT_DIR,
    "usc_dpo_preference_pair_audit_v6.json",
)

SUMMARY_PATH = os.path.join(
    ARTIFACT_DIR,
    "usc_dpo_preference_pair_audit_summary_v6.json",
)


# Remove any stale result from a previous audit.
globals().pop("dpo_dataset_audited", None)
pair_audit_passed = False


# ============================================================
# 2. VALIDATE PREREQUISITES
# ============================================================

_REQUIRED = (
    "dpo_dataset_validated",
    "candidate_data",
)

_missing = [
    name for name in _REQUIRED
    if name not in globals()
]

if _missing:
    raise RuntimeError(
        "Missing Cell 36 prerequisites: "
        + ", ".join(_missing)
        + ". Run Cells 24–35 before this audit."
    )

pairs = list(dpo_dataset_validated)
source_candidates = list(candidate_data)

if not pairs:
    raise RuntimeError(
        "dpo_dataset_validated is empty. "
        "No audit or training can proceed."
    )

if not source_candidates:
    raise RuntimeError(
        "candidate_data is empty. "
        "Cannot establish prompt provenance."
    )


# ============================================================
# 3. NORMALIZATION HELPERS
# ============================================================

def audit_normalize_question(value):
    text = "" if value is None else str(value)
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()


def audit_normalize_prompt(value):
    """
    Normalize only line-ending style and outer whitespace.
    Internal prompt content remains unchanged.
    """
    text = "" if value is None else str(value)
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()


def audit_normalize_number(value):
    if value is None:
        return None

    text = (
        str(value)
        .strip()
        .replace(",", "")
        .replace("$", "")
    )

    if not text:
        return None

    try:
        number = Decimal(text)

        if not number.is_finite():
            return None

        if number == number.to_integral_value():
            return str(int(number))

        return format(number.normalize(), "f")

    except (InvalidOperation, ValueError, TypeError):
        return None


def audit_token_count(tokenizer_obj, text, add_special_tokens):
    encoded = tokenizer_obj(
        str(text),
        add_special_tokens=add_special_tokens,
        truncation=False,
    )

    token_ids = encoded["input_ids"]

    if token_ids and isinstance(token_ids[0], list):
        token_ids = token_ids[0]

    return len(token_ids)


def audit_candidate_id(value):
    return str(value)


# ============================================================
# 4. BUILD IMMUTABLE PROMPT REFERENCE FROM CELL 24
# ============================================================

candidate_by_qid = {}

for item in source_candidates:

    try:
        qid = int(item["question_index"])
    except (KeyError, TypeError, ValueError):
        raise RuntimeError(
            "candidate_data contains an invalid question_index."
        )

    if qid in candidate_by_qid:
        raise RuntimeError(
            f"Duplicate candidate_data question_index={qid}."
        )

    question = audit_normalize_question(
        item.get("question")
    )

    prompt = item.get("prompt")

    if not question:
        raise RuntimeError(
            f"Empty source question at question_index={qid}."
        )

    if prompt is None or not str(prompt).strip():
        raise RuntimeError(
            f"Missing source prompt at question_index={qid}."
        )

    candidate_by_qid[qid] = {
        "question": question,
        "prompt": str(prompt),
        "gold_answer": str(
            item.get("gold_answer", "")
        ).strip(),
    }


# ------------------------------------------------------------
# Optional independent source check:
# candidate_data was created from preference_pool in Cell 24.
# Compare against the stored preference_pool prompt where present.
# ------------------------------------------------------------

pool_prompt_checks = 0
pool_prompt_mismatches = []

if "preference_pool" in globals():

    try:
        pool_items = list(preference_pool)
    except Exception:
        pool_items = []

    for qid, source in candidate_by_qid.items():

        if qid >= len(pool_items):
            continue

        pool_item = pool_items[qid]

        pool_question = audit_normalize_question(
            pool_item.get("question")
        )

        pool_prompt = pool_item.get("prompt")

        if pool_question != source["question"]:
            pool_prompt_mismatches.append({
                "question_index": qid,
                "reason": "candidate_data_question_differs_from_preference_pool",
            })
            continue

        if pool_prompt is not None:
            pool_prompt_checks += 1

            if (
                audit_normalize_prompt(pool_prompt)
                != audit_normalize_prompt(source["prompt"])
            ):
                pool_prompt_mismatches.append({
                    "question_index": qid,
                    "reason": "candidate_data_prompt_differs_from_preference_pool",
                })


# ------------------------------------------------------------
# Diagnostic only:
# Check whether the CURRENT make_math_prompt helper still matches
# the Cell 24 prompt. This is not used as the audit reference.
# ------------------------------------------------------------

current_helper_checks = 0
current_helper_mismatches = 0

if callable(globals().get("make_math_prompt")):

    for qid, source in candidate_by_qid.items():

        try:
            current_prompt = make_math_prompt(
                source["question"]
            )

            current_helper_checks += 1

            if (
                audit_normalize_prompt(current_prompt)
                != audit_normalize_prompt(source["prompt"])
            ):
                current_helper_mismatches += 1

        except Exception:
            current_helper_mismatches += 1


# ============================================================
# 5. TOKENIZER
# ============================================================

audit_tokenizer = globals().get("dpo_tokenizer")

if audit_tokenizer is None:
    audit_tokenizer = globals().get("tokenizer")

if audit_tokenizer is None:
    raise RuntimeError(
        "Tokenizer not found. Expected dpo_tokenizer or tokenizer. "
        "Run the existing tokenizer setup cell before auditing."
    )


# ============================================================
# 6. AUDIT EACH PREFERENCE PAIR
# ============================================================

audit_records = []
audit_errors = []

pair_type_counts = Counter()
status_counts = Counter()

seen_pair_ids = set()

max_prompt_tokens_seen = 0
max_chosen_tokens_seen = 0
max_rejected_tokens_seen = 0

token_limit_violations = 0

for pair_index, pair in enumerate(pairs):

    errors = []

    pair_type = str(
        pair.get("pair_type", "answer_correction")
    ).strip().lower()

    pair_type_counts[pair_type] += 1

    # --------------------------------------------------------
    # 6.1 Pair identity
    # --------------------------------------------------------

    try:
        qid = int(pair["question_index"])
    except (KeyError, TypeError, ValueError):
        qid = None
        errors.append("invalid_question_index")

    if qid is None or qid not in candidate_by_qid:

        source = None
        errors.append("question_index_missing_from_candidate_data")

    else:
        source = candidate_by_qid[qid]

    chosen_id = pair.get("chosen_candidate_id")
    rejected_id = pair.get("rejected_candidate_id")

    pair_identity = (
        pair_type,
        qid,
        audit_candidate_id(chosen_id),
        audit_candidate_id(rejected_id),
    )

    if pair_identity in seen_pair_ids:
        errors.append("duplicate_pair_identity")
    else:
        seen_pair_ids.add(pair_identity)

    if (
        chosen_id is not None
        and rejected_id is not None
        and audit_candidate_id(chosen_id)
        == audit_candidate_id(rejected_id)
    ):
        errors.append("chosen_and_rejected_candidate_are_identical")

    # --------------------------------------------------------
    # 6.2 Question and prompt provenance
    # --------------------------------------------------------

    question = audit_normalize_question(
        pair.get("question")
    )

    prompt = str(pair.get("prompt") or "")

    if source is not None:

        if question != source["question"]:
            errors.append(
                "question_mismatch_against_cell24_candidate_data"
            )

        source_prompt = str(source["prompt"])

        if (
            audit_normalize_prompt(prompt)
            != audit_normalize_prompt(source_prompt)
        ):
            errors.append(
                "prompt_mismatch_against_cell24_candidate_data"
            )

    if not prompt.strip():
        errors.append("empty_prompt")

    # --------------------------------------------------------
    # 6.3 Completion presence
    # --------------------------------------------------------

    chosen = str(pair.get("chosen") or "")
    rejected = str(pair.get("rejected") or "")

    if not chosen.strip():
        errors.append("empty_chosen_completion")

    if not rejected.strip():
        errors.append("empty_rejected_completion")

    # --------------------------------------------------------
    # 6.4 Stored answer-label validation
    # --------------------------------------------------------

    gold = audit_normalize_number(
        pair.get("gold_answer")
    )

    chosen_answer = audit_normalize_number(
        pair.get("chosen_answer")
    )

    rejected_answer = audit_normalize_number(
        pair.get("rejected_answer")
    )

    if gold is None:
        errors.append("missing_or_invalid_gold_answer")

    if chosen_answer is None:
        errors.append("missing_or_invalid_chosen_answer")

    if rejected_answer is None:
        errors.append("missing_or_invalid_rejected_answer")

    if (
        gold is not None
        and chosen_answer is not None
        and chosen_answer != gold
    ):
        errors.append("chosen_answer_does_not_match_gold")

    # --------------------------------------------------------
    # 6.5 Pair-type-specific constraints
    # --------------------------------------------------------

    if pair_type == "answer_correction":

        if (
            gold is not None
            and rejected_answer is not None
            and rejected_answer == gold
        ):
            errors.append(
                "answer_correction_rejected_answer_equals_gold"
            )

        if pair.get("chosen_is_correct") is False:
            errors.append("answer_correction_chosen_flag_is_false")

        if pair.get("rejected_is_correct") is True:
            errors.append("answer_correction_rejected_flag_is_true")

    elif pair_type == "reasoning_correction":

        if (
            gold is not None
            and rejected_answer is not None
            and rejected_answer != gold
        ):
            errors.append(
                "reasoning_correction_final_answers_are_not_equal"
            )

        rejected_reasoning_status = str(
            pair.get("rejected_reasoning_status", "")
        ).upper()

        if rejected_reasoning_status != "INVALID":
            errors.append(
                "reasoning_correction_lacks_INVALID_reasoning_status"
            )

        try:
            invalid_steps = int(
                pair.get("rejected_invalid_steps", 0)
            )
        except (TypeError, ValueError):
            invalid_steps = 0

        if invalid_steps < 1:
            errors.append(
                "reasoning_correction_has_no_confirmed_invalid_step"
            )

        try:
            unknown_steps = int(
                pair.get("rejected_unknown_steps", 0)
            )
        except (TypeError, ValueError):
            unknown_steps = -1

        if unknown_steps != 0:
            errors.append(
                "reasoning_correction_contains_unknown_steps"
            )

        failed_equations = pair.get("failed_equations") or []

        if not failed_equations:
            errors.append(
                "reasoning_correction_has_no_failed_equation_evidence"
            )
        else:
            invalid_equation_evidence = [
                item for item in failed_equations
                if isinstance(item, dict)
                and item.get("is_correct") is False
            ]

            if not invalid_equation_evidence:
                errors.append(
                    "failed_equation_evidence_not_confirmed_false"
                )

        if pair.get("chosen_is_correct") is False:
            errors.append(
                "reasoning_correction_chosen_flag_is_false"
            )

        if pair.get("rejected_is_correct") is False:
            errors.append(
                "reasoning_correction_rejected_final_answer_flag_is_false"
            )

    elif pair_type == "step_efficiency":

        # V5 intentionally disables this category because the
        # current pipeline does not provide semantic certification.
        errors.append(
            "step_efficiency_disabled_without_semantic_certification"
        )

    else:
        errors.append("unsupported_pair_type")

    # --------------------------------------------------------
    # 6.6 Verifier provenance
    # --------------------------------------------------------

    verifier_version = str(
        pair.get("verifier_version", "")
    )

    if verifier_version != EXPECTED_VERIFIER:
        errors.append("unexpected_or_missing_verifier_version")

    # --------------------------------------------------------
    # 6.7 Token-length audit
    # --------------------------------------------------------

    prompt_tokens = None
    chosen_tokens = None
    rejected_tokens = None

    if prompt.strip() and chosen.strip() and rejected.strip():

        try:
            prompt_tokens = audit_token_count(
                audit_tokenizer,
                prompt,
                add_special_tokens=True,
            )

            chosen_tokens = audit_token_count(
                audit_tokenizer,
                chosen,
                add_special_tokens=False,
            )

            rejected_tokens = audit_token_count(
                audit_tokenizer,
                rejected,
                add_special_tokens=False,
            )

            max_prompt_tokens_seen = max(
                max_prompt_tokens_seen,
                prompt_tokens,
            )

            max_chosen_tokens_seen = max(
                max_chosen_tokens_seen,
                chosen_tokens,
            )

            max_rejected_tokens_seen = max(
                max_rejected_tokens_seen,
                rejected_tokens,
            )

            if prompt_tokens > MAX_PROMPT_TOKENS:
                errors.append("prompt_token_limit_exceeded")
                token_limit_violations += 1

            if chosen_tokens > MAX_COMPLETION_TOKENS:
                errors.append("chosen_token_limit_exceeded")
                token_limit_violations += 1

            if rejected_tokens > MAX_COMPLETION_TOKENS:
                errors.append("rejected_token_limit_exceeded")
                token_limit_violations += 1

        except Exception as exc:
            errors.append(
                "tokenization_error: " + repr(exc)
            )

    # --------------------------------------------------------
    # 6.8 Record result
    # --------------------------------------------------------

    status = "PASS" if not errors else "FAIL"
    status_counts[status] += 1

    record = {
        "pair_index": pair_index,
        "pair_type": pair_type,
        "question_index": qid,
        "chosen_candidate_id": chosen_id,
        "rejected_candidate_id": rejected_id,
        "gold_answer": pair.get("gold_answer"),
        "chosen_answer": pair.get("chosen_answer"),
        "rejected_answer": pair.get("rejected_answer"),
        "prompt_tokens": prompt_tokens,
        "chosen_tokens": chosen_tokens,
        "rejected_tokens": rejected_tokens,
        "status": status,
        "errors": errors,
    }

    audit_records.append(record)

    if errors:
        audit_errors.append(record)


# ============================================================
# 7. DATASET-LEVEL CHECKS
# ============================================================

global_errors = []

if len(pairs) != EXPECTED_TOTAL_PAIRS:
    global_errors.append(
        f"Expected {EXPECTED_TOTAL_PAIRS} pairs, got {len(pairs)}."
    )

for pair_type, expected_count in EXPECTED_PAIR_COUNTS.items():

    actual_count = pair_type_counts.get(pair_type, 0)

    if actual_count != expected_count:
        global_errors.append(
            f"{pair_type}: expected {expected_count}, got {actual_count}."
        )

unexpected_pair_types = (
    set(pair_type_counts)
    - set(EXPECTED_PAIR_COUNTS)
)

if unexpected_pair_types:
    global_errors.append(
        "Unexpected pair types: "
        + ", ".join(sorted(unexpected_pair_types))
    )

if pool_prompt_mismatches:
    global_errors.append(
        "Cell 24 candidate_data does not fully match preference_pool."
    )

distinct_questions = len({
    record["question_index"]
    for record in audit_records
    if record["question_index"] is not None
})

# ------------------------------------------------------------
# 7.1 Prompt mismatch diagnostics
# ------------------------------------------------------------

prompt_mismatch_records = [
    record for record in audit_records
    if any(
        "prompt_mismatch" in error
        for error in record["errors"]
    )
]

prompt_diagnostics = []

for record in prompt_mismatch_records[:5]:

    pair = pairs[record["pair_index"]]
    qid = record["question_index"]

    if qid not in candidate_by_qid:
        continue

    expected = candidate_by_qid[qid]["prompt"]
    actual = str(pair.get("prompt") or "")

    prompt_diagnostics.append({
        "pair_index": record["pair_index"],
        "question_index": qid,
        "expected_prompt_length": len(expected),
        "actual_prompt_length": len(actual),
        "expected_prompt_preview": expected[:350],
        "actual_prompt_preview": actual[:350],
    })


# ============================================================
# 8. FINAL AUDIT STATUS
# ============================================================

pair_audit_passed = (
    len(audit_errors) == 0
    and len(global_errors) == 0
    and len(audit_records) == len(pairs)
    and status_counts.get("PASS", 0) == len(pairs)
)

if pair_audit_passed:
    dpo_dataset_audited = [
        dict(pair) for pair in pairs
    ]
else:
    dpo_dataset_audited = None


# ============================================================
# 9. SAVE AUDIT ARTIFACTS
# ============================================================

audit_payload = {
    "audit_version": "usc_dpo_preference_pair_audit_v6",
    "verifier_version": EXPECTED_VERIFIER,
    "pair_count": len(pairs),
    "distinct_source_questions": distinct_questions,
    "pair_type_counts": dict(pair_type_counts),
    "status_counts": dict(status_counts),
    "token_limits": {
        "prompt": MAX_PROMPT_TOKENS,
        "completion": MAX_COMPLETION_TOKENS,
    },
    "tokenizer_available": True,
    "tokenizer_name": (
        getattr(audit_tokenizer, "name_or_path", None)
        or audit_tokenizer.__class__.__name__
    ),
    "maximum_observed_tokens": {
        "prompt": max_prompt_tokens_seen,
        "chosen": max_chosen_tokens_seen,
        "rejected": max_rejected_tokens_seen,
    },
    "token_limit_violations": token_limit_violations,
    "prompt_reference": (
        "Cell 24 candidate_data, joined by question_index"
    ),
    "preference_pool_prompt_checks": pool_prompt_checks,
    "preference_pool_prompt_mismatches": pool_prompt_mismatches,
    "current_make_math_prompt_checks": current_helper_checks,
    "current_make_math_prompt_mismatches": current_helper_mismatches,
    "prompt_mismatch_count": len(prompt_mismatch_records),
    "prompt_diagnostics": prompt_diagnostics,
    "global_errors": global_errors,
    "pair_audit_passed": pair_audit_passed,
    "records": audit_records,
}

summary_payload = {
    "audit_version": "usc_dpo_preference_pair_audit_v6",
    "pair_audit_passed": pair_audit_passed,
    "training_allowed": pair_audit_passed,
    "pair_count": len(pairs),
    "distinct_source_questions": distinct_questions,
    "pair_type_counts": dict(pair_type_counts),
    "status_counts": dict(status_counts),
    "audit_error_count": len(audit_errors),
    "global_error_count": len(global_errors),
    "token_limit_violations": token_limit_violations,
    "maximum_observed_tokens": {
        "prompt": max_prompt_tokens_seen,
        "chosen": max_chosen_tokens_seen,
        "rejected": max_rejected_tokens_seen,
    },
    "prompt_reference": (
        "Cell 24 candidate_data, joined by question_index"
    ),
    "preference_pool_prompt_checks": pool_prompt_checks,
    "preference_pool_prompt_mismatches": len(pool_prompt_mismatches),
    "current_make_math_prompt_checks": current_helper_checks,
    "current_make_math_prompt_mismatches": current_helper_mismatches,
    "prompt_mismatch_count": len(prompt_mismatch_records),
    "global_errors": global_errors,
}

with open(DETAIL_PATH, "w", encoding="utf-8") as f:
    json.dump(
        audit_payload,
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(
        summary_payload,
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )


# ============================================================
# 10. REPORT
# ============================================================

print("=" * 72)
print("USC-DPO — PREFERENCE PAIR AUDIT V6")
print("=" * 72)

print("Prompt reference: Cell 24 candidate_data")
print("Prompt join key: question_index")
print("Current make_math_prompt is diagnostic only.")
print("Preference pairs:", len(pairs))
print("Distinct source questions:", distinct_questions)

print("\n===== PAIR TYPE COUNTS =====")

for pair_type in (
    "answer_correction",
    "reasoning_correction",
    "step_efficiency",
):
    print(
        f"{pair_type}: "
        f"{pair_type_counts.get(pair_type, 0)}"
    )

print("\n===== PAIR AUDIT STATUS =====")
print("Passed:", status_counts.get("PASS", 0))
print("Failed:", status_counts.get("FAIL", 0))
print("Pair audit errors:", len(audit_errors))
print("Dataset-level errors:", len(global_errors))

print("\n===== PROMPT PROVENANCE =====")
print("Cell 24 candidate prompts:", len(candidate_by_qid))
print("Preference-pool prompt checks:", pool_prompt_checks)
print("Preference-pool prompt mismatches:",
      len(pool_prompt_mismatches))
print("Current make_math_prompt checks:", current_helper_checks)
print("Current helper differs from Cell 24:",
      current_helper_mismatches)
print("Pair prompt mismatches against Cell 24:",
      len(prompt_mismatch_records))

print("\n===== TOKEN LENGTH AUDIT =====")
print("Tokenizer:", (
    getattr(audit_tokenizer, "name_or_path", None)
    or audit_tokenizer.__class__.__name__
))
print("Maximum prompt tokens:", max_prompt_tokens_seen)
print("Maximum chosen tokens:", max_chosen_tokens_seen)
print("Maximum rejected tokens:", max_rejected_tokens_seen)
print("Prompt token limit:", MAX_PROMPT_TOKENS)
print("Completion token limit:", MAX_COMPLETION_TOKENS)
print("Token-limit violations:", token_limit_violations)

print("\n===== UPLOADED NOTEBOOK CELL 37 SCHEMA =====")
print("Supported pair types in latest uploaded notebook:")
print("- answer_correction")
print("- reasoning_correction")
print("- step_efficiency")
print("Reasoning-correction validation branch: present")
print("Note: this confirms the uploaded notebook source, "
      "not independently edited live Colab code.")

if prompt_diagnostics:
    print("\n===== FIRST PROMPT MISMATCH DIAGNOSTICS =====")

    for item in prompt_diagnostics:
        print(json.dumps(
            item,
            ensure_ascii=False,
            indent=2,
        ))

if global_errors:
    print("\n===== DATASET-LEVEL ERRORS =====")

    for error in global_errors:
        print("-", error)

if audit_errors:
    print("\n===== PAIR ERRORS (FIRST 20) =====")

    for record in audit_errors[:20]:
        print(json.dumps(
            record,
            ensure_ascii=False,
            default=str,
        ))

print("\n===== SAVED AUDIT =====")
print("Detailed audit:", DETAIL_PATH)
print("Summary:", SUMMARY_PATH)

print("\n===== FINAL STATUS =====")

if pair_audit_passed:

    print("PAIR AUDIT: PASS")
    print("AUDITED PAIRS:", len(dpo_dataset_audited))
    print("TRAINING: ALLOWED TO PROCEED TO CELL 37")

else:

    print("PAIR AUDIT: FAIL")
    print("TRAINING: BLOCKED")
    print("dpo_dataset_audited is None.")
    print("No preference pair was regenerated or modified.")

print("\nCell 36 V6 completed.")

USC-DPO — PREFERENCE PAIR AUDIT V6
Prompt reference: Cell 24 candidate_data
Prompt join key: question_index
Current make_math_prompt is diagnostic only.
Preference pairs: 146
Distinct source questions: 68

===== PAIR TYPE COUNTS =====
answer_correction: 121
reasoning_correction: 25
step_efficiency: 0

===== PAIR AUDIT STATUS =====
Passed: 146
Failed: 0
Pair audit errors: 0
Dataset-level errors: 0

===== PROMPT PROVENANCE =====
Cell 24 candidate prompts: 80
Preference-pool prompt checks: 80
Preference-pool prompt mismatches: 0
Current make_math_prompt checks: 80
Current helper differs from Cell 24: 0
Pair prompt mismatches against Cell 24: 0

===== TOKEN LENGTH AUDIT =====
Tokenizer: Qwen/Qwen2.5-Math-1.5B
Maximum prompt tokens: 205
Maximum chosen tokens: 310
Maximum rejected tokens: 307
Prompt token limit: 256
Completion token limit: 384
Token-limit violations: 0

===== UPLOADED NOTEBOOK CELL 37 SCHEMA =====
Supported pair types in latest uploaded notebook:
- answer_correction
- reason

In [56]:

# USC-DPO — Verified Mixed Preference Retraining
# Supports three separately audited preference objectives:
#   1. answer_correction: chosen answer correct, rejected answer wrong
#   2. reasoning_correction: both answers correct, rejected chain has confirmed INVALID steps
#   3. step_efficiency: both answers correct, chosen chain is shorter and verified
#
# Consumes Cell 35's audited dpo_dataset_validated. Stored labels remain
# authoritative; this cell does not silently relabel examples from its parser.

import os
import re
import gc
import json
import random
import zipfile
from decimal import Decimal

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig, get_peft_model


# =========================================================
# 1. Configuration
# =========================================================

SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"

NUM_EPOCHS = 3
LEARNING_RATE = 1e-6
DPO_BETA = 0.1

MAX_PROMPT_TOKENS = 256
MAX_COMPLETION_TOKENS = 384
MAX_GRAD_NORM = 1.0
LOG_EVERY = 10

ARTIFACT_DIR = "/content/USC_DPO_artifacts"
SFT_ADAPTER_DIR = os.path.join(
    ARTIFACT_DIR, "sft_lora_adapter"
)
ADAPTER_DIR = os.path.join(
    ARTIFACT_DIR, "dpo_lora_adapter"
)

os.makedirs(ARTIFACT_DIR, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required.")

torch.cuda.manual_seed_all(SEED)

if not os.path.isfile(
    os.path.join(SFT_ADAPTER_DIR, "adapter_config.json")
):
    raise FileNotFoundError(
        f"SFT adapter missing: {SFT_ADAPTER_DIR}"
    )


# =========================================================
# 2. Load the existing preference pairs
# =========================================================

training_pairs_raw = globals().get(
    "dpo_dataset_validated", None
)

if not training_pairs_raw:
    pairs_path = os.path.join(
        ARTIFACT_DIR,
        "dpo_preference_pairs_original_validated.json",
    )

    if not os.path.isfile(pairs_path):
        raise FileNotFoundError(
            "Preference dataset missing. No training started."
        )

    with open(pairs_path, "r", encoding="utf-8") as f:
        training_pairs_raw = json.load(f)

if not training_pairs_raw:
    raise RuntimeError("Preference dataset is empty.")


# =========================================================
# 3. Answer parsing and conservative completion cleanup
# =========================================================

def normalize_number(value):
    try:
        value = str(value).strip().replace(",", "").replace("$", "")
        number = Decimal(value)

        if not number.is_finite():
            return None

        if number == number.to_integral_value():
            return str(int(number))

        return format(number.normalize(), "f")
    except Exception:
        return None


def last_number(text):
    values = re.findall(
        r"[-+]?\d[\d,]*(?:\.\d+)?",
        str(text or ""),
    )
    if not values:
        return None
    return normalize_number(values[-1])


def parse_answer(text):
    """Best-effort independent parser. Stored labels remain authoritative."""
    text = str(text or "")
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    if not lines:
        return None

    # GSM8K marker: use the result after the last '=' on the answer line.
    for line in reversed(lines):
        if "####" in line:
            suffix = line.split("####", 1)[1].strip()
            if suffix:
                result = last_number(suffix.rsplit("=", 1)[-1])
                if result is not None:
                    return result

    # Simple boxed number.
    boxed = list(re.finditer(
        r"\\boxed\s*\{\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*\}",
        text,
    ))
    if boxed:
        result = normalize_number(boxed[-1].group(1))
        if result is not None:
            return result

    # Explicit answer cue. Handles "12 * (50/60) = 10".
    answer_cue = re.compile(
        r"\b(?:final answer|the answer is|answer is|answer)\b\s*:?\s*",
        re.IGNORECASE,
    )

    for line in reversed(lines):
        match = answer_cue.search(line)
        if match:
            suffix = line[match.end():].strip()
            result = last_number(suffix.rsplit("=", 1)[-1])
            if result is not None:
                return result

    return last_number(lines[-1].rsplit("=", 1)[-1])


def balanced_box_end(text, box_start):
    brace_start = text.find("{", box_start)
    if brace_start < 0:
        return None

    depth = 0
    for pos in range(brace_start, len(text)):
        if text[pos] == "{":
            depth += 1
        elif text[pos] == "}":
            depth -= 1
            if depth == 0:
                return pos + 1

    return None


def trim_after_final_answer(text):
    """
    Remove fenced code blocks and trailing text after a clear final answer.
    Caller accepts the cleaned result only when its answer matches the
    dataset label; otherwise the original completion is preserved.
    """
    text = str(text or "").strip()

    text = re.sub(
        r"(?ms)^[ \t]*```[^\n]*\n.*?^[ \t]*```[ \t]*$",
        "",
        text,
    )
    text = re.sub(
        r"(?ms)^[ \t]*```[^\n]*\n.*\Z",
        "",
        text,
    )
    text = re.sub(r"<<.*?>>", "", text)
    text = "\n".join(
        line.rstrip() for line in text.splitlines()
    ).strip()

    lines = text.splitlines()

    # Prefer an explicit final-answer marker.
    for i in range(len(lines) - 1, -1, -1):
        if "####" not in lines[i]:
            continue

        suffix = lines[i].split("####", 1)[1].strip()
        if last_number(suffix.rsplit("=", 1)[-1]) is not None:
            return "\n".join(lines[:i + 1]).strip()

    # Next: preserve through the last boxed answer.
    box_starts = [
        match.start()
        for match in re.finditer(r"\\boxed\s*\{", text)
    ]

    if box_starts:
        end = balanced_box_end(text, box_starts[-1])
        if end is not None:
            return text[:end].strip()

    # Next: preserve through the last explicit answer-cue line.
    cue = re.compile(
        r"\b(?:final answer|the answer is|answer is|answer)\b\s*:?",
        re.IGNORECASE,
    )

    for i in range(len(lines) - 1, -1, -1):
        if cue.search(lines[i]):
            return "\n".join(lines[:i + 1]).strip()

    return text


# Parser regression checks.
assert parse_answer(
    "The answer is 12 * (50/60) = 10"
) == "10"

assert parse_answer("Work\n#### 16") == "16"

assert parse_answer(
    r"Answer: $\boxed{\color{blue}{45}}$"
) == "45"

print("Training answer-parser regression checks: PASS")


# =========================================================
# 4. Validate stored labels; clean only when safe
# =========================================================

training_pairs = []
validation_audit = []

dropped_efficiency_pairs = 0
cleanup_accepted = 0
cleanup_fallback = 0
parser_unparsed_after_cleanup = 0
parser_label_disagreements = 0

for i, raw in enumerate(training_pairs_raw):
    pair = dict(raw)

    pair_type = pair.get("pair_type", "answer_correction")

    if pair_type not in {
        "answer_correction",
        "reasoning_correction",
        "step_efficiency",
    }:
        raise ValueError(
            f"Unknown pair_type at pair {i}: {pair_type!r}"
        )

    prompt = str(pair.get("prompt", "")).strip()
    original_chosen = str(pair.get("chosen", "")).strip()
    original_rejected = str(pair.get("rejected", "")).strip()

    if not prompt or not original_chosen or not original_rejected:
        raise ValueError(f"Empty prompt/completion at pair {i}.")

    # These labels were explicitly stored and validated in Cell 35.
    gold = normalize_number(pair.get("gold_answer"))
    chosen_label = normalize_number(pair.get("chosen_answer"))
    rejected_label = normalize_number(pair.get("rejected_answer"))

    if gold is None or chosen_label is None or rejected_label is None:
        raise ValueError(
            f"Missing stored answer labels at pair {i}. "
            "No GPU training started."
        )

    if chosen_label != gold:
        raise ValueError(
            f"Stored chosen label disagrees with gold at pair {i}. "
            "No GPU training started."
        )

    if pair_type == "answer_correction":
        if rejected_label == gold:
            raise ValueError(
                f"Answer-correction pair {i} has a rejected answer equal to gold. "
                "No GPU training started."
            )

    elif pair_type == "reasoning_correction":
        if rejected_label != gold:
            raise ValueError(
                f"Reasoning-correction pair {i} must have the same correct final answer on both sides. "
                "No GPU training started."
            )
        if str(pair.get("rejected_reasoning_status", "")).upper() != "INVALID":
            raise ValueError(
                f"Reasoning-correction pair {i} lacks an INVALID reasoning label. "
                "No GPU training started."
            )
        try:
            confirmed_invalid_steps = int(pair.get("rejected_invalid_steps", 0))
        except (TypeError, ValueError):
            confirmed_invalid_steps = 0
        if confirmed_invalid_steps < 1:
            raise ValueError(
                f"Reasoning-correction pair {i} does not include a confirmed invalid step. "
                "No GPU training started."
            )

    else:  # step_efficiency
        if rejected_label != gold:
            raise ValueError(
                f"Efficiency pair {i} does not have two correct labels. "
                "No GPU training started."
            )
        if str(pair.get("chosen_reasoning_status", "")).upper() != "VERIFIED_GOOD":
            raise ValueError(
                f"Efficiency pair {i} lacks verified-good chosen reasoning. "
                "No GPU training started."
            )

    # Try to clean trailing code/prose.
    candidate_chosen = trim_after_final_answer(original_chosen)
    candidate_rejected = trim_after_final_answer(original_rejected)

    parsed_chosen = parse_answer(candidate_chosen)
    parsed_rejected = parse_answer(candidate_rejected)

    # Accept a cleaned string only if the independent parser confirms its
    # answer matches the already-validated stored label.
    if parsed_chosen == chosen_label:
        chosen = candidate_chosen
        cleanup_accepted += int(chosen != original_chosen)
    else:
        chosen = original_chosen
        cleanup_fallback += 1

    if parsed_rejected == rejected_label:
        rejected = candidate_rejected
        cleanup_accepted += int(rejected != original_rejected)
    else:
        rejected = original_rejected
        cleanup_fallback += 1

    # Record parser uncertainty/disagreement instead of failing just because
    # the independent parser cannot parse a format Cell 34 already handled.
    final_parse_chosen = parse_answer(chosen)
    final_parse_rejected = parse_answer(rejected)

    if final_parse_chosen is None or final_parse_rejected is None:
        parser_unparsed_after_cleanup += 1

    if (
        final_parse_chosen is not None
        and final_parse_chosen != chosen_label
    ):
        parser_label_disagreements += 1

    if (
        final_parse_rejected is not None
        and final_parse_rejected != rejected_label
    ):
        parser_label_disagreements += 1

    pair["chosen"] = chosen
    pair["rejected"] = rejected
    pair["gold_answer"] = gold
    pair["chosen_answer"] = chosen_label
    pair["rejected_answer"] = rejected_label

    # Both sides must remain parseable for model training, even when no
    # cleanup was safe. An unparsed side is not automatically relabeled.
    if not chosen or not rejected:
        raise ValueError(f"Empty completion after validation at pair {i}.")

    if pair_type == "step_efficiency":
        # The filter preference is valid only if chosen remains shorter.
        if len(chosen) >= len(rejected):
            dropped_efficiency_pairs += 1
            validation_audit.append({
                "pair_index": i,
                "pair_type": pair_type,
                "status": "dropped",
                "reason": "chosen_not_shorter_after_safe_cleanup",
            })
            continue

    training_pairs.append(pair)
    validation_audit.append({
        "pair_index": i,
        "pair_type": pair_type,
        "status": "accepted",
        "gold_answer": gold,
        "chosen_label": chosen_label,
        "rejected_label": rejected_label,
        "rejected_reasoning_status": pair.get("rejected_reasoning_status"),
        "rejected_invalid_steps": pair.get("rejected_invalid_steps"),
        "chosen_reasoning_status": pair.get("chosen_reasoning_status"),
        "chosen_parser_result": final_parse_chosen,
        "rejected_parser_result": final_parse_rejected,
    })


if not training_pairs:
    raise RuntimeError(
        "No preference pairs remain. No GPU training started."
    )

pair_type_counts = {}
for pair in training_pairs:
    kind = pair["pair_type"]
    pair_type_counts[kind] = pair_type_counts.get(kind, 0) + 1

distinct_questions = len({
    str(pair.get("question_index"))
    for pair in training_pairs
})

print("\n" + "=" * 64)
print("USC-DPO — MIXED DATASET VALIDATION")
print("=" * 64)
print("Validated pairs:", len(training_pairs))
print("Answer-correction pairs:", pair_type_counts.get(
    "answer_correction", 0
))
print("Reasoning-correction pairs:", pair_type_counts.get(
    "reasoning_correction", 0
))
print("Step-efficiency pairs:", pair_type_counts.get(
    "step_efficiency", 0
))
print("Distinct source questions:", distinct_questions)
print("Safe cleanup operations accepted:", cleanup_accepted)
print("Cleanup fallbacks to original text:", cleanup_fallback)
print("Unparsed completions retained using stored labels:",
      parser_unparsed_after_cleanup)
print("Independent parser label disagreements:",
      parser_label_disagreements)
print("Efficiency pairs dropped:", dropped_efficiency_pairs)
print("GPU training has not started yet.")


# Save validation audit before GPU allocation.
audit_path = os.path.join(
    ARTIFACT_DIR, "usc_dpo_mixed_training_validation_audit.json"
)

with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(
        validation_audit,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================
# 5. Load the SFT initialization
# =========================================================

for name in (
    "candidate_model",
    "dpo_model",
    "dpo_base_model",
    "dpo_base",
    "dpo_tokenizer",
):
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass

gc.collect()
torch.cuda.empty_cache()

print("\n" + "=" * 64)
print("USC-DPO — MIXED PREFERENCE RETRAINING")
print("=" * 64)
print("Base model:", MODEL_NAME)
print("Initialization: SFT adapter merged into base")
print("Preference pairs:", len(training_pairs))
print("Distinct source questions:", distinct_questions)
print("Epochs:", NUM_EPOCHS)
print("Planned optimizer steps:", NUM_EPOCHS * len(training_pairs))
print("Learning rate:", LEARNING_RATE)
print("DPO beta:", DPO_BETA)
print("LoRA target modules: q_proj, v_proj")
print("GPU:", torch.cuda.get_device_name(0))

dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

dpo_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if dpo_tokenizer.pad_token_id is None:
    dpo_tokenizer.pad_token = dpo_tokenizer.eos_token

dpo_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)

sft_initialized = PeftModel.from_pretrained(
    dpo_base,
    SFT_ADAPTER_DIR,
    is_trainable=False,
)

dpo_base_model = sft_initialized.merge_and_unload()

del dpo_base, sft_initialized
gc.collect()
torch.cuda.empty_cache()

dpo_base_model.config.use_cache = False

try:
    dpo_base_model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
except TypeError:
    dpo_base_model.gradient_checkpointing_enable()

if hasattr(dpo_base_model, "enable_input_require_grads"):
    dpo_base_model.enable_input_require_grads()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

dpo_model = get_peft_model(dpo_base_model, lora_config)

del dpo_base_model
gc.collect()
torch.cuda.empty_cache()

dpo_model.print_trainable_parameters()

trainable_params = [
    parameter
    for parameter in dpo_model.parameters()
    if parameter.requires_grad
]

if not trainable_params:
    raise RuntimeError("No trainable LoRA parameters found.")

device = next(dpo_model.parameters()).device
pad_id = dpo_tokenizer.pad_token_id
eos_id = dpo_tokenizer.eos_token_id


# =========================================================
# 6. Tokenize pairs
# =========================================================

def encode_text(prompt, completion):
    prompt_ids = dpo_tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_PROMPT_TOKENS,
    )["input_ids"]

    completion_ids = dpo_tokenizer(
        completion,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_COMPLETION_TOKENS,
    )["input_ids"]

    if not prompt_ids:
        raise ValueError("Empty prompt after tokenization.")
    if not completion_ids:
        raise ValueError("Empty completion after tokenization.")

    if eos_id is not None and completion_ids[-1] != eos_id:
        completion_ids.append(eos_id)

    return {
        "input_ids": prompt_ids + completion_ids,
        "completion_start": len(prompt_ids),
    }


encoded_pairs = [
    {
        "chosen": encode_text(pair["prompt"], pair["chosen"]),
        "rejected": encode_text(pair["prompt"], pair["rejected"]),
        "question_index": pair.get("question_index"),
        "pair_type": pair["pair_type"],
    }
    for pair in training_pairs
]

print("Pairs encoded:", len(encoded_pairs))


# =========================================================
# 7. Completion log probabilities
# =========================================================

def sequence_log_probability(model, encoded):
    ids = torch.tensor(
        [encoded["input_ids"]],
        dtype=torch.long,
        device=device,
    )
    attention_mask = torch.ones_like(ids)

    outputs = model(
        input_ids=ids,
        attention_mask=attention_mask,
        use_cache=False,
    )

    logits = outputs.logits[:, :-1, :].float()
    labels = ids[:, 1:]

    positions = torch.arange(1, ids.shape[1], device=device)
    completion_mask = (
        positions >= encoded["completion_start"]
    ).unsqueeze(0)

    log_probs = F.log_softmax(logits, dim=-1)
    token_logps = log_probs.gather(
        dim=-1,
        index=labels.unsqueeze(-1),
    ).squeeze(-1)

    score = (token_logps * completion_mask).sum()

    del outputs, logits, log_probs, token_logps
    return score


# =========================================================
# 8. DPO training
# =========================================================

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=0.0,
)

global_step = 0
training_history = []

metrics_path = os.path.join(
    ARTIFACT_DIR, "dpo_training_metrics.json"
)
history_path = os.path.join(
    ARTIFACT_DIR, "dpo_training_history.json"
)
pairs_used_path = os.path.join(
    ARTIFACT_DIR, "dpo_training_pairs_used.json"
)

def json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (set, tuple)):
        values = list(obj)
        if isinstance(obj, set):
            values = sorted(values, key=str)
        return [json_safe(v) for v in values]
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if hasattr(obj, "item") and callable(obj.item):
        try:
            return obj.item()
        except Exception:
            pass
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def write_metrics():
    metrics = {
        "method": "USC-DPO",
        "base_model": MODEL_NAME,
        "initialization": "SFT adapter merged into base",
        "training_pairs": len(training_pairs),
        "pair_type_counts": pair_type_counts,
        "distinct_source_questions": distinct_questions,
        "epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "beta": DPO_BETA,
        "seed": SEED,
        "optimizer_steps": global_step,
        "cleanup_fallbacks": cleanup_fallback,
        "parser_unparsed_after_cleanup": parser_unparsed_after_cleanup,
        "parser_label_disagreements": parser_label_disagreements,
        "lora": {
            "r": 8,
            "alpha": 16,
            "dropout": 0.05,
            "target_modules": ["q_proj", "v_proj"],
        },
        "training_history": training_history,
    }

    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(
            json_safe(metrics),
            f,
            ensure_ascii=False,
            indent=2,
        )

    with open(history_path, "w", encoding="utf-8") as f:
        json.dump(
            json_safe(training_history),
            f,
            ensure_ascii=False,
            indent=2,
        )


with open(pairs_used_path, "w", encoding="utf-8") as f:
    json.dump(
        json_safe(training_pairs),
        f,
        ensure_ascii=False,
        indent=2,
    )

for epoch in range(NUM_EPOCHS):
    order = list(range(len(encoded_pairs)))
    random.Random(SEED + epoch).shuffle(order)

    dpo_model.train()

    losses = []
    margins = []
    type_losses = {}
    type_margins = {}

    print(f"\n===== EPOCH {epoch + 1}/{NUM_EPOCHS} =====")

    for step, index in enumerate(order, start=1):
        pair = encoded_pairs[index]
        pair_type = pair["pair_type"]

        dpo_model.eval()

        with torch.no_grad():
            with dpo_model.disable_adapter():
                ref_chosen = sequence_log_probability(
                    dpo_model, pair["chosen"]
                )
                ref_rejected = sequence_log_probability(
                    dpo_model, pair["rejected"]
                )

        dpo_model.train()

        policy_chosen = sequence_log_probability(
            dpo_model, pair["chosen"]
        )
        policy_rejected = sequence_log_probability(
            dpo_model, pair["rejected"]
        )

        policy_ratio = policy_chosen - policy_rejected
        reference_ratio = ref_chosen - ref_rejected

        margin = DPO_BETA * (policy_ratio - reference_ratio)
        loss = -F.logsigmoid(margin)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch + 1}, step {step}."
            )

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            MAX_GRAD_NORM,
        )
        optimizer.step()

        global_step += 1

        loss_value = float(loss.detach().cpu())
        margin_value = float(margin.detach().cpu())

        losses.append(loss_value)
        margins.append(margin_value)
        type_losses.setdefault(pair_type, []).append(loss_value)
        type_margins.setdefault(pair_type, []).append(margin_value)

        training_history.append({
            "epoch": epoch + 1,
            "step": global_step,
            "pair_index": step,
            "question_index": pair["question_index"],
            "pair_type": pair_type,
            "loss": loss_value,
            "preference_margin": margin_value,
        })

        if step % LOG_EVERY == 0 or step == len(order):
            print(
                f"Step {step}/{len(order)}"
                f" | loss={loss_value:.4f}"
                f" | margin={margin_value:.4f}",
                flush=True,
            )

        del ref_chosen, ref_rejected
        del policy_chosen, policy_rejected
        del policy_ratio, reference_ratio, margin, loss

    print(f"Epoch mean loss: {sum(losses) / len(losses):.4f}")
    print(f"Epoch mean margin: {sum(margins) / len(margins):.4f}")

    for kind in sorted(type_losses):
        print(
            f"{kind}: "
            f"mean loss={sum(type_losses[kind]) / len(type_losses[kind]):.4f}, "
            f"mean margin={sum(type_margins[kind]) / len(type_margins[kind]):.4f}"
        )

    # Save after each completed epoch.
    dpo_model.save_pretrained(ADAPTER_DIR)
    dpo_tokenizer.save_pretrained(ADAPTER_DIR)
    write_metrics()

    gc.collect()
    torch.cuda.empty_cache()


# =========================================================
# 9. Final artifact save
# =========================================================

dpo_model.save_pretrained(ADAPTER_DIR)
dpo_tokenizer.save_pretrained(ADAPTER_DIR)
write_metrics()

zip_path = "/content/USC_DPO_training_artifacts.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, filenames in os.walk(ADAPTER_DIR):
        for filename in filenames:
            path = os.path.join(root, filename)
            zf.write(
                path,
                arcname=os.path.relpath(path, ARTIFACT_DIR),
            )

    for path in (
        metrics_path,
        history_path,
        pairs_used_path,
        audit_path,
    ):
        if os.path.isfile(path):
            zf.write(
                path,
                arcname=os.path.basename(path),
            )

print("\n" + "=" * 64)
print("USC-DPO — MIXED PREFERENCE TRAINING FINISHED")
print("=" * 64)
print("Pairs:", len(training_pairs))
print("Pair types:", pair_type_counts)
print("Distinct source questions:", distinct_questions)
print("Epochs:", NUM_EPOCHS)
print("Optimizer steps:", global_step)
print("Final loss:", training_history[-1]["loss"])
print("Adapter:", ADAPTER_DIR)
print("Metrics:", metrics_path)
print("Training pairs used:", pairs_used_path)
print("ZIP:", zip_path)
print("ZIP size (MB):", round(os.path.getsize(zip_path) / 1024**2, 2))

del dpo_model
gc.collect()
torch.cuda.empty_cache()

try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print("Automatic download unavailable:", repr(exc))
    print("ZIP remains at:", zip_path)


Training answer-parser regression checks: PASS

USC-DPO — MIXED DATASET VALIDATION
Validated pairs: 146
Answer-correction pairs: 121
Reasoning-correction pairs: 25
Step-efficiency pairs: 0
Distinct source questions: 68
Safe cleanup operations accepted: 260
Cleanup fallbacks to original text: 3
Unparsed completions retained using stored labels: 0
Independent parser label disagreements: 2
Efficiency pairs dropped: 0
GPU training has not started yet.

USC-DPO — MIXED PREFERENCE RETRAINING
Base model: Qwen/Qwen2.5-Math-1.5B
Initialization: SFT adapter merged into base
Preference pairs: 146
Distinct source questions: 68
Epochs: 3
Planned optimizer steps: 438
Learning rate: 1e-06
DPO beta: 0.1
LoRA target modules: q_proj, v_proj
GPU: Tesla T4


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705
Pairs encoded: 146

===== EPOCH 1/3 =====
Step 10/146 | loss=0.6845 | margin=0.0174
Step 20/146 | loss=0.7011 | margin=-0.0158
Step 30/146 | loss=0.6929 | margin=0.0004
Step 40/146 | loss=0.6775 | margin=0.0316
Step 50/146 | loss=0.7155 | margin=-0.0443
Step 60/146 | loss=0.7013 | margin=-0.0163
Step 70/146 | loss=0.7031 | margin=-0.0197
Step 80/146 | loss=0.6810 | margin=0.0244
Step 90/146 | loss=0.6961 | margin=-0.0059
Step 100/146 | loss=0.7021 | margin=-0.0178
Step 110/146 | loss=0.6959 | margin=-0.0054
Step 120/146 | loss=0.6693 | margin=0.0483
Step 130/146 | loss=0.6992 | margin=-0.0120
Step 140/146 | loss=0.6714 | margin=0.0439
Step 146/146 | loss=0.7031 | margin=-0.0198
Epoch mean loss: 0.6967
Epoch mean margin: -0.0067
answer_correction: mean loss=0.6975, mean margin=-0.0084
reasoning_correction: mean loss=0.6924, mean margin=0.0018

===== EPOCH 2/3 =====
Step 10/146 | loss=0.6806 | margin=0.0253
St

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [60]:
# USC-DPO — Audit and Training Artifact Recovery
# This cell never trains or reloads a language model.
# It packages what exists and explicitly reports missing artifacts.

import os
import json
import zipfile
from pathlib import Path

ARTIFACT_DIR = "/content/USC_DPO_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

ARTIFACT_FILES = [
    "usc_dpo_candidate_quality_audit.json",
    "usc_dpo_gold_reference_pairs_audit.json",
    "usc_dpo_gold_reference_summary.json",
    "dpo_preference_pairs_original_validated.json",
    "dpo_training_metrics.json",
    "dpo_training_history.json",
    "dpo_training_pairs_used.json",
    "usc_dpo_mixed_training_validation_audit.json",
    "final_evaluation_summary.json",
    "final_evaluation_predictions.json",
    "final_evaluation_comparison.csv",
]

ADAPTER_DIRS = {
    "SFT adapter": os.path.join(ARTIFACT_DIR, "sft_lora_adapter"),
    "USC-DPO adapter": os.path.join(ARTIFACT_DIR, "dpo_lora_adapter"),
}

print("=" * 68)
print("USC-DPO — ARTIFACT INVENTORY")
print("=" * 68)

included_files = []
missing_files = []
for name in ARTIFACT_FILES:
    path = os.path.join(ARTIFACT_DIR, name)
    if os.path.isfile(path):
        included_files.append(path)
        print(f"FOUND   {name} ({os.path.getsize(path):,} bytes)")
    else:
        missing_files.append(name)
        print(f"MISSING {name}")

for label, adapter_dir in ADAPTER_DIRS.items():
    config = os.path.join(adapter_dir, "adapter_config.json")
    weight_files = [
        os.path.join(adapter_dir, name)
        for name in ("adapter_model.safetensors", "adapter_model.bin")
    ]
    found_weights = [path for path in weight_files if os.path.isfile(path)]
    ready = os.path.isfile(config) and bool(found_weights)
    print(f"{label}: {'READY' if ready else 'NOT READY'} ({adapter_dir})")

# Recover persisted training metrics only when present; never fabricate a
# completed run from an adapter directory or from stale notebook variables.
metrics_path = os.path.join(ARTIFACT_DIR, "dpo_training_metrics.json")
history_path = os.path.join(ARTIFACT_DIR, "dpo_training_history.json")
if os.path.isfile(metrics_path):
    with open(metrics_path, "r", encoding="utf-8") as f:
        recovered_metrics = json.load(f)
    print("Training metrics loaded from disk.")
else:
    recovered_metrics = None
    print("Training metrics unavailable; no training-completion claim is made.")

if os.path.isfile(history_path):
    with open(history_path, "r", encoding="utf-8") as f:
        recovered_history = json.load(f)
    print("Training-history records:", len(recovered_history))
else:
    recovered_history = None

bundle_path = "/content/USC_DPO_project_artifacts.zip"
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in included_files:
        zf.write(path, arcname=os.path.basename(path))

    for label, adapter_dir in ADAPTER_DIRS.items():
        if os.path.isdir(adapter_dir):
            for root, _, files in os.walk(adapter_dir):
                for filename in files:
                    path = os.path.join(root, filename)
                    zf.write(path, arcname=os.path.relpath(path, ARTIFACT_DIR))

print("\nBundle:", bundle_path)
print("Files packaged:", len(included_files))
print("Missing optional files:", len(missing_files))
print("This cell did not train the model.")

try:
    from google.colab import files
    files.download(bundle_path)
except Exception as exc:
    print("Automatic download unavailable:", repr(exc))
    print("Bundle remains at:", bundle_path)


USC-DPO — ARTIFACT INVENTORY
MISSING usc_dpo_candidate_quality_audit.json
MISSING usc_dpo_gold_reference_pairs_audit.json
MISSING usc_dpo_gold_reference_summary.json
MISSING dpo_preference_pairs_original_validated.json
FOUND   dpo_training_metrics.json (98,896 bytes)
FOUND   dpo_training_history.json (90,394 bytes)
FOUND   dpo_training_pairs_used.json (319,415 bytes)
FOUND   usc_dpo_mixed_training_validation_audit.json (55,213 bytes)
FOUND   final_evaluation_summary.json (4,727 bytes)
FOUND   final_evaluation_predictions.json (305,272 bytes)
FOUND   final_evaluation_comparison.csv (447 bytes)
SFT adapter: READY (/content/USC_DPO_artifacts/sft_lora_adapter)
USC-DPO adapter: READY (/content/USC_DPO_artifacts/dpo_lora_adapter)
Training metrics loaded from disk.
Training-history records: 438

Bundle: /content/USC_DPO_project_artifacts.zip
Files packaged: 7
Missing optional files: 4
This cell did not train the model.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:

# ============================================================
# USC-DPO — FINAL EVALUATION V2
# 100 GSM8K TEST QUESTIONS
#
# Models:
#   1. Base
#   2. SFT-CoT
#   3. USC-DPO
#
# Evaluation:
#   - First 100 questions from official GSM8K test split.
#   - Same questions, prompts, parser, decoding settings.
#   - Strict answer parser.
#   - Checkpoint every 5 predictions.
#   - Resume after Colab interruption.
#   - Paired per-question comparison.
#   - McNemar exact test.
#   - Wilson 95% confidence intervals.
#
# No training is performed in this cell.
# ============================================================

import os
import re
import gc
import csv
import json
import time
import math
import shutil
import hashlib
import zipfile

import torch

from decimal import Decimal
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import PeftModel


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"

NUM_TEST_SAMPLES = 100

MAX_NEW_TOKENS = 384
MAX_INPUT_TOKENS = 1024

REPETITION_PENALTY = 1.05

CHECKPOINT_EVERY = 5

PARSER_VERSION = (
    "strict_marker_boxed_final_line_v2"
)

MODEL_ORDER = (
    "Base",
    "SFT-CoT",
    "USC-DPO",
)

ARTIFACT_DIR = "/content/USC_DPO_artifacts"

SFT_ADAPTER_DIR = os.path.join(
    ARTIFACT_DIR,
    "sft_lora_adapter",
)

DPO_ADAPTER_DIR = os.path.join(
    ARTIFACT_DIR,
    "dpo_lora_adapter",
)

os.makedirs(
    ARTIFACT_DIR,
    exist_ok=True,
)

# Keep the 100-question run separate from the old
# 50-question evaluation and the 1,319-question experiment.

EVALUATION_NAME = "final_evaluation_100_v2"


# ============================================================
# 2. GOOGLE DRIVE CHECKPOINT
# ============================================================

try:

    from google.colab import drive

    if not os.path.isdir(
        "/content/drive/MyDrive"
    ):

        print("Mounting Google Drive...")

        drive.mount("/content/drive")

except ImportError:

    print(
        "Google Colab Drive API unavailable. "
        "Using local checkpoint storage."
    )


DRIVE_ROOT = "/content/drive/MyDrive"

if os.path.isdir(DRIVE_ROOT):

    CHECKPOINT_ROOT = os.path.join(
        DRIVE_ROOT,
        "USC_DPO_artifacts",
        EVALUATION_NAME,
    )

    CHECKPOINT_STORAGE = "Google Drive"

else:

    CHECKPOINT_ROOT = os.path.join(
        ARTIFACT_DIR,
        EVALUATION_NAME,
    )

    CHECKPOINT_STORAGE = "Local Colab storage"

os.makedirs(
    CHECKPOINT_ROOT,
    exist_ok=True,
)

MANIFEST_PATH = os.path.join(
    CHECKPOINT_ROOT,
    "evaluation_manifest.json",
)

MODEL_KEYS = {
    "Base": "base",
    "SFT-CoT": "sft_cot",
    "USC-DPO": "usc_dpo",
}

CHECKPOINT_FILES = {
    model: os.path.join(
        CHECKPOINT_ROOT,
        f"{MODEL_KEYS[model]}_predictions.jsonl",
    )
    for model in MODEL_ORDER
}


# ============================================================
# 3. GPU AND ADAPTER VALIDATION
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU is required. "
        "Enable GPU in Google Colab."
    )


for adapter_dir in (
    SFT_ADAPTER_DIR,
    DPO_ADAPTER_DIR,
):

    config_path = os.path.join(
        adapter_dir,
        "adapter_config.json",
    )

    weight_paths = [
        os.path.join(
            adapter_dir,
            "adapter_model.safetensors",
        ),
        os.path.join(
            adapter_dir,
            "adapter_model.bin",
        ),
    ]

    if not os.path.isfile(config_path):

        raise FileNotFoundError(
            f"Adapter config missing: {config_path}"
        )

    if not any(
        os.path.isfile(path)
        for path in weight_paths
    ):

        raise FileNotFoundError(
            f"Adapter weights missing: {adapter_dir}"
        )


eval_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)


print("=" * 76)
print("USC-DPO — FINAL EVALUATION V2")
print("=" * 76)

print("Base model:", MODEL_NAME)
print("Test questions:", NUM_TEST_SAMPLES)
print("Models:", len(MODEL_ORDER))
print("Expected total predictions:", NUM_TEST_SAMPLES * len(MODEL_ORDER))
print("Max new tokens:", MAX_NEW_TOKENS)
print("Checkpoint interval:", CHECKPOINT_EVERY)
print("Checkpoint storage:", CHECKPOINT_STORAGE)
print("Checkpoint directory:", CHECKPOINT_ROOT)
print("GPU:", torch.cuda.get_device_name(0))
print("Evaluation dtype:", eval_dtype)
print("SFT adapter:", SFT_ADAPTER_DIR)
print("USC-DPO adapter:", DPO_ADAPTER_DIR)


# ============================================================
# 4. LOAD GSM8K TEST SPLIT
# ============================================================

print("\nLoading GSM8K...")

if "gsm8k" not in globals():

    gsm8k = load_dataset(
        "openai/gsm8k",
        "main",
    )


if "test" not in gsm8k:

    raise RuntimeError(
        "GSM8K test split not found."
    )


test_split = gsm8k["test"]

if len(test_split) < NUM_TEST_SAMPLES:

    raise RuntimeError(
        f"Expected at least {NUM_TEST_SAMPLES} test questions, "
        f"found {len(test_split)}."
    )


# Fixed first 100 test questions.
# This preserves overlap with the previous 50-question evaluation.

eval_indices = list(
    range(NUM_TEST_SAMPLES)
)

eval_questions = [
    test_split[i]
    for i in eval_indices
]

print("Loaded:", len(eval_questions), "questions")
print("Test indices:", eval_indices[0], "to", eval_indices[-1])


# ============================================================
# 5. TOKENIZER AND PROMPT
# ============================================================

eval_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

if eval_tokenizer.pad_token_id is None:

    eval_tokenizer.pad_token = (
        eval_tokenizer.eos_token
    )


if not callable(
    globals().get("make_math_prompt")
):

    raise RuntimeError(
        "make_math_prompt not found. "
        "Check Cell14."
    )


def make_eval_prompt(question):

    prompt = make_math_prompt(question)

    if (
        not isinstance(prompt, str)
        or not prompt.strip()
    ):

        raise ValueError(
            "make_math_prompt returned an invalid prompt."
        )

    return prompt


# Generate every prompt once.
# All three models receive the exact same prompt.

eval_prompts = [
    make_eval_prompt(
        example["question"]
    )
    for example in eval_questions
]


# ============================================================
# 6. ANSWER PARSER
# Same strict parsing policy as original Cell39.
# No arbitrary last-number fallback.
# ============================================================

def normalize_numeric_answer(value):

    if value is None:
        return None

    value = (
        str(value)
        .strip()
        .replace(",", "")
        .replace("$", "")
    )

    try:

        number = Decimal(value)

        if not number.is_finite():
            return None

        if number == number.to_integral_value():

            return str(
                int(number)
            )

        return format(
            number.normalize(),
            "f",
        )

    except Exception:

        return None


def extract_boxed_answer(text):

    answers = []

    for match in re.finditer(
        r"\\boxed\s*\{",
        text,
    ):

        start = match.end()

        depth = 1
        pos = start

        while pos < len(text) and depth > 0:

            if text[pos] == "{":
                depth += 1

            elif text[pos] == "}":
                depth -= 1

            pos += 1

        if depth != 0:
            continue

        content = text[start:pos - 1]

        numbers = re.findall(
            r"[-+]?\d[\d,]*(?:\.\d+)?",
            content,
        )

        if len(numbers) == 1:

            answer = normalize_numeric_answer(
                numbers[0]
            )

            if answer is not None:
                answers.append(answer)

        elif numbers:

            answer = normalize_numeric_answer(
                numbers[-1]
            )

            if answer is not None:
                answers.append(answer)

    return (
        answers[-1]
        if answers
        else None
    )


def extract_prediction_detail(text):

    text = str(text or "")

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    number = r"([-+]?\d[\d,]*(?:\.\d+)?)"

    # 1. GSM8K marker #### 123

    marker_pattern = re.compile(
        rf"^\s*####\s*\$?\s*{number}\s*[.!]?\s*$"
    )

    marker_answers = []

    for line in lines:

        match = marker_pattern.fullmatch(line)

        if match:

            value = normalize_numeric_answer(
                match.group(1)
            )

            if value is not None:

                marker_answers.append(
                    (value, line)
                )

    if marker_answers:

        value, evidence = marker_answers[-1]

        return {
            "value": value,
            "method": "gsm8k_marker",
            "evidence": evidence,
        }

    # 2. Boxed answer.

    boxed = extract_boxed_answer(text)

    if boxed is not None:

        return {
            "value": boxed,
            "method": "boxed",
            "evidence": "last parseable \\boxed{...}",
        }

    if not lines:

        return {
            "value": None,
            "method": None,
            "evidence": None,
        }

    # 3. Strict final-line matching.

    final_line = lines[-1]

    final_patterns = [

        (
            re.compile(
                rf"^\s*final\s+answer\s*(?::|=|is)?\s*\$?{number}\s*[.!]?\s*$",
                re.I,
            ),
            "final_answer_line",
        ),

        (
            re.compile(
                rf"^\s*the\s+answer\s+is\s*\$?{number}\s*[.!]?\s*$",
                re.I,
            ),
            "answer_phrase_final_line",
        ),

        (
            re.compile(
                rf"^\s*answer\s*(?::|=)\s*\$?{number}\s*[.!]?\s*$",
                re.I,
            ),
            "answer_label_final_line",
        ),

        (
            re.compile(
                rf"^\s*\$?{number}\s*[.!]?\s*$"
            ),
            "standalone_final_number",
        ),

    ]

    for pattern, method in final_patterns:

        match = pattern.fullmatch(
            final_line
        )

        if match:

            value = normalize_numeric_answer(
                match.group(1)
            )

            if value is not None:

                return {
                    "value": value,
                    "method": method,
                    "evidence": final_line,
                }

    return {
        "value": None,
        "method": None,
        "evidence": final_line[-180:],
    }


def extract_gold(example):

    match = re.search(
        r"####\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)",
        example["answer"],
    )

    if not match:
        return None

    return normalize_numeric_answer(
        match.group(1)
    )


gold_answers = [
    extract_gold(example)
    for example in eval_questions
]


if any(
    answer is None
    for answer in gold_answers
):

    raise RuntimeError(
        "At least one GSM8K gold answer could not be parsed."
    )


# ============================================================
# 7. VALIDATE PROMPT LENGTH
# Do not silently truncate prompts.
# ============================================================

prompt_lengths = []

for prompt in eval_prompts:

    encoded = eval_tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=False,
    )

    prompt_lengths.append(
        len(encoded["input_ids"])
    )


max_prompt_tokens = max(
    prompt_lengths
)

if max_prompt_tokens > MAX_INPUT_TOKENS:

    raise RuntimeError(
        f"Prompt length exceeds {MAX_INPUT_TOKENS}. "
        f"Observed maximum: {max_prompt_tokens}. "
        "Increase MAX_INPUT_TOKENS before evaluating."
    )


print("\nPrompt validation: PASS")
print("Prompt count:", len(eval_prompts))
print("Maximum prompt tokens:", max_prompt_tokens)
print("Maximum input tokens:", MAX_INPUT_TOKENS)


# ============================================================
# 8. FINGERPRINTS — PREVENT WRONG CHECKPOINT RESUME
# ============================================================

def sha256_text(value):

    return hashlib.sha256(
        str(value).encode("utf-8")
    ).hexdigest()


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def adapter_fingerprint(adapter_dir):

    config_path = os.path.join(
        adapter_dir,
        "adapter_config.json",
    )

    weights_path = None

    for filename in (
        "adapter_model.safetensors",
        "adapter_model.bin",
    ):

        candidate = os.path.join(
            adapter_dir,
            filename,
        )

        if os.path.isfile(candidate):

            weights_path = candidate
            break

    if weights_path is None:

        raise FileNotFoundError(
            f"Adapter weights missing: {adapter_dir}"
        )

    return {
        "config_sha256": sha256_file(config_path),
        "weights_sha256": sha256_file(weights_path),
    }


dataset_payload = json.dumps(
    [
        {
            "question": example["question"],
            "answer": example["answer"],
        }
        for example in eval_questions
    ],
    ensure_ascii=False,
    separators=(",", ":"),
)

dataset_fingerprint = sha256_text(
    dataset_payload
)

prompt_fingerprint = sha256_text(
    json.dumps(
        eval_prompts,
        ensure_ascii=False,
        separators=(",", ":"),
    )
)

sft_fingerprint = adapter_fingerprint(
    SFT_ADAPTER_DIR
)

dpo_fingerprint = adapter_fingerprint(
    DPO_ADAPTER_DIR
)


run_configuration = {
    "evaluation_name": EVALUATION_NAME,
    "model_name": MODEL_NAME,
    "dataset": "openai/gsm8k",
    "config": "main",
    "split": "test",
    "num_test_samples": NUM_TEST_SAMPLES,
    "test_indices": eval_indices,
    "dataset_fingerprint": dataset_fingerprint,
    "prompt_fingerprint": prompt_fingerprint,
    "sft_adapter_fingerprint": sft_fingerprint,
    "dpo_adapter_fingerprint": dpo_fingerprint,
    "parser_version": PARSER_VERSION,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "repetition_penalty": REPETITION_PENALTY,
    "do_sample": False,
    "seed": SEED,
}

run_signature = sha256_text(
    json.dumps(
        run_configuration,
        sort_keys=True,
        ensure_ascii=False,
    )
)


print("\nDataset SHA256:", dataset_fingerprint[:20])
print("Prompt SHA256:", prompt_fingerprint[:20])
print(
    "SFT adapter SHA256:",
    sft_fingerprint["weights_sha256"][:20],
)
print(
    "DPO adapter SHA256:",
    dpo_fingerprint["weights_sha256"][:20],
)


# ============================================================
# 9. CHECKPOINT MANIFEST
# ============================================================

def atomic_write_json(path, payload):

    temporary_path = path + ".tmp"

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            ensure_ascii=False,
            indent=2,
        )

        f.flush()
        os.fsync(f.fileno())

    os.replace(
        temporary_path,
        path,
    )


if os.path.isfile(MANIFEST_PATH):

    with open(
        MANIFEST_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        evaluation_manifest = json.load(f)

    if (
        evaluation_manifest.get("run_signature")
        != run_signature
    ):

        raise RuntimeError(
            "Checkpoint belongs to a different evaluation "
            "configuration or adapter version. "
            "No existing checkpoint was overwritten."
        )

    print("\nCompatible checkpoint found.")
    print("Resume mode: ENABLED")

else:

    existing_checkpoints = [
        path
        for path in CHECKPOINT_FILES.values()
        if os.path.isfile(path)
    ]

    if existing_checkpoints:

        raise RuntimeError(
            "Prediction checkpoint files exist without a matching "
            "manifest. Refusing to mix evaluation runs."
        )

    evaluation_manifest = {
        "run_configuration": run_configuration,
        "run_signature": run_signature,
        "checkpoint_storage": CHECKPOINT_STORAGE,
        "model_progress": {
            model: 0
            for model in MODEL_ORDER
        },
        "completed_models": [],
        "status": "in_progress",
    }

    atomic_write_json(
        MANIFEST_PATH,
        evaluation_manifest,
    )

    print("\nNew evaluation checkpoint initialized.")


# ============================================================
# 10. LOAD EXISTING PREDICTIONS
# ============================================================

def load_model_checkpoint(model_name):

    path = CHECKPOINT_FILES[model_name]

    records = {}

    if not os.path.isfile(path):

        return records

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        lines = f.readlines()

    valid_lines = []

    nonempty_lines = [
        line
        for line in lines
        if line.strip()
    ]

    for line_index, line in enumerate(nonempty_lines):

        try:

            row = json.loads(line)

        except json.JSONDecodeError:

            # Only discard an incomplete final line.
            if line_index == len(nonempty_lines) - 1:

                print(
                    f"Recovering incomplete last line: "
                    f"{model_name}"
                )

                break

            raise RuntimeError(
                f"Corrupted checkpoint inside {path}, "
                f"line {line_index + 1}."
            )

        if row.get("model") != model_name:

            raise RuntimeError(
                f"Model identity mismatch in {path}."
            )

        test_index = int(
            row["test_index"]
        )

        if test_index not in eval_indices:

            raise RuntimeError(
                f"Unexpected test index {test_index}."
            )

        if test_index in records:

            raise RuntimeError(
                f"Duplicate test index {test_index}."
            )

        if (
            row.get("question")
            != eval_questions[test_index]["question"]
        ):

            raise RuntimeError(
                "Question mismatch in checkpoint."
            )

        if (
            row.get("gold")
            != gold_answers[test_index]
        ):

            raise RuntimeError(
                "Gold-answer mismatch in checkpoint."
            )

        # Recompute parsing from generated_text.
        # This guarantees the current strict parser is applied.

        detail = extract_prediction_detail(
            row.get("generated_text", "")
        )

        predicted = detail["value"]

        row["prediction"] = predicted
        row["prediction_extraction_method"] = detail["method"]
        row["prediction_extraction_evidence"] = detail["evidence"]
        row["prediction_parse_status"] = (
            "PARSED"
            if predicted is not None
            else "UNPARSED"
        )

        row["correct"] = bool(
            predicted is not None
            and predicted == gold_answers[test_index]
        )

        records[test_index] = row

        valid_lines.append(
            json.dumps(
                row,
                ensure_ascii=False,
            )
        )

    # Repair an incomplete final JSONL line if necessary.
    if len(valid_lines) != len(nonempty_lines):

        with open(
            path,
            "w",
            encoding="utf-8",
        ) as f:

            for line in valid_lines:

                f.write(
                    line + "\n"
                )

    return records


checkpoint_records = {
    model: load_model_checkpoint(model)
    for model in MODEL_ORDER
}


for model in MODEL_ORDER:

    count = len(
        checkpoint_records[model]
    )

    evaluation_manifest[
        "model_progress"
    ][model] = count

    print(
        f"{model}: {count}/{NUM_TEST_SAMPLES} "
        "predictions already saved"
    )


atomic_write_json(
    MANIFEST_PATH,
    evaluation_manifest,
)


# ============================================================
# 11. CHECKPOINT WRITER
# ============================================================

def flush_checkpoint(
    model_name,
    pending_rows,
    records,
):

    if not pending_rows:
        return

    checkpoint_path = CHECKPOINT_FILES[
        model_name
    ]

    with open(
        checkpoint_path,
        "a",
        encoding="utf-8",
    ) as f:

        for row in pending_rows:

            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )

        f.flush()
        os.fsync(f.fileno())

    pending_rows.clear()

    evaluation_manifest[
        "model_progress"
    ][model_name] = len(records)

    evaluation_manifest[
        "last_checkpoint_at"
    ] = time.strftime(
        "%Y-%m-%dT%H:%M:%S",
        time.gmtime(),
    )

    atomic_write_json(
        MANIFEST_PATH,
        evaluation_manifest,
    )


# ============================================================
# 12. LOAD MODEL VARIANT
# ============================================================

def load_evaluation_model(variant_name):

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=eval_dtype,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

    if variant_name == "Base":

        eval_model = base

    elif variant_name == "SFT-CoT":

        eval_model = PeftModel.from_pretrained(
            base,
            SFT_ADAPTER_DIR,
            is_trainable=False,
        )

    elif variant_name == "USC-DPO":

        # Reconstruct the exact training initialization:
        #
        # Original Base
        #     -> SFT adapter
        #     -> merge SFT weights
        #     -> USC-DPO adapter

        sft_initialized = PeftModel.from_pretrained(
            base,
            SFT_ADAPTER_DIR,
            is_trainable=False,
        )

        merged_sft = (
            sft_initialized.merge_and_unload()
        )

        del sft_initialized
        del base

        gc.collect()
        torch.cuda.empty_cache()

        base = merged_sft

        eval_model = PeftModel.from_pretrained(
            base,
            DPO_ADAPTER_DIR,
            is_trainable=False,
        )

    else:

        del base

        raise ValueError(
            f"Unknown model variant: {variant_name}"
        )

    eval_model.eval()

    return eval_model, base


# ============================================================
# 13. EVALUATE ONE MODEL — WITH RESUME
# ============================================================

def evaluate_variant(variant_name):

    print("\n" + "=" * 76)
    print("EVALUATING:", variant_name)
    print("=" * 76, flush=True)

    records = checkpoint_records[
        variant_name
    ]

    if len(records) == NUM_TEST_SAMPLES:

        print(
            f"{variant_name} already completed "
            f"({NUM_TEST_SAMPLES}/{NUM_TEST_SAMPLES}). "
            "Skipping model loading."
        )

        return

    missing_indices = [
        i
        for i in eval_indices
        if i not in records
    ]

    print("Already completed:", len(records))
    print("Questions remaining:", len(missing_indices))

    eval_model = None
    backbone = None

    pending_rows = []

    start_variant = time.time()

    try:

        eval_model, backbone = load_evaluation_model(
            variant_name
        )

        device = next(
            eval_model.parameters()
        ).device

        print("Model loaded on:", device, flush=True)

        eval_model.eval()

        for position, test_index in enumerate(
            missing_indices,
            start=1,
        ):

            example = eval_questions[test_index]

            question = example["question"]
            gold = gold_answers[test_index]
            prompt = eval_prompts[test_index]

            inputs = eval_tokenizer(
                prompt,
                return_tensors="pt",
                add_special_tokens=True,
                truncation=False,
            ).to(device)

            actual_prompt_tokens = (
                inputs["input_ids"].shape[1]
            )

            if actual_prompt_tokens > MAX_INPUT_TOKENS:

                raise RuntimeError(
                    f"Prompt {test_index} exceeds token budget."
                )

            if torch.cuda.is_available():

                torch.cuda.synchronize()

            start_generation = time.time()

            with torch.inference_mode():

                output_ids = eval_model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    repetition_penalty=REPETITION_PENALTY,
                    pad_token_id=eval_tokenizer.pad_token_id,
                    eos_token_id=eval_tokenizer.eos_token_id,
                    use_cache=True,
                )

            if torch.cuda.is_available():

                torch.cuda.synchronize()

            latency = (
                time.time() - start_generation
            )

            new_tokens = output_ids[
                0,
                actual_prompt_tokens:
            ]

            generated_text = eval_tokenizer.decode(
                new_tokens,
                skip_special_tokens=True,
            ).strip()

            prediction_detail = (
                extract_prediction_detail(
                    generated_text
                )
            )

            predicted = prediction_detail["value"]

            correct = bool(
                predicted is not None
                and gold is not None
                and predicted == gold
            )

            row = {
                "model": variant_name,
                "test_index": test_index,
                "question": question,
                "prediction": predicted,
                "prediction_extraction_method": (
                    prediction_detail["method"]
                ),
                "prediction_extraction_evidence": (
                    prediction_detail["evidence"]
                ),
                "prediction_parse_status": (
                    "PARSED"
                    if predicted is not None
                    else "UNPARSED"
                ),
                "gold": gold,
                "correct": correct,
                "latency_seconds": round(
                    latency,
                    4,
                ),
                "generated_tokens": int(
                    new_tokens.numel()
                ),
                "generated_text": generated_text,
            }

            records[test_index] = row

            pending_rows.append(
                row
            )

            if len(pending_rows) >= CHECKPOINT_EVERY:

                flush_checkpoint(
                    variant_name,
                    pending_rows,
                    records,
                )

            if (
                position % 5 == 0
                or position == len(missing_indices)
            ):

                processed = len(records)

                correct_so_far = sum(
                    bool(item["correct"])
                    for item in records.values()
                )

                accuracy_so_far = (
                    100.0
                    * correct_so_far
                    / processed
                )

                print(
                    f"[{processed}/{NUM_TEST_SAMPLES}] "
                    f"Correct={correct_so_far} "
                    f"Accuracy={accuracy_so_far:.2f}% | "
                    f"Last={predicted} | Gold={gold}",
                    flush=True,
                )

            del inputs
            del output_ids
            del new_tokens

        # Save remaining predictions.
        flush_checkpoint(
            variant_name,
            pending_rows,
            records,
        )

        if len(records) != NUM_TEST_SAMPLES:

            raise RuntimeError(
                f"{variant_name}: incomplete results "
                f"{len(records)}/{NUM_TEST_SAMPLES}."
            )

        completed_models = set(
            evaluation_manifest.get(
                "completed_models",
                [],
            )
        )

        completed_models.add(
            variant_name
        )

        evaluation_manifest[
            "completed_models"
        ] = [
            model
            for model in MODEL_ORDER
            if model in completed_models
        ]

        evaluation_manifest[
            "model_progress"
        ][variant_name] = NUM_TEST_SAMPLES

        atomic_write_json(
            MANIFEST_PATH,
            evaluation_manifest,
        )

        elapsed = (
            time.time() - start_variant
        )

        print(
            f"\n{variant_name} completed: "
            f"{NUM_TEST_SAMPLES}/{NUM_TEST_SAMPLES}"
        )

        print(
            "Elapsed minutes:",
            round(elapsed / 60, 2),
        )

    finally:

        # Keep predictions produced before an exception.
        if pending_rows:

            flush_checkpoint(
                variant_name,
                pending_rows,
                records,
            )

        if eval_model is not None:
            del eval_model

        if backbone is not None:
            del backbone

        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# 14. RUN OR RESUME THREE MODELS
# ============================================================

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

overall_start = time.time()

for variant in MODEL_ORDER:

    evaluate_variant(
        variant
    )


print("\n" + "=" * 76)
print("MODEL EVALUATION PROGRESS")
print("=" * 76)

for variant in MODEL_ORDER:

    count = len(
        checkpoint_records[variant]
    )

    print(
        f"{variant}: {count}/{NUM_TEST_SAMPLES}"
    )


if any(
    len(checkpoint_records[variant]) != NUM_TEST_SAMPLES
    for variant in MODEL_ORDER
):

    evaluation_manifest["status"] = "incomplete"

    atomic_write_json(
        MANIFEST_PATH,
        evaluation_manifest,
    )

    raise RuntimeError(
        "Evaluation is incomplete. "
        "Run Cell39 again to resume from saved checkpoints. "
        "No final comparison was exported."
    )


# ============================================================
# 15. COLLECT COMPLETE PREDICTIONS
# ============================================================

all_predictions = []

for variant in MODEL_ORDER:

    for test_index in eval_indices:

        if test_index not in checkpoint_records[variant]:

            raise RuntimeError(
                f"Missing prediction: {variant}, index={test_index}"
            )

        all_predictions.append(
            checkpoint_records[variant][test_index]
        )


expected_total = (
    NUM_TEST_SAMPLES
    * len(MODEL_ORDER)
)

if len(all_predictions) != expected_total:

    raise RuntimeError(
        f"Expected {expected_total} predictions, "
        f"received {len(all_predictions)}."
    )


# ============================================================
# 16. WILSON 95% CONFIDENCE INTERVAL
# ============================================================

def wilson_interval(
    correct,
    total,
    z=1.959963984540054,
):

    if total <= 0:
        return None, None

    p = correct / total

    z2 = z * z

    denominator = (
        1.0 + z2 / total
    )

    center = (
        p + z2 / (2.0 * total)
    ) / denominator

    margin = (
        z
        * math.sqrt(
            (
                p * (1.0 - p) / total
            )
            + (
                z2 / (4.0 * total * total)
            )
        )
        / denominator
    )

    lower = max(
        0.0,
        center - margin,
    )

    upper = min(
        1.0,
        center + margin,
    )

    return (
        100.0 * lower,
        100.0 * upper,
    )


# ============================================================
# 17. MODEL SUMMARIES
# ============================================================

rows_by_model = {}

all_summaries = []

for variant in MODEL_ORDER:

    rows = [
        checkpoint_records[variant][i]
        for i in eval_indices
    ]

    rows_by_model[variant] = {
        row["test_index"]: row
        for row in rows
    }

    correct_count = sum(
        bool(row["correct"])
        for row in rows
    )

    total_count = len(rows)

    unparsed_count = sum(
        row["prediction"] is None
        for row in rows
    )

    accuracy = (
        100.0
        * correct_count
        / total_count
    )

    ci_lower, ci_upper = wilson_interval(
        correct_count,
        total_count,
    )

    summary = {
        "model": variant,
        "base_model": MODEL_NAME,
        "evaluation_dataset": (
            "openai/gsm8k/main/test"
        ),
        "correct": correct_count,
        "total": total_count,
        "incorrect": total_count - correct_count,
        "unparsed_predictions": unparsed_count,
        "accuracy_percent": round(
            accuracy,
            4,
        ),
        "accuracy_ci95_lower_percent": round(
            ci_lower,
            4,
        ),
        "accuracy_ci95_upper_percent": round(
            ci_upper,
            4,
        ),
        "avg_latency_seconds": round(
            sum(
                row["latency_seconds"]
                for row in rows
            ) / total_count,
            4,
        ),
        "avg_generated_tokens": round(
            sum(
                row["generated_tokens"]
                for row in rows
            ) / total_count,
            4,
        ),
        "answer_parser": PARSER_VERSION,
    }

    all_summaries.append(
        summary
    )


# ============================================================
# 18. EXACT McNEMAR TEST
# ============================================================

def exact_mcnemar_pvalue(
    left_only,
    right_only,
):

    discordant = (
        left_only + right_only
    )

    if discordant == 0:
        return 1.0

    try:

        from scipy.stats import binomtest

        result = binomtest(
            k=min(
                left_only,
                right_only,
            ),
            n=discordant,
            p=0.5,
            alternative="two-sided",
        )

        return float(
            result.pvalue
        )

    except Exception:

        # Exact binomial fallback using log probabilities.

        smaller = min(
            left_only,
            right_only,
        )

        log_terms = []

        for k in range(smaller + 1):

            log_combination = (
                math.lgamma(discordant + 1)
                - math.lgamma(k + 1)
                - math.lgamma(discordant - k + 1)
            )

            log_probability = (
                log_combination
                - discordant * math.log(2.0)
            )

            log_terms.append(
                log_probability
            )

        maximum = max(
            log_terms
        )

        log_sum = (
            maximum
            + math.log(
                sum(
                    math.exp(value - maximum)
                    for value in log_terms
                )
            )
        )

        return min(
            1.0,
            2.0 * math.exp(log_sum),
        )


# ============================================================
# 19. PAIRED PER-QUESTION COMPARISON
# ============================================================

paired_comparison = {}

comparison_pairs = (
    ("USC-DPO", "SFT-CoT"),
    ("USC-DPO", "Base"),
    ("SFT-CoT", "Base"),
)

for left, right in comparison_pairs:

    left_rows = rows_by_model[left]
    right_rows = rows_by_model[right]

    common = sorted(
        set(left_rows)
        & set(right_rows)
    )

    if len(common) != NUM_TEST_SAMPLES:

        raise RuntimeError(
            f"Paired comparison mismatch: "
            f"{left} vs {right} has {len(common)} questions."
        )

    both_correct = 0
    left_only_correct = 0
    right_only_correct = 0
    both_wrong = 0

    for index in common:

        left_correct = bool(
            left_rows[index]["correct"]
        )

        right_correct = bool(
            right_rows[index]["correct"]
        )

        if left_correct and right_correct:

            both_correct += 1

        elif left_correct and not right_correct:

            left_only_correct += 1

        elif not left_correct and right_correct:

            right_only_correct += 1

        else:

            both_wrong += 1

    pvalue = exact_mcnemar_pvalue(
        left_only_correct,
        right_only_correct,
    )

    difference_pp = (
        100.0
        * (
            left_only_correct
            - right_only_correct
        )
        / len(common)
    )

    paired_comparison[
        f"{left}_vs_{right}"
    ] = {
        "left_model": left,
        "right_model": right,
        "num_shared_questions": len(common),
        "both_correct": both_correct,
        "left_only_correct": left_only_correct,
        "right_only_correct": right_only_correct,
        "both_wrong_or_unparsed": both_wrong,
        "accuracy_difference_percentage_points": round(
            difference_pp,
            4,
        ),
        "mcnemar_exact_p_value": pvalue,
    }


# ============================================================
# 20. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 94)
print("FINAL COMPARISON — GSM8K 100-QUESTION TEST")
print("=" * 94)

print(
    f"{'Model':<14}"
    f"{'Correct':>12}"
    f"{'Accuracy':>12}"
    f"{'95% CI':>25}"
    f"{'Unparsed':>12}"
)

for row in all_summaries:

    ci_text = (
        f"[{row['accuracy_ci95_lower_percent']:.2f}, "
        f"{row['accuracy_ci95_upper_percent']:.2f}]"
    )

    print(
        f"{row['model']:<14}"
        f"{str(row['correct']) + '/' + str(row['total']):>12}"
        f"{row['accuracy_percent']:>11.2f}%"
        f"{ci_text:>25}"
        f"{row['unparsed_predictions']:>12}"
    )


print("\n===== PAIRED COMPARISONS =====")

for name, result in paired_comparison.items():

    print("\n", name)

    print(
        "Both correct:",
        result["both_correct"],
    )

    print(
        "Left only correct:",
        result["left_only_correct"],
    )

    print(
        "Right only correct:",
        result["right_only_correct"],
    )

    print(
        "Accuracy difference (percentage points):",
        result["accuracy_difference_percentage_points"],
    )

    print(
        "McNemar exact p-value:",
        result["mcnemar_exact_p_value"],
    )


# ============================================================
# 21. BACK UP PREVIOUS 50-QUESTION RESULTS
# Preserve old evaluation files before replacing canonical names.
# Cell38 uses the canonical filenames below.
# ============================================================

legacy_files = [
    "final_evaluation_summary.json",
    "final_evaluation_predictions.json",
    "final_evaluation_comparison.csv",
]

for filename in legacy_files:

    source_path = os.path.join(
        ARTIFACT_DIR,
        filename,
    )

    backup_path = os.path.join(
        ARTIFACT_DIR,
        filename.replace(
            ".json",
            "_50q_legacy.json",
        ).replace(
            ".csv",
            "_50q_legacy.csv",
        ),
    )

    if (
        os.path.isfile(source_path)
        and not os.path.isfile(backup_path)
    ):

        shutil.copy2(
            source_path,
            backup_path,
        )

        print(
            "Preserved previous result:",
            os.path.basename(backup_path),
        )


# ============================================================
# 22. SAVE FINAL ARTIFACTS
# Use canonical filenames so existing Cell38 can package them.
# ============================================================

summary_path = os.path.join(
    ARTIFACT_DIR,
    "final_evaluation_summary.json",
)

predictions_path = os.path.join(
    ARTIFACT_DIR,
    "final_evaluation_predictions.json",
)

csv_path = os.path.join(
    ARTIFACT_DIR,
    "final_evaluation_comparison.csv",
)

paired_csv_path = os.path.join(
    ARTIFACT_DIR,
    "final_evaluation_paired_comparison_100.csv",
)

summary_payload = {
    "evaluation_protocol": {
        "evaluation_version": EVALUATION_NAME,
        "dataset": "openai/gsm8k",
        "config": "main",
        "split": "test",
        "num_questions": NUM_TEST_SAMPLES,
        "test_indices": eval_indices,
        "test_selection": (
            "First 100 questions from the official test split; "
            "fixed indices 0-99."
        ),
        "model_count": len(MODEL_ORDER),
        "total_prediction_records": len(all_predictions),
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "repetition_penalty": REPETITION_PENALTY,
        "do_sample": False,
        "seed": SEED,
        "answer_parser": PARSER_VERSION,
        "unparsed_predictions_count_as_incorrect": True,
        "same_questions_for_all_models": True,
        "same_prompts_for_all_models": True,
        "same_decoding_settings": True,
        "dataset_fingerprint": dataset_fingerprint,
        "prompt_fingerprint": prompt_fingerprint,
        "run_signature": run_signature,
        "checkpoint_directory": CHECKPOINT_ROOT,
    },
    "model_summaries": all_summaries,
    "paired_comparison": paired_comparison,
    "elapsed_total_seconds": round(
        time.time() - overall_start,
        2,
    ),
}


with open(
    summary_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        summary_payload,
        f,
        ensure_ascii=False,
        indent=2,
    )


with open(
    predictions_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        all_predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )


# Model-level comparison CSV.

summary_fields = [
    "model",
    "correct",
    "total",
    "incorrect",
    "unparsed_predictions",
    "accuracy_percent",
    "accuracy_ci95_lower_percent",
    "accuracy_ci95_upper_percent",
    "avg_latency_seconds",
    "avg_generated_tokens",
    "answer_parser",
]

with open(
    csv_path,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=summary_fields,
        extrasaction="ignore",
    )

    writer.writeheader()

    writer.writerows(
        all_summaries
    )


# Paired comparison CSV.

paired_fields = [
    "left_model",
    "right_model",
    "num_shared_questions",
    "both_correct",
    "left_only_correct",
    "right_only_correct",
    "both_wrong_or_unparsed",
    "accuracy_difference_percentage_points",
    "mcnemar_exact_p_value",
]

with open(
    paired_csv_path,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=paired_fields,
        extrasaction="ignore",
    )

    writer.writeheader()

    writer.writerows(
        paired_comparison.values()
    )


# ============================================================
# 23. FINAL MANIFEST
# ============================================================

evaluation_manifest["status"] = "completed"

evaluation_manifest["completed_models"] = list(
    MODEL_ORDER
)

evaluation_manifest["model_progress"] = {
    model: NUM_TEST_SAMPLES
    for model in MODEL_ORDER
}

evaluation_manifest["final_summary_path"] = summary_path
evaluation_manifest["final_predictions_path"] = predictions_path

evaluation_manifest["finished_at"] = time.strftime(
    "%Y-%m-%dT%H:%M:%S",
    time.gmtime(),
)

atomic_write_json(
    MANIFEST_PATH,
    evaluation_manifest,
)

# Also save a copy beside the final artifacts.

artifact_manifest_path = os.path.join(
    ARTIFACT_DIR,
    "final_evaluation_manifest_100.json",
)

atomic_write_json(
    artifact_manifest_path,
    evaluation_manifest,
)


# ============================================================
# 24. FINAL STATUS
# Cell38 will create the project ZIP afterward.
# ============================================================

print("\n" + "=" * 94)
print("USC-DPO — 100-QUESTION EVALUATION COMPLETED")
print("=" * 94)

print("Test questions per model:", NUM_TEST_SAMPLES)
print("Models:", len(MODEL_ORDER))
print("Total predictions:", len(all_predictions))
print("Coverage of selected 100-question subset: 100%")

print("\nSaved artifacts:")

print("Summary:", summary_path)
print("Predictions:", predictions_path)
print("Comparison CSV:", csv_path)
print("Paired comparison CSV:", paired_csv_path)
print("Manifest:", artifact_manifest_path)

print("\nCheckpoint directory:", CHECKPOINT_ROOT)

print("\nEvaluation status: COMPLETED")
print("GPU training: NOT PERFORMED")

print(
    "\nInterpretation note: this is a fixed 100-question "
    "subset of GSM8K test, not the full 1,319-question test split."
)

print("\nCell39 V2 completed.")


USC-DPO — FINAL EVALUATION V2
Base model: Qwen/Qwen2.5-Math-1.5B
Test questions: 100
Models: 3
Expected total predictions: 300
Max new tokens: 384
Checkpoint interval: 5
Checkpoint storage: Google Drive
Checkpoint directory: /content/drive/MyDrive/USC_DPO_artifacts/final_evaluation_100_v2
GPU: Tesla T4
Evaluation dtype: torch.bfloat16
SFT adapter: /content/USC_DPO_artifacts/sft_lora_adapter
USC-DPO adapter: /content/USC_DPO_artifacts/dpo_lora_adapter

Loading GSM8K...
Loaded: 100 questions
Test indices: 0 to 99

Prompt validation: PASS
Prompt count: 100
Maximum prompt tokens: 186
Maximum input tokens: 1024

Dataset SHA256: 18ad95eac61b1ff7f9d7
Prompt SHA256: 1096b6afd6bb098271cd
SFT adapter SHA256: d86b569741e1f8af24b1
DPO adapter SHA256: 813fc45c867d0c7bb56c

New evaluation checkpoint initialized.
Base: 0/100 predictions already saved
SFT-CoT: 0/100 predictions already saved
USC-DPO: 0/100 predictions already saved

EVALUATING: Base
Already completed: 0
Questions remaining: 100


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded on: cuda:0
[5/100] Correct=1 Accuracy=20.00% | Last=20 | Gold=20
[10/100] Correct=1 Accuracy=10.00% | Last=None | Gold=460
[15/100] Correct=2 Accuracy=13.33% | Last=None | Gold=60
[20/100] Correct=3 Accuracy=15.00% | Last=None | Gold=6
[25/100] Correct=3 Accuracy=12.00% | Last=None | Gold=26
[30/100] Correct=3 Accuracy=10.00% | Last=None | Gold=104
[35/100] Correct=3 Accuracy=8.57% | Last=None | Gold=23
[40/100] Correct=4 Accuracy=10.00% | Last=None | Gold=18
[45/100] Correct=4 Accuracy=8.89% | Last=None | Gold=20
[50/100] Correct=5 Accuracy=10.00% | Last=None | Gold=30
[55/100] Correct=5 Accuracy=9.09% | Last=48 | Gold=40
[60/100] Correct=5 Accuracy=8.33% | Last=None | Gold=187
[65/100] Correct=5 Accuracy=7.69% | Last=None | Gold=300
[70/100] Correct=9 Accuracy=12.86% | Last=60 | Gold=60
[75/100] Correct=9 Accuracy=12.00% | Last=None | Gold=88
[80/100] Correct=11 Accuracy=13.75% | Last=70 | Gold=70
[85/100] Correct=11 Accuracy=12.94% | Last=None | Gold=15
[90/100] Correct

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded on: cuda:0
[5/100] Correct=4 Accuracy=80.00% | Last=20 | Gold=20
[10/100] Correct=7 Accuracy=70.00% | Last=460 | Gold=460
[15/100] Correct=9 Accuracy=60.00% | Last=None | Gold=60
[20/100] Correct=13 Accuracy=65.00% | Last=6 | Gold=6
[25/100] Correct=16 Accuracy=64.00% | Last=26 | Gold=26
[30/100] Correct=19 Accuracy=63.33% | Last=61 | Gold=104
[35/100] Correct=23 Accuracy=65.71% | Last=23 | Gold=23
[40/100] Correct=26 Accuracy=65.00% | Last=48 | Gold=18
[45/100] Correct=29 Accuracy=64.44% | Last=20 | Gold=20
[50/100] Correct=32 Accuracy=64.00% | Last=30 | Gold=30
[55/100] Correct=34 Accuracy=61.82% | Last=33 | Gold=40
[60/100] Correct=38 Accuracy=63.33% | Last=187 | Gold=187
[65/100] Correct=40 Accuracy=61.54% | Last=300 | Gold=300
[70/100] Correct=44 Accuracy=62.86% | Last=60 | Gold=60
[75/100] Correct=47 Accuracy=62.67% | Last=86 | Gold=88
[80/100] Correct=50 Accuracy=62.50% | Last=70 | Gold=70
[85/100] Correct=55 Accuracy=64.71% | Last=15 | Gold=15
[90/100] Correct=58 A

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded on: cuda:0
[5/100] Correct=4 Accuracy=80.00% | Last=20 | Gold=20
[10/100] Correct=7 Accuracy=70.00% | Last=460 | Gold=460
[15/100] Correct=10 Accuracy=66.67% | Last=60 | Gold=60
[20/100] Correct=14 Accuracy=70.00% | Last=6 | Gold=6
[25/100] Correct=17 Accuracy=68.00% | Last=26 | Gold=26
[30/100] Correct=20 Accuracy=66.67% | Last=61 | Gold=104
[35/100] Correct=24 Accuracy=68.57% | Last=23 | Gold=23
[40/100] Correct=26 Accuracy=65.00% | Last=48 | Gold=18
[45/100] Correct=29 Accuracy=64.44% | Last=20 | Gold=20
[50/100] Correct=32 Accuracy=64.00% | Last=30 | Gold=30
[55/100] Correct=34 Accuracy=61.82% | Last=33 | Gold=40
[60/100] Correct=38 Accuracy=63.33% | Last=187 | Gold=187
[65/100] Correct=41 Accuracy=63.08% | Last=300 | Gold=300
[70/100] Correct=46 Accuracy=65.71% | Last=60 | Gold=60
[75/100] Correct=49 Accuracy=65.33% | Last=85.55 | Gold=88
[80/100] Correct=51 Accuracy=63.75% | Last=70 | Gold=70
[85/100] Correct=56 Accuracy=65.88% | Last=15 | Gold=15
[90/100] Correct=60